In [ ]:
import time, uuid
_GUARD_STARTED = time.monotonic()
import json, os
os.environ['SF_COMMIT_TASKS'] = '8'
os.environ['SF_DEADLINE_S'] = '41400'
os.environ['SF_METHOD'] = 'score'
os.environ['SF_MODE'] = 'submit'
os.environ['SF_N_GEN'] = '8'
os.environ['SF_N_SCORE'] = '8'
os.environ['SF_TASK_CAP_S'] = '900'
os.environ['SF_TTT_ALPHA'] = '64'
os.environ['SF_TTT_FRAC'] = '0.7'
os.environ['SF_TTT_LR'] = '1e-4'
os.environ['SF_TTT_MAX_TOKENS'] = '8192'
os.environ['SF_TTT_RANK'] = '64'
os.environ['SF_TTT_STEPS'] = '48'
CODE_DIR = os.environ.get('SF_KAGGLE_ROOT', '/kaggle') + '/working/sf_code_guarded_' + uuid.uuid4().hex
CODE_SHA = '2d876c820e923316cd9f8f3971c3a4a86a22dff8ba627056f0b8a01fb668eb24'
FILES = json.loads('{"sf/__init__.py": "", "sf/grids.py": "\\"\\"\\"Grid utilities: validation, the eight symmetries of the square (D8), colour permutations, and task augmentation\\nwith exact inverses. Written for this project from first principles.\\"\\"\\"\\nimport random\\n\\nimport numpy as np\\n\\nN_COLOURS = 10\\nMAX_SIDE = 30\\n\\n\\ndef as_array(grid):\\n    a = np.asarray(grid, dtype=np.int64)\\n    if a.ndim != 2:\\n        raise ValueError(\\"grid must be 2-D\\")\\n    return a\\n\\n\\ndef is_valid_grid(grid):\\n    \\"\\"\\"A valid ARC grid: a non-empty rectangular list of rows of ints in 0..9, at most 30 x 30.\\"\\"\\"\\n    if not isinstance(grid, (list, tuple)) or not grid:\\n        return False\\n    w = None\\n    for row in grid:\\n        if not isinstance(row, (list, tuple)) or not row:\\n            return False\\n        if w is None:\\n            w = len(row)\\n        elif len(row) != w:\\n            return False\\n        for v in row:\\n            if isinstance(v, bool) or not isinstance(v, (int, np.integer)) or not 0 <= int(v) < N_COLOURS:\\n                return False\\n    return len(grid) <= MAX_SIDE and w <= MAX_SIDE\\n\\n\\ndef to_list(a):\\n    return [[int(v) for v in row] for row in np.asarray(a)]\\n\\n\\ndef key(grid):\\n    \\"\\"\\"Hashable canonical key of a grid.\\"\\"\\"\\n    a = np.asarray(grid)\\n    return (a.shape[0], a.shape[1], a.astype(np.int8).tobytes())\\n\\n\\n# D8: t in 0..7; k = t % 4 quarter turns counter-clockwise, then a left-right flip when t >= 4.\\ndef d8(a, t):\\n    a = np.rot90(np.asarray(a), t % 4)\\n    if t >= 4:\\n        a = np.fliplr(a)\\n    return np.ascontiguousarray(a)\\n\\n\\ndef d8_inv(a, t):\\n    a = np.asarray(a)\\n    if t >= 4:\\n        a = np.fliplr(a)\\n    return np.ascontiguousarray(np.rot90(a, -(t % 4)))\\n\\n\\ndef identity_perm():\\n    return np.arange(N_COLOURS)\\n\\n\\ndef random_perm(rng, keep_zero=True):\\n    p = np.arange(N_COLOURS)\\n    if keep_zero:\\n        rest = list(range(1, N_COLOURS))\\n        rng.shuffle(rest)\\n        p[1:] = rest\\n    else:\\n        rest = list(range(N_COLOURS))\\n        rng.shuffle(rest)\\n        p[:] = rest\\n    return p\\n\\n\\nclass Aug:\\n    \\"\\"\\"One augmentation: a D8 element, a colour permutation (applied after the symmetry) and a demonstration order.\\"\\"\\"\\n\\n    def __init__(self, t=0, perm=None, order=None):\\n        self.t = int(t)\\n        self.perm = identity_perm() if perm is None else np.asarray(perm, dtype=np.int64)\\n        self.inv_perm = np.argsort(self.perm)\\n        self.order = None if order is None else list(order)\\n\\n    def fwd(self, g):\\n        return self.perm[d8(as_array(g), self.t)]\\n\\n    def inv(self, g):\\n        return d8_inv(self.inv_perm[as_array(g)], self.t)\\n\\n    def apply_task(self, train_pairs, test_inputs):\\n        pairs = [(self.fwd(i), self.fwd(o)) for i, o in train_pairs]\\n        if self.order is not None:\\n            pairs = [pairs[j] for j in self.order]\\n        return pairs, [self.fwd(g) for g in test_inputs]\\n\\n    def describe(self):\\n        return {\\"t\\": self.t, \\"perm\\": [int(v) for v in self.perm], \\"order\\": self.order}\\n\\n    @staticmethod\\n    def from_desc(d):\\n        return Aug(d[\\"t\\"], d[\\"perm\\"], d.get(\\"order\\"))\\n\\n\\ndef make_augs(n, n_train, seed, keep_zero=True, shuffle_order=True):\\n    \\"\\"\\"n augmentations: the identity first, then the other seven symmetries with identity colours, then random\\n    (symmetry, colour permutation, order) triples. Deterministic in seed.\\"\\"\\"\\n    rng = random.Random(seed)\\n    augs = [Aug(0)]\\n    for t in range(1, 8):\\n        if len(augs) >= n:\\n            break\\n        augs.append(Aug(t))\\n    while len(augs) < n:\\n        t = rng.randrange(8)\\n        perm = random_perm(rng, keep_zero)\\n        order = list(range(n_train))\\n        if shuffle_order:\\n            rng.shuffle(order)\\n        augs.append(Aug(t, perm, order))\\n    return augs[:n]\\n\\n\\ndef palette(g):\\n    return set(int(v) for v in np.unique(as_array(g)))\\n", "sf/lm.py": "\\"\\"\\"The grid language model: loading, prompt prefill, batched constrained greedy decoding and teacher-forced scoring.\\n\\nOwn implementation on transformers\' public model API (Apache-2.0). Prompts are prefilled one at a time (no padding,\\nso the causal attention kernels apply), then their key/value caches are left-padded into one batch for decoding.\\n\\"\\"\\"\\nimport time\\n\\nimport torch\\nfrom transformers import AutoModelForCausalLM, DynamicCache, StaticCache\\n\\nfrom . import textfmt as F\\n\\nPREFILL_CHUNK = 4096                 # tokens per prefill forward (1024 on pre-Ampere GPUs, see LM)\\n\\n\\ndef make_cache(kv):\\n    \\"\\"\\"A DynamicCache holding the given per-layer (keys, values).\\"\\"\\"\\n    try:\\n        return DynamicCache(ddp_cache_data=kv)\\n    except TypeError:\\n        cache = DynamicCache()\\n        for i, (k, v) in enumerate(kv):\\n            cache.update(k, v, i)\\n        return cache\\n\\n\\nclass LM:\\n    def __init__(self, path, device=\\"cuda:0\\", dtype=None, attn=\\"sdpa\\"):\\n        self.device = torch.device(device)\\n        if dtype is None:\\n            if self.device.type != \\"cuda\\":\\n                dtype = torch.float32\\n            else:\\n                major = torch.cuda.get_device_capability(self.device)[0]\\n                dtype = torch.bfloat16 if major >= 8 else torch.float16\\n        self.dtype = dtype\\n        self.chunk = PREFILL_CHUNK\\n        self.use_static = True\\n        if self.device.type == \\"cuda\\" and torch.cuda.get_device_capability(self.device)[0] < 8:\\n            self.chunk = 1024\\n        t0 = time.time()\\n        self.model = AutoModelForCausalLM.from_pretrained(path, dtype=dtype, device_map={\\"\\": str(self.device)},\\n                                                          attn_implementation=attn)\\n        self.model.eval()\\n        self.load_s = time.time() - t0\\n        self.stats = {\\"prefill_tokens\\": 0, \\"decode_steps\\": 0, \\"decode_seq_steps\\": 0, \\"score_tokens\\": 0,\\n                      \\"prefill_s\\": 0.0, \\"decode_s\\": 0.0, \\"score_s\\": 0.0}\\n\\n    # ----- prefill -------------------------------------------------------------------------------------------\\n    @torch.no_grad()\\n    def prefill(self, ids):\\n        \\"\\"\\"Run a prompt (batch 1); return (cache, log-probs of the next token).\\"\\"\\"\\n        t0 = time.time()\\n        cache = DynamicCache()\\n        out = None\\n        for s in range(0, len(ids), self.chunk):\\n            chunk = torch.tensor([ids[s:s + self.chunk]], device=self.device)\\n            out = self.model(input_ids=chunk, past_key_values=cache, use_cache=True, logits_to_keep=1)\\n            cache = out.past_key_values\\n        lp = torch.log_softmax(out.logits[0, -1].float(), -1)\\n        self.stats[\\"prefill_tokens\\"] += len(ids)\\n        self.stats[\\"prefill_s\\"] += time.time() - t0\\n        return cache, lp\\n\\n    # ----- constrained greedy decoding -----------------------------------------------------------------------\\n    @torch.no_grad()\\n    def _static_batch(self, prompts, max_len):\\n        \\"\\"\\"Prefill each prompt alone and copy its keys/values, left-padded, into one preallocated static cache\\n        (decoding then writes in place instead of re-concatenating the whole cache at every step).\\"\\"\\"\\n        cfg = self.model.config\\n        B, lens = len(prompts), [len(p) for p in prompts]\\n        L = max(lens)\\n        cache = StaticCache(config=cfg, max_cache_len=max_len)\\n        cache.early_initialization(B, cfg.num_key_value_heads, cfg.head_dim, self.dtype, self.device)\\n        first = []\\n        for b, p in enumerate(prompts):\\n            c, lp = self.prefill(p)\\n            for dst, src in zip(cache.layers, c.layers):\\n                dst.keys[b, :, L - lens[b]:L].copy_(src.keys[0])\\n                dst.values[b, :, L - lens[b]:L].copy_(src.values[0])\\n            first.append(lp)\\n            del c\\n        for layer in cache.layers:\\n            if torch.is_tensor(layer.cumulative_length):\\n                layer.cumulative_length.fill_(L)\\n            else:\\n                layer.cumulative_length = L\\n        return cache, torch.stack(first), lens, L\\n\\n    @torch.no_grad()\\n    def _dynamic_batch(self, prompts):\\n        \\"\\"\\"Fallback when the static cache API is unavailable: left-padded concatenation into a DynamicCache.\\"\\"\\"\\n        caches, first = [], []\\n        for p in prompts:\\n            c, lp = self.prefill(p)\\n            caches.append(c)\\n            first.append(lp)\\n        lens = [len(p) for p in prompts]\\n        L = max(lens)\\n        merged = []\\n        for li in range(len(caches[0].layers)):\\n            ks, vs = [], []\\n            for b, c in enumerate(caches):\\n                pad = L - lens[b]\\n                ks.append(torch.nn.functional.pad(c.layers[li].keys, (0, 0, pad, 0)))\\n                vs.append(torch.nn.functional.pad(c.layers[li].values, (0, 0, pad, 0)))\\n            merged.append((torch.cat(ks, 0), torch.cat(vs, 0)))\\n        del caches\\n        return make_cache(merged), torch.stack(first), lens, L\\n\\n    @torch.no_grad()\\n    def greedy(self, prompts, max_new=F.MAX_OUT_TOKENS):\\n        \\"\\"\\"Constrained greedy decoding of a batch of prompts. Returns, per prompt, (tokens incl. end, sum logp).\\"\\"\\"\\n        B = len(prompts)\\n        L0 = max(len(p) for p in prompts)\\n        static = self.use_static\\n        if static:\\n            try:\\n                cache, lp, lens, L = self._static_batch(prompts, L0 + max_new)\\n            except (TypeError, AttributeError, NotImplementedError):\\n                self.use_static = static = False\\n        if not static:\\n            cache, lp, lens, L = self._dynamic_batch(prompts)\\n        t0 = time.time()\\n        mask = torch.zeros((B, L + max_new), dtype=torch.long, device=self.device)\\n        for b in range(B):\\n            mask[b, L - lens[b]:L] = 1\\n        cons = [F.GridConstraint() for _ in range(B)]\\n        toks = [[] for _ in range(B)]\\n        score = [0.0] * B\\n        pos = torch.tensor(lens, device=self.device)\\n        V = lp.shape[-1]\\n        for step in range(max_new):\\n            allow = torch.zeros((B, V), dtype=torch.bool)\\n            live = []\\n            for b in range(B):\\n                a = cons[b].allowed()\\n                if a:\\n                    allow[b, list(a)] = True\\n                    live.append(b)\\n            if not live:\\n                break\\n            allow = allow.to(self.device, non_blocking=True)\\n            nxt = lp.masked_fill(~allow, float(\\"-inf\\")).argmax(-1)\\n            chosen = lp.gather(1, nxt[:, None])[:, 0]\\n            both = torch.stack([nxt.float(), chosen]).tolist()\\n            nl = [int(v) for v in both[0]]\\n            for b in live:\\n                toks[b].append(nl[b])\\n                score[b] += both[1][b]\\n                cons[b].step(nl[b])\\n            if all(c.done for c in cons):\\n                break\\n            mask[:, L + step] = 1\\n            if static:\\n                out = self.model(input_ids=nxt[:, None], attention_mask=mask, position_ids=pos[:, None],\\n                                 past_key_values=cache, use_cache=True,\\n                                 cache_position=torch.tensor([L + step], device=self.device))\\n            else:\\n                out = self.model(input_ids=nxt[:, None], attention_mask=mask[:, :L + step + 1],\\n                                 position_ids=pos[:, None], past_key_values=cache, use_cache=True)\\n                cache = out.past_key_values\\n            pos = pos + 1\\n            lp = torch.log_softmax(out.logits[:, -1].float(), -1)\\n            self.stats[\\"decode_steps\\"] += 1\\n            self.stats[\\"decode_seq_steps\\"] += len(live)\\n        del cache\\n        self.stats[\\"decode_s\\"] += time.time() - t0\\n        return [(toks[b], score[b]) for b in range(B)]\\n\\n    # ----- teacher-forced scoring ----------------------------------------------------------------------------\\n    @torch.no_grad()\\n    def score(self, prompt, targets, cache=None, first_lp=None):\\n        \\"\\"\\"Sum of log-probs of each target token sequence after the prompt (prompt cache reused, cropped back).\\"\\"\\"\\n        if cache is None:\\n            cache, first_lp = self.prefill(prompt)\\n        t0 = time.time()\\n        L = len(prompt)\\n        res = []\\n        for tg in targets:\\n            total = float(first_lp[tg[0]])\\n            if len(tg) > 1:\\n                x = torch.tensor([tg[:-1]], device=self.device)\\n                out = self.model(input_ids=x, past_key_values=cache, use_cache=True,\\n                                 position_ids=torch.arange(L, L + len(tg) - 1, device=self.device)[None])\\n                lps = torch.log_softmax(out.logits[0].float(), -1)\\n                total += float(lps.gather(1, torch.tensor(tg[1:], device=self.device)[:, None]).sum())\\n                cache = out.past_key_values\\n                cache.crop(-(len(tg) - 1))          # drop the target tokens again\\n            self.stats[\\"score_tokens\\"] += len(tg)\\n            res.append(total)\\n        self.stats[\\"score_s\\"] += time.time() - t0\\n        return res\\n", "sf/pilot.py": "\\"\\"\\"TTT pilot (authorized by the author on 26 Oct; Kaggle L4x4 only, at most 2 h of runtime in total).\\n\\n  smoke: one LoRA configuration per GPU. Peak memory and time per step at the longest pilot sequence and at 8,192\\n         tokens, adapter inference time, and a leave-one-demonstration-out (LOO) check on three NON-pilot puzzles\\n         (adapt on all demonstrations but the last, then predict the last one\'s output: its NLL and greedy vote\\n         before and after). Demonstrations only.\\n  pilot: for each pilot puzzle, in one process: the LOO check, the no-TTT arm (fresh adapter = the base model) and\\n         the TTT arm (a fresh adapter trained on all demonstrations), both with the same inference. Query answers\\n         are never read.\\n  python -m sf.pilot smoke|pilot --rank R --challenges C --model M --out DIR --tasks a,b,c [--cfg JSON]\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport time\\n\\nimport numpy as np\\nimport torch\\n\\nfrom . import grids as G\\nfrom . import textfmt as F\\nfrom .lm import LM\\nfrom .predict import predict_output, task_seed\\nfrom .runner import atomic_json\\nfrom .ttt import TTT\\n\\n\\ndef peak_reset():\\n    if torch.cuda.is_available():\\n        torch.cuda.synchronize()\\n        torch.cuda.reset_peak_memory_stats()\\n\\n\\ndef peak():\\n    if not torch.cuda.is_available():\\n        return 0.0\\n    torch.cuda.synchronize()\\n    return round(torch.cuda.max_memory_allocated() / 1e9, 2)\\n\\n\\ndef device():\\n    return os.environ.get(\\"SF_DEVICE\\", \\"cuda:0\\")\\n\\n\\ndef pairs_of(task):\\n    return [(np.asarray(p[\\"input\\"]), np.asarray(p[\\"output\\"])) for p in task[\\"train\\"]]\\n\\n\\ndef loo_eval(lm, pairs, n_aug, seed):\\n    \\"\\"\\"Predict the last demonstration\'s output from the others: NLL of the true output (identity prompt), and\\n    whether the top vote of a greedy decode under n_aug augmentations equals it.\\"\\"\\"\\n    ctx, (hin, hout) = pairs[:-1], pairs[-1]\\n    p = F.prompt_ids(ctx, hin)\\n    nll = -lm.score(p, [F.target_ids(hout)])[0]\\n    res = predict_output(lm, ctx, hin, n_gen=n_aug, n_score=0, seed=seed)\\n    pool = sorted(res[\\"candidates\\"], key=lambda c: -c[\\"votes\\"])\\n    top = pool[0][\\"grid\\"] if pool else None\\n    hit_any = any(c[\\"grid\\"] == G.to_list(hout) for c in pool)\\n    return {\\"nll\\": round(nll, 4), \\"top_vote_correct\\": top == G.to_list(hout), \\"in_pool\\": hit_any,\\n            \\"n_cands\\": len(pool), \\"s\\": res[\\"total_s\\"]}\\n\\n\\ndef synthetic_pairs(n_tokens, rng):\\n    \\"\\"\\"Random 30 x 30 pairs until the LOO sequence reaches about n_tokens (memory test only).\\"\\"\\"\\n    pairs = []\\n    while sum(len(F.grid_ids(i)) + len(F.grid_ids(o)) + 12 for i, o in pairs) < n_tokens:\\n        pairs.append((rng.integers(0, 10, (30, 30)), rng.integers(0, 10, (30, 30))))\\n    return pairs\\n\\n\\ndef smoke(a, cfg, ch):\\n    lm = LM(a.model, device=device())\\n    out = {\\"cfg\\": cfg, \\"load_s\\": round(lm.load_s, 1),\\n           \\"gpu\\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\"cpu\\"}\\n    peak_reset()\\n    ttt = TTT(lm, rank=cfg[\\"rank\\"], alpha=cfg[\\"alpha\\"], lr=cfg[\\"lr\\"], max_tokens=cfg.get(\\"max_tokens\\", 8192))\\n    out[\\"lora_params\\"] = ttt.n_params\\n    out[\\"mem_after_wrap_gb\\"] = peak()\\n    big = cfg[\\"big_task\\"]\\n    rng = np.random.default_rng(0)\\n    for name, pairs in ((\\"longest_pilot\\", pairs_of(ch[big])), (\\"synthetic_8192\\", synthetic_pairs(8000, rng))):\\n        peak_reset()\\n        st = ttt.train(pairs, steps=cfg.get(\\"mem_steps\\", 4), seed=1, log_every=1)\\n        out[f\\"mem_{name}\\"] = {\\"peak_gb\\": peak(), \\"s_per_step\\": round(st[\\"train_s\\"] / max(1, st[\\"steps\\"]), 2),\\n                              \\"tokens_per_step\\": st[\\"tokens\\"] // max(1, st[\\"steps\\"]), \\"loss\\": st[\\"loss_curve\\"]}\\n        atomic_json(os.path.join(a.out, f\\"smoke_{a.rank}.json\\"), out)\\n    peak_reset()\\n    ttt.reset(seed=3)\\n    t0 = time.time()\\n    x = ch[big][\\"test\\"][0][\\"input\\"]\\n    r = predict_output(lm, pairs_of(ch[big]), np.asarray(x), cfg[\\"n_gen\\"], cfg[\\"n_score\\"], task_seed(big, 0))\\n    out[\\"adapter_inference\\"] = {\\"task\\": big, \\"s\\": round(time.time() - t0, 2), \\"peak_gb\\": peak(),\\n                                \\"n_cands\\": len(r[\\"candidates\\"]), \\"prompt_tokens\\": r[\\"prompt_tokens\\"]}\\n    out[\\"loo\\"] = []\\n    for tid in cfg[\\"loo_tasks\\"]:\\n        pairs = pairs_of(ch[tid])\\n        ttt.reset(seed=task_seed(tid, 99))\\n        before = loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7))\\n        peak_reset()\\n        st = ttt.train(pairs[:-1], steps=cfg[\\"steps\\"], seed=task_seed(tid, 99))\\n        after = loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7))\\n        out[\\"loo\\"].append({\\"task\\": tid, \\"before\\": before, \\"after\\": after, \\"train\\": st, \\"peak_gb\\": peak()})\\n        atomic_json(os.path.join(a.out, f\\"smoke_{a.rank}.json\\"), out)\\n    out[\\"lm_stats\\"] = lm.stats\\n    out[\\"done\\"] = True\\n    atomic_json(os.path.join(a.out, f\\"smoke_{a.rank}.json\\"), out)\\n\\n\\ndef pilot(a, cfg, ch):\\n    lm = LM(a.model, device=device())\\n    ttt = TTT(lm, rank=cfg[\\"rank\\"], alpha=cfg[\\"alpha\\"], lr=cfg[\\"lr\\"], max_tokens=cfg.get(\\"max_tokens\\", 8192))\\n    for tid in a.tasks.split(\\",\\"):\\n        path = os.path.join(a.out, \\"pilot\\", f\\"{tid}.json\\")\\n        if os.path.exists(path):\\n            continue\\n        pairs = pairs_of(ch[tid])\\n        rec = {\\"task\\": tid, \\"cfg\\": cfg, \\"n_demos\\": len(pairs)}\\n        seed = task_seed(tid, 99)\\n        if cfg.get(\\"loo\\", True) and len(pairs) >= 2:\\n            ttt.reset(seed)\\n            before = loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7))\\n            peak_reset()\\n            st = ttt.train(pairs[:-1], steps=cfg[\\"steps\\"], seed=seed)\\n            rec[\\"loo\\"] = {\\"before\\": before, \\"after\\": loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7)),\\n                          \\"train\\": st, \\"peak_gb\\": peak()}\\n        for arm in (\\"no_ttt\\", \\"ttt\\"):\\n            ttt.reset(seed)\\n            peak_reset()\\n            st = None\\n            if arm == \\"ttt\\":\\n                st = ttt.train(pairs, steps=cfg[\\"steps\\"], seed=seed)\\n            t0 = time.time()\\n            outs = [predict_output(lm, pairs, np.asarray(x[\\"input\\"]), cfg[\\"n_gen\\"], cfg[\\"n_score\\"],\\n                                   task_seed(tid, j)) for j, x in enumerate(ch[tid][\\"test\\"])]\\n            rec[arm] = {\\"outputs\\": outs, \\"train\\": st, \\"infer_s\\": round(time.time() - t0, 2), \\"peak_gb\\": peak()}\\n        atomic_json(path, rec)\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"mode\\", choices=[\\"smoke\\", \\"pilot\\"])\\n    ap.add_argument(\\"--rank\\", type=int, default=0)\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--model\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--tasks\\", default=\\"\\")\\n    ap.add_argument(\\"--cfg\\", required=True)\\n    a = ap.parse_args()\\n    cfg = json.loads(a.cfg)\\n    ch = json.load(open(a.challenges))\\n    os.makedirs(os.path.join(a.out, \\"pilot\\"), exist_ok=True)\\n    (smoke if a.mode == \\"smoke\\" else pilot)(a, cfg, ch)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sf/predict.py": "\\"\\"\\"Per-task candidate generation without test-time training.\\n\\nFor each test input: build the prompt under several augmentations (D8 symmetry, colour permutation keeping 0,\\ndemonstration order), decode greedily with the grid constraint, map every decoded grid back to the original frame,\\nand score every distinct candidate under every augmentation (teacher forcing). The full candidate pool is returned\\nso that selection rules can be compared offline on the same pool.\\n\\"\\"\\"\\nimport copy\\nimport math\\nimport time\\nimport zlib\\n\\nimport numpy as np\\n\\nfrom . import grids as G\\nfrom . import textfmt as F\\n\\nMAX_PROMPT = 12000            # tokens; longer prompts drop their longest demonstrations first\\nBATCH_TOKENS = 64000          # cap on (batch size x (prompt + answer)) tokens for one decoding batch\\n\\n\\ndef task_seed(task_id, idx):\\n    return zlib.crc32(f\\"{task_id}:{idx}\\".encode()) & 0x7FFFFFFF\\n\\n\\ndef fit_prompt(pairs, test_input, max_tokens=MAX_PROMPT):\\n    \\"\\"\\"Drop the longest demonstrations until the prompt fits; returns (kept pair indices, n_tokens).\\"\\"\\"\\n    keep = list(range(len(pairs)))\\n\\n    def n_tok(ix):\\n        return len(F.prompt_ids([pairs[i] for i in ix], test_input))\\n    n = n_tok(keep)\\n    while n > max_tokens and len(keep) > 1:\\n        longest = max(keep, key=lambda i: len(F.grid_ids(pairs[i][0])) + len(F.grid_ids(pairs[i][1])))\\n        keep.remove(longest)\\n        n = n_tok(keep)\\n    return keep, n\\n\\n\\ndef predict_output(lm, pairs, test_input, n_gen, n_score, seed, max_prompt=MAX_PROMPT, deadline=None, _prefix_token=None):\\n    \\"\\"\\"Candidate pool for one test input. n_gen augmentations decode; n_score >= n_gen augmentations score.\\n    The first n_gen scoring augmentations are the decoding ones.\\"\\"\\"\\n    t0 = time.time()\\n    keep, n_prompt = fit_prompt(pairs, test_input, max_prompt)\\n    kp = [pairs[i] for i in keep]\\n    augs = G.make_augs(max(n_gen, n_score), len(kp), seed)\\n    prompts = []\\n    for a in augs:\\n        ap, at = a.apply_task(kp, [test_input])\\n        prompts.append(F.prompt_ids(ap, at[0]))\\n    cands = {}\\n    gen_info = []\\n    bs = max(1, BATCH_TOKENS // (max(len(p) for p in prompts) + F.MAX_OUT_TOKENS))\\n    i = 0\\n    while i < n_gen:\\n        idx = list(range(i, min(n_gen, i + bs)))\\n        for j, (toks, lp) in zip(idx, lm.greedy([prompts[j] for j in idx])):\\n            g = F.parse_ids(toks)\\n            if g is None or not toks or toks[-1] != F.IME:\\n                gen_info.append({\\"aug\\": j, \\"valid\\": False})\\n                continue\\n            orig = augs[j].inv(np.asarray(g))\\n            k = G.key(orig)\\n            c = cands.setdefault(k, {\\"grid\\": G.to_list(orig), \\"votes\\": 0, \\"gen_augs\\": [], \\"gen_logp\\": []})\\n            c[\\"votes\\"] += 1\\n            c[\\"gen_augs\\"].append(j)\\n            c[\\"gen_logp\\"].append(round(lp, 4))\\n            gen_info.append({\\"aug\\": j, \\"valid\\": True})\\n        i = idx[-1] + 1\\n        if deadline is not None and time.time() > deadline:\\n            break\\n    pool = list(cands.values())\\n    t_gen = time.time() - t0\\n    n_scored = 0\\n    for c in pool:\\n        c[\\"scores\\"] = []\\n    def output():\\n        return {\\"candidates\\": pool, \\"kept_demos\\": keep, \\"prompt_tokens\\": n_prompt, \\"n_gen\\": n_gen,\\n                \\"n_score_done\\": n_scored, \\"augs\\": [a.describe() for a in augs[:max(n_gen, n_score)]],\\n                \\"gen_info\\": gen_info, \\"gen_s\\": round(t_gen, 2), \\"total_s\\": round(time.time() - t0, 2)}\\n    try:\\n        if pool:\\n            for j, a in enumerate(augs[:n_score]):\\n                if deadline is not None and time.time() > deadline and j >= 1:\\n                    break\\n                targets = [F.target_ids(a.fwd(np.asarray(c[\\"grid\\"]))) for c in pool]\\n                raw = list(lm.score(prompts[j], targets))\\n                if len(raw) != len(pool):\\n                    raise ValueError(\\"incomplete all-candidate scoring pass\\")\\n                if any(isinstance(s, (bool, np.bool_)) or not isinstance(s, (int, float, np.integer, np.floating))\\n                       or not math.isfinite(float(s)) for s in raw):\\n                    raise ValueError(\\"nonfinite or nonnumeric candidate score\\")\\n                values = [round(float(s), 4) for s in raw]\\n                # Commit this augmentation only after every candidate\'s score is valid.\\n                for c, s in zip(pool, values):\\n                    c[\\"scores\\"].append(s)\\n                n_scored += 1\\n    except Exception as error:\\n        # A reused exception object must never carry a stale prefix into another query.\\n        for attribute in (\\"completed_score_prefix\\", \\"completed_score_prefix_token\\", \\"recovered_query_index\\", \\"prefix_checkpoint_error\\"):\\n            if hasattr(error, attribute):\\n                delattr(error, attribute)\\n        if n_scored > 0 and all(len(c[\\"scores\\"]) == n_scored for c in pool):\\n            error.completed_score_prefix = copy.deepcopy(output())\\n            error.completed_score_prefix_token = _prefix_token\\n            error.completed_score_prefix[\\"recovery\\"] = {\\"kind\\": \\"complete_score_prefix\\",\\n                                                       \\"error\\": type(error).__name__}\\n        raise\\n    return output()\\n\\n\\ndef predict_task(lm, task_id, task, n_gen=8, n_score=8, max_prompt=MAX_PROMPT, deadline=None, on_query=None):\\n    pairs = [(np.asarray(p[\\"input\\"]), np.asarray(p[\\"output\\"])) for p in task[\\"train\\"]]\\n    outs = []\\n    for idx, t in enumerate(task[\\"test\\"]):\\n        prefix_token = object()\\n        try:\\n            if deadline is not None and time.time() >= deadline:\\n                raise TimeoutError(\\"task deadline reached before query\\")\\n            outs.append(predict_output(lm, pairs, np.asarray(t[\\"input\\"]), n_gen, n_score, task_seed(task_id, idx),\\n                                       max_prompt, deadline, _prefix_token=prefix_token))\\n            if on_query is not None:\\n                on_query(idx, outs[:])\\n        except Exception as e:\\n            recovered = (getattr(e, \\"completed_score_prefix\\", None)\\n                         if getattr(e, \\"completed_score_prefix_token\\", None) is prefix_token else None)\\n            if recovered is not None:\\n                outs.append(recovered)\\n                e.recovered_query_index = idx\\n                if on_query is not None:\\n                    try:\\n                        on_query(idx, outs[:])\\n                    except Exception as checkpoint_error:\\n                        # Preserve both the scoring fault and in-memory prefix if persistence fails.\\n                        e.prefix_checkpoint_error = type(checkpoint_error).__name__\\n            e.partial_outputs = outs\\n            raise\\n    return outs\\n", "sf/runner.py": "\\"\\"\\"Orchestrator and GPU workers: independent replicas (one per GPU) pull tasks from a shared queue, write each task\'s\\ncandidate pool atomically, and the orchestrator keeps submission.json valid at all times (fallbacks first, then\\nupdated atomically as pools arrive). The orchestrator is also the global watchdog: at its deadline it stops the\\nworkers and writes the final file.\\n\\n  python -m sf.runner orchestrate --challenges C --model M --out DIR --submission PATH [options]\\n  python -m sf.runner worker --rank R --challenges C --model M --out DIR --deadline UNIX_TS [options]\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport signal\\nimport subprocess\\nimport sys\\nimport time\\nimport traceback\\n\\nfrom . import select as S\\nfrom . import textfmt as F\\n\\n\\ndef atomic_json(path, obj):\\n    tmp = f\\"{path}.tmp{os.getpid()}\\"\\n    with open(tmp, \\"w\\") as f:\\n        json.dump(obj, f)\\n        f.flush()\\n        os.fsync(f.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef task_cost(t):\\n    n = sum(len(F.grid_ids(g)) for p in t[\\"train\\"] for g in (p[\\"input\\"], p[\\"output\\"]))\\n    return len(t[\\"test\\"]) * (n + max(len(F.grid_ids(x[\\"input\\"])) for x in t[\\"test\\"]))\\n\\n\\ndef order_tasks(ch):\\n    \\"\\"\\"Largest first, so that the long tasks do not finish last.\\"\\"\\"\\n    return sorted(ch, key=lambda k: -task_cost(ch[k]))\\n\\n\\n# ----- worker ----------------------------------------------------------------------------------------------------\\ndef claim(out, tid, rank):\\n    try:\\n        fd = os.open(os.path.join(out, \\"claims\\", tid), os.O_CREAT | os.O_EXCL | os.O_WRONLY)\\n    except FileExistsError:\\n        return False\\n    os.write(fd, str(rank).encode())\\n    os.close(fd)\\n    return True\\n\\n\\ndef peak_gb():\\n    import torch\\n    return round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0\\n\\n\\ndef preserve_query_results(current, incoming):\\n    \\"\\"\\"Retain the latest nonempty pool at each original query index.\\n\\n    Do not mix scores across budgets or compact empty slots. A nonempty retry\\n    keeps the baseline choice; an empty/missing retry cannot erase a prior pool.\\n    \\"\\"\\"\\n    if not isinstance(incoming, list):\\n        return current\\n    if current is None:\\n        return incoming[:]\\n    kept = current[:]\\n    for index, result in enumerate(incoming):\\n        if index >= len(kept):\\n            kept.append(result)\\n        elif result.get(\\"candidates\\"):\\n            kept[index] = result\\n    return kept\\n\\n\\ndef selected_snapshot(task, outputs, method):\\n    selected = []\\n    for j, query in enumerate(task[\\"test\\"]):\\n        pool = outputs[j].get(\\"candidates\\", []) if outputs and j < len(outputs) else []\\n        selected.append(S.two_attempts(pool, query[\\"input\\"], S.rank(pool, method)) if pool else None)\\n    return selected\\n\\n\\ndef checkpoint_outputs(path, task_id):\\n    try:\\n        if os.path.getsize(path) > 64 * 1024 * 1024:\\n            return None\\n        with open(path) as f:\\n            value = json.load(f)\\n        if not isinstance(value, dict):\\n            return None\\n        outputs = value.get(\\"outputs\\")\\n        if (value.get(\\"task\\") == task_id and isinstance(outputs, list)\\n                and all(isinstance(o, dict) and isinstance(o.get(\\"candidates\\"), list) for o in outputs)):\\n            return outputs\\n    except (OSError, ValueError, RecursionError):\\n        pass\\n    return None\\n\\n\\ndef worker(a):\\n    import torch\\n    from .lm import LM\\n    from .predict import predict_task, task_seed\\n    ch = json.load(open(a.challenges))\\n    queue = json.load(open(os.path.join(a.out, \\"queue.json\\")))\\n    log = open(os.path.join(a.out, \\"workers\\", f\\"{a.rank}.log\\"), \\"a\\")\\n\\n    def say(**kw):\\n        kw[\\"t\\"] = round(time.time(), 1)\\n        log.write(json.dumps(kw) + \\"\\\\n\\")\\n        log.flush()\\n    dev = os.environ.get(\\"SF_DEVICE\\", \\"cuda:0\\")\\n    lm = LM(a.model, device=dev)\\n    say(event=\\"loaded\\", load_s=round(lm.load_s, 1), dtype=str(lm.dtype),\\n        gpu=torch.cuda.get_device_name(0) if dev.startswith(\\"cuda\\") else dev)\\n    ttt = None\\n    if a.ttt_steps > 0:\\n        from .ttt import TTT\\n        ttt = TTT(lm, rank=a.ttt_rank, alpha=a.ttt_alpha, lr=a.ttt_lr, max_tokens=a.ttt_max_tokens)\\n        import peft\\n        say(event=\\"ttt\\", rank=a.ttt_rank, alpha=a.ttt_alpha, lr=a.ttt_lr, steps=a.ttt_steps, params=ttt.n_params,\\n            peft=peft.__version__, peft_file=peft.__file__)\\n    done = 0\\n    for tid in queue:\\n        if time.time() > a.deadline - 30:\\n            say(event=\\"deadline\\")\\n            break\\n        pool_path = os.path.join(a.out, \\"pools\\", f\\"{tid}.json\\")\\n        if os.path.exists(pool_path) or not claim(a.out, tid, a.rank):\\n            continue\\n        left = sum(1 for k in queue if not os.path.exists(os.path.join(a.out, \\"claims\\", k)))\\n        share = (a.deadline - time.time() - a.reserve_s) / max(1.0, left / a.n_workers + 1)\\n        task_deadline = min(a.deadline, time.time() + max(20.0, min(a.task_cap_s, share)))\\n        t0 = time.time()\\n        progress_path = os.path.join(a.out, \\"progress\\", f\\"{tid}.json\\")\\n        res, err, tstats = checkpoint_outputs(progress_path, tid), None, None\\n\\n        def save_query(index, outputs):\\n            nonlocal res\\n            res = preserve_query_results(res, outputs)\\n            try:\\n                atomic_json(progress_path, {\\"task\\": tid, \\"outputs\\": res,\\n                            \\"selected\\": selected_snapshot(ch[tid], res, a.method),\\n                            \\"query\\": index, \\"rank\\": a.rank, \\"stage\\": \\"QUERY_CHECKPOINT\\"})\\n            except OSError as error:\\n                say(event=\\"checkpoint_error\\", error=type(error).__name__)\\n\\n        if ttt is not None:                   # a fresh adapter per task, trained on its demonstrations only\\n            import numpy as np\\n            seed = task_seed(tid, 99)\\n            pairs = [(np.asarray(p[\\"input\\"]), np.asarray(p[\\"output\\"])) for p in ch[tid][\\"train\\"]]\\n            try:\\n                tstats = ttt.train(pairs, a.ttt_steps, seed, deadline=t0 + a.ttt_frac * (task_deadline - t0))\\n            except Exception as e:\\n                torch.cuda.empty_cache()\\n                ttt.reset(seed)                 # back to the base model for this task\\n                tstats = {\\"error\\": type(e).__name__}\\n        for attempt in range(2):\\n            try:\\n                kw = dict(n_gen=a.n_gen, n_score=a.n_score, max_prompt=a.max_prompt, deadline=task_deadline, on_query=save_query)\\n                if attempt:\\n                    kw.update(n_gen=max(1, a.n_gen // 2), n_score=max(1, a.n_score // 2),\\n                              max_prompt=a.max_prompt // 2)\\n                incoming = predict_task(lm, tid, ch[tid], **kw)\\n                res = preserve_query_results(res, incoming)\\n                break\\n            except torch.cuda.OutOfMemoryError as e:\\n                res = preserve_query_results(res, getattr(e, \\"partial_outputs\\", None))\\n                err = \\"oom\\"\\n                torch.cuda.empty_cache()\\n            except Exception as e:\\n                res = preserve_query_results(res, getattr(e, \\"partial_outputs\\", None))\\n                err = traceback.format_exc(limit=3)\\n                torch.cuda.empty_cache()\\n        atomic_json(pool_path, {\\"task\\": tid, \\"outputs\\": res, \\"selected\\": selected_snapshot(ch[tid], res, a.method), \\"error\\": None if res is not None and len(res) == len(ch[tid][\\"test\\"]) else err,\\n                                \\"rank\\": a.rank, \\"wall_s\\": round(time.time() - t0, 2), \\"ttt\\": tstats,\\n                                \\"peak_gb\\": peak_gb(), \\"static_cache\\": lm.use_static})\\n        done += 1\\n        say(event=\\"task\\", task=tid, wall_s=round(time.time() - t0, 2), ok=res is not None and len(res) == len(ch[tid][\\"test\\"]), err=err,\\n            partial=res is not None and len(res) < len(ch[tid][\\"test\\"]),\\n            n_cands=[len(o[\\"candidates\\"]) for o in res] if res else None)\\n    say(event=\\"exit\\", done=done, stats=lm.stats, peak_gb=peak_gb(), static_cache=lm.use_static)\\n\\n\\n# ----- orchestrator ----------------------------------------------------------------------------------------------\\ndef build_submission(ch, out, method, selector=None):\\n    sub, n_pool = {}, 0\\n    for tid, t in ch.items():\\n        p = os.path.join(out, \\"pools\\", f\\"{tid}.json\\")\\n        res = None\\n        if os.path.exists(p):\\n            try:\\n                res = json.load(open(p)).get(\\"outputs\\")\\n            except (OSError, ValueError):\\n                res = None\\n        entry = []\\n        for j, x in enumerate(t[\\"test\\"]):\\n            pool = res[j][\\"candidates\\"] if res and j < len(res) else []\\n            if pool:\\n                n_pool += 1\\n            order = selector(tid, j, pool) if selector else S.rank(pool, method)\\n            entry.append(S.two_attempts(pool, x[\\"input\\"], order))\\n        sub[tid] = entry\\n    return sub, n_pool\\n\\n\\ndef release_failed_worker_claims(out, queue, rank):\\n    \\"\\"\\"Release only unfinished claims owned by a worker already verified dead.\\n\\n    Called by the sole orchestrator after poll() reports a nonzero exit, before\\n    its replacement starts. Completed pools and other workers\' claims stay put.\\n    \\"\\"\\"\\n    released = []\\n    for tid in queue:\\n        if os.path.exists(os.path.join(out, \\"pools\\", f\\"{tid}.json\\")):\\n            continue\\n        path = os.path.join(out, \\"claims\\", tid)\\n        try:\\n            with open(path) as f:\\n                owner = f.read()\\n            if owner == str(rank):\\n                os.unlink(path)\\n                released.append(tid)\\n        except FileNotFoundError:\\n            pass\\n    return released\\n\\n\\ndef orchestrate(a):\\n    t_start = time.time()\\n    ch = json.load(open(a.challenges))\\n    if a.task_ids:\\n        ids = [x for x in a.task_ids.split(\\",\\") if x in ch]\\n        ch = {k: ch[k] for k in ids}\\n    elif a.limit:\\n        ch = {k: ch[k] for k in list(ch)[:a.limit]}\\n    for d in (\\"claims\\", \\"pools\\", \\"workers\\", \\"progress\\"):\\n        os.makedirs(os.path.join(a.out, d), exist_ok=True)\\n    sub_ch = os.path.join(a.out, \\"challenges_used.json\\")\\n    atomic_json(sub_ch, ch)\\n    atomic_json(os.path.join(a.out, \\"queue.json\\"), order_tasks(ch))\\n    sub, _ = build_submission(ch, a.out, a.method)\\n    atomic_json(a.submission, sub)                         # valid fallback file before any GPU work\\n    deadline = t_start + a.deadline_s\\n    worker_deadline = deadline - min(a.final_reserve_s, 0.1 * a.deadline_s)\\n    procs = {}\\n\\n    def spawn(r):\\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(r), PYTHONUNBUFFERED=\\"1\\")\\n        cmd = [sys.executable, \\"-m\\", \\"sf.runner\\", \\"worker\\", \\"--rank\\", str(r), \\"--challenges\\", sub_ch,\\n               \\"--model\\", a.model, \\"--out\\", a.out, \\"--deadline\\", str(worker_deadline), \\"--n-gen\\", str(a.n_gen),\\n               \\"--n-score\\", str(a.n_score), \\"--max-prompt\\", str(a.max_prompt), \\"--task-cap-s\\", str(a.task_cap_s),\\n               \\"--reserve-s\\", str(a.reserve_s), \\"--n-workers\\", str(a.gpus), \\"--method\\", a.method, \\"--ttt-steps\\", str(a.ttt_steps),\\n               \\"--ttt-rank\\", str(a.ttt_rank), \\"--ttt-alpha\\", str(a.ttt_alpha), \\"--ttt-lr\\", str(a.ttt_lr),\\n               \\"--ttt-max-tokens\\", str(a.ttt_max_tokens), \\"--ttt-frac\\", str(a.ttt_frac)]\\n        errf = open(os.path.join(a.out, \\"workers\\", f\\"{r}.stderr\\"), \\"a\\")\\n        return subprocess.Popen(cmd, env=env, stdout=errf, stderr=errf, cwd=a.code_dir)\\n    for r in range(a.gpus):\\n        procs[r] = spawn(r)\\n    restarts = {r: 0 for r in procs}\\n    status = {\\"start\\": t_start, \\"deadline\\": deadline}\\n    while True:\\n        time.sleep(a.poll_s)\\n        n_done = sum(1 for f in os.listdir(os.path.join(a.out, \\"pools\\")) if f.endswith(\\".json\\"))\\n        sub, n_pool = build_submission(ch, a.out, a.method)\\n        atomic_json(a.submission, sub)\\n        alive = {r: p.poll() is None for r, p in procs.items()}\\n        for r, ok in alive.items():\\n            if not ok and procs[r].returncode != 0 and restarts[r] < 2 and time.time() < worker_deadline - 120:\\n                release_failed_worker_claims(a.out, order_tasks(ch), r)\\n                restarts[r] += 1\\n                procs[r] = spawn(r)\\n                alive[r] = True\\n        status.update(t=time.time(), pools=n_done, outputs_with_pool=n_pool, alive=alive, restarts=restarts)\\n        atomic_json(os.path.join(a.out, \\"status.json\\"), status)\\n        if n_done >= len(ch) or not any(alive.values()):\\n            break\\n        if time.time() > deadline:\\n            status[\\"watchdog\\"] = \\"deadline reached; workers stopped\\"\\n            break\\n    for p in procs.values():\\n        if p.poll() is None:\\n            p.send_signal(signal.SIGTERM)\\n    for p in procs.values():\\n        try:\\n            p.wait(timeout=30)\\n        except subprocess.TimeoutExpired:\\n            p.kill()\\n    sub, n_pool = build_submission(ch, a.out, a.method)\\n    atomic_json(a.submission, sub)\\n    problems = S.check_submission(sub, ch)\\n    status.update(t=time.time(), wall_s=round(time.time() - t_start, 1), final_outputs_with_pool=n_pool,\\n                  schema_problems=problems[:20], n_schema_problems=len(problems))\\n    atomic_json(os.path.join(a.out, \\"status.json\\"), status)\\n    print(json.dumps({k: status[k] for k in (\\"wall_s\\", \\"final_outputs_with_pool\\", \\"n_schema_problems\\")}))\\n    return status\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"mode\\", choices=[\\"orchestrate\\", \\"worker\\"])\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--model\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--submission\\")\\n    ap.add_argument(\\"--gpus\\", type=int, default=4)\\n    ap.add_argument(\\"--rank\\", type=int, default=0)\\n    ap.add_argument(\\"--deadline\\", type=float)\\n    ap.add_argument(\\"--deadline-s\\", type=float, default=11.5 * 3600)\\n    ap.add_argument(\\"--final-reserve-s\\", type=float, default=600)\\n    ap.add_argument(\\"--reserve-s\\", type=float, default=300)\\n    ap.add_argument(\\"--task-cap-s\\", type=float, default=900)\\n    ap.add_argument(\\"--n-gen\\", type=int, default=8)\\n    ap.add_argument(\\"--n-score\\", type=int, default=16)\\n    ap.add_argument(\\"--max-prompt\\", type=int, default=12000)\\n    ap.add_argument(\\"--n-workers\\", type=int, default=4)\\n    ap.add_argument(\\"--poll-s\\", type=float, default=20)\\n    ap.add_argument(\\"--limit\\", type=int, default=0)\\n    ap.add_argument(\\"--task-ids\\", default=\\"\\")\\n    ap.add_argument(\\"--method\\", default=\\"vote_score\\")\\n    ap.add_argument(\\"--ttt-steps\\", type=int, default=0)\\n    ap.add_argument(\\"--ttt-rank\\", type=int, default=64)\\n    ap.add_argument(\\"--ttt-alpha\\", type=int, default=64)\\n    ap.add_argument(\\"--ttt-lr\\", type=float, default=1e-4)\\n    ap.add_argument(\\"--ttt-max-tokens\\", type=int, default=8192)\\n    ap.add_argument(\\"--ttt-frac\\", type=float, default=0.7)\\n    ap.add_argument(\\"--code-dir\\", default=os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\\n    a = ap.parse_args(argv)\\n    if a.mode == \\"worker\\":\\n        worker(a)\\n    else:\\n        orchestrate(a)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sf/select.py": "\\"\\"\\"Selection of two attempts from a fixed candidate pool, and the submission schema check.\\n\\nNeural rules (no CORA): votes = number of decoding augmentations that produced the candidate; score = mean\\nteacher-forced log-probability over the first k scoring augmentations.\\n\\"\\"\\"\\nimport numpy as np\\n\\nfrom . import grids as G\\n\\n\\ndef mean_score(c, k=None):\\n    s = c.get(\\"scores\\") or []\\n    s = s[:k] if k else s\\n    return float(np.mean(s)) if s else float(\\"-inf\\")\\n\\n\\ndef rank(pool, method=\\"vote_score\\", k=None):\\n    \\"\\"\\"Candidate indices, best first. Ties are broken by the generation order (stable).\\"\\"\\"\\n    ix = list(range(len(pool)))\\n    if method == \\"vote_score\\":\\n        key = lambda i: (-pool[i][\\"votes\\"], -mean_score(pool[i], k))\\n    elif method == \\"score\\":\\n        key = lambda i: (-mean_score(pool[i], k),)\\n    elif method == \\"vote_gen\\":\\n        key = lambda i: (-pool[i][\\"votes\\"], -max(pool[i][\\"gen_logp\\"]))\\n    elif method == \\"score_plus_logvotes\\":\\n        key = lambda i: (-(mean_score(pool[i], k) + 2.0 * np.log(pool[i][\\"votes\\"] + 1e-9)),)\\n    else:\\n        raise ValueError(method)\\n    return sorted(ix, key=key)\\n\\n\\ndef fallback_grids(test_input):\\n    g = G.to_list(np.asarray(test_input))\\n    return [g, g]\\n\\n\\ndef two_attempts(pool, test_input, order):\\n    picks = [pool[i][\\"grid\\"] for i in order[:2]]\\n    fb = fallback_grids(test_input)\\n    while len(picks) < 2:\\n        picks.append(fb[len(picks)])\\n    return {\\"attempt_1\\": picks[0], \\"attempt_2\\": picks[1]}\\n\\n\\ndef check_submission(sub, challenges):\\n    \\"\\"\\"Schema check against the challenge file: every task, one entry per test input, two valid grids each.\\"\\"\\"\\n    problems = []\\n    if not isinstance(sub, dict):\\n        return [\\"submission is not a dict\\"]\\n    for tid, t in challenges.items():\\n        e = sub.get(tid)\\n        if not isinstance(e, list) or len(e) != len(t[\\"test\\"]):\\n            problems.append(f\\"{tid}: wrong entry count\\")\\n            continue\\n        for j, a in enumerate(e):\\n            if not isinstance(a, dict) or set(a) != {\\"attempt_1\\", \\"attempt_2\\"}:\\n                problems.append(f\\"{tid}[{j}]: keys\\")\\n                continue\\n            for k in (\\"attempt_1\\", \\"attempt_2\\"):\\n                if not G.is_valid_grid(a[k]):\\n                    problems.append(f\\"{tid}[{j}].{k}: invalid grid\\")\\n    extra = set(sub) - set(challenges)\\n    if extra:\\n        problems.append(f\\"{len(extra)} unknown task ids\\")\\n    return problems\\n", "sf/textfmt.py": "\\"\\"\\"The checkpoint\'s grid text format, as found empirically (probes/format_probe.py; PROGRESS.md, 26 Oct).\\n\\nThe vocabulary has 16 tokens: digits 0-9 are ids 0-9, newline 10, \\"user\\" 11, \\"assistant\\" 12, <|endoftext|> 13,\\n<|im_start|> 14, <|im_end|> 15. A task is a multi-turn chat: each demonstration input is a user turn and its output\\nthe following assistant turn; the test input is the last user turn and the answer is generated as the assistant turn.\\nInside a turn a grid is its rows of digits joined by newlines, with no trailing newline.\\n\\"\\"\\"\\nimport numpy as np\\n\\nNL, USER, ASSIST, EOT, IMS, IME = 10, 11, 12, 13, 14, 15\\nDIGITS = tuple(range(10))\\nMAX_SIDE = 30\\nMAX_OUT_TOKENS = MAX_SIDE * (MAX_SIDE + 1)        # 30 rows of 30 digits, 29 newlines, the end token\\n\\n\\ndef grid_ids(g):\\n    out = []\\n    for r, row in enumerate(np.asarray(g).tolist()):\\n        if r:\\n            out.append(NL)\\n        out.extend(int(v) for v in row)\\n    return out\\n\\n\\ndef user_turn(g):\\n    return [IMS, USER, NL] + grid_ids(g) + [IME, NL]\\n\\n\\ndef assistant_turn(g):\\n    return [IMS, ASSIST, NL] + grid_ids(g) + [IME, NL]\\n\\n\\nANSWER_PREFIX = [IMS, ASSIST, NL]\\n\\n\\ndef prompt_ids(pairs, test_input):\\n    ids = []\\n    for i, o in pairs:\\n        ids += user_turn(i) + assistant_turn(o)\\n    return ids + user_turn(test_input) + ANSWER_PREFIX\\n\\n\\ndef target_ids(g):\\n    \\"\\"\\"The tokens the model should emit for grid g: its rows, then the end-of-turn token.\\"\\"\\"\\n    return grid_ids(g) + [IME]\\n\\n\\ndef parse_ids(ids):\\n    \\"\\"\\"Tokens of a generated answer (end token optional) to a grid, or None if they are not a valid grid.\\"\\"\\"\\n    rows, cur = [], []\\n    for t in ids:\\n        if t == IME:\\n            break\\n        if t == NL:\\n            rows.append(cur)\\n            cur = []\\n        elif 0 <= t <= 9:\\n            cur.append(int(t))\\n        else:\\n            return None\\n    rows.append(cur)\\n    if not rows or not rows[0] or len(rows) > MAX_SIDE:\\n        return None\\n    w = len(rows[0])\\n    if w > MAX_SIDE or any(len(r) != w for r in rows):\\n        return None\\n    return rows\\n\\n\\nclass GridConstraint:\\n    \\"\\"\\"Per-sequence state for constrained decoding: every emitted sequence is a rectangular grid of at most 30 x 30\\n    followed by the end token. allowed() returns the set of permitted next tokens.\\"\\"\\"\\n\\n    __slots__ = (\\"width\\", \\"row_len\\", \\"n_rows\\", \\"done\\")\\n\\n    def __init__(self):\\n        self.width, self.row_len, self.n_rows, self.done = None, 0, 0, False\\n\\n    def allowed(self):\\n        if self.done:\\n            return ()\\n        if self.width is None:\\n            if self.row_len == 0:\\n                return DIGITS\\n            if self.row_len >= MAX_SIDE:\\n                return (NL, IME)\\n            return DIGITS + (NL, IME)\\n        if self.row_len < self.width:\\n            return DIGITS\\n        if self.n_rows + 1 >= MAX_SIDE:\\n            return (IME,)\\n        return (NL, IME)\\n\\n    def step(self, t):\\n        if t == IME:\\n            self.done = True\\n        elif t == NL:\\n            if self.width is None:\\n                self.width = self.row_len\\n            self.n_rows += 1\\n            self.row_len = 0\\n        else:\\n            self.row_len += 1\\n", "sf/ttt.py": "\\"\\"\\"Per-puzzle LoRA test-time training (pilot; authorized 26 Oct for Kaggle L4x4 only).\\n\\nOwn implementation on peft (Apache-2.0). One LoRA wrapper is created around the frozen base; before each puzzle its\\nadapter weights are re-initialised (A: Kaiming-uniform, B: zero, i.e. peft\'s default fresh adapter, so the adapted\\nmodel starts exactly at the base) with a puzzle-specific seed, and a new optimiser is built. Training uses only\\nthat puzzle\'s visible demonstrations: each sequence is a leave-one-demonstration-out chat (the other\\ndemonstrations, then the held-out one), under one pair-consistent augmentation (a D8 symmetry and a colour\\npermutation keeping 0 applied to every grid of the sequence, and a shuffled demonstration order). The loss is the\\ncross-entropy of every demonstration output after the first (each is predicted from the pairs before it).\\nTest inputs and their answers are never used.\\n\\"\\"\\"\\nimport importlib\\nimport math\\nimport random\\nimport time\\n\\nimport numpy as np\\nimport torch\\nfrom peft import LoraConfig, get_peft_model\\n\\nfrom . import grids as G\\nfrom . import textfmt as F\\n\\nTARGETS = [\\"q_proj\\", \\"k_proj\\", \\"v_proj\\", \\"o_proj\\", \\"gate_proj\\", \\"up_proj\\", \\"down_proj\\"]\\n\\n\\ndef _disable_torchao():\\n    \\"\\"\\"peft probes for torchao when it dispatches LoRA layers and raises on the old torchao of the Kaggle image;\\n    torchao is not used here (no quantisation), so the probe is answered False.\\"\\"\\"\\n    for mod in (\\"peft.import_utils\\", \\"peft.tuners.lora.torchao\\", \\"peft.utils.quantization_utils\\"):\\n        try:\\n            m = importlib.import_module(mod)\\n        except Exception:\\n            continue\\n        if hasattr(m, \\"is_torchao_available\\"):\\n            m.is_torchao_available = lambda: False\\n\\n\\ndef lora_modules(model):\\n    for m in model.modules():\\n        if hasattr(m, \\"lora_A\\") and \\"default\\" in getattr(m, \\"lora_A\\", {}):\\n            yield m\\n\\n\\nclass TTT:\\n    def __init__(self, lm, rank=64, alpha=64, lr=1e-4, max_tokens=8192, grad_ckpt=True, targets=None):\\n        self.lm, self.rank, self.alpha, self.lr, self.max_tokens = lm, rank, alpha, lr, max_tokens\\n        _disable_torchao()\\n        cfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.0, bias=\\"none\\", task_type=\\"CAUSAL_LM\\",\\n                         target_modules=targets or TARGETS)\\n        self.model = get_peft_model(lm.model, cfg)\\n        lm.model = self.model                              # inference now runs through the adapter\\n        if grad_ckpt:\\n            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\\"use_reentrant\\": False})\\n        self.params = [p for p in self.model.parameters() if p.requires_grad]\\n        self.n_params = sum(p.numel() for p in self.params)\\n        self.model.eval()\\n\\n    def reset(self, seed):\\n        \\"\\"\\"A fresh adapter: the adapted model equals the frozen base again.\\"\\"\\"\\n        g = torch.Generator().manual_seed(seed)\\n        for p in self.params:                              # no gradient survives an interrupted step\\n            p.grad = None\\n        with torch.no_grad():\\n            for m in lora_modules(self.model):\\n                a, b = m.lora_A[\\"default\\"].weight, m.lora_B[\\"default\\"].weight\\n                bound = 1.0 / math.sqrt(a.shape[1])        # kaiming_uniform_(a=sqrt(5)) bound for fan_in\\n                a.copy_(torch.empty(a.shape).uniform_(-bound, bound, generator=g).to(a.dtype))\\n                b.zero_()\\n\\n    def examples(self, pairs, seed):\\n        \\"\\"\\"An endless deterministic stream of (ids, labels) training sequences from the demonstrations.\\"\\"\\"\\n        rng = random.Random(seed)\\n        n = len(pairs)\\n        k = 0\\n        while True:\\n            hold = k % n\\n            k += 1\\n            t = rng.randrange(8)\\n            perm = G.random_perm(rng, keep_zero=True)\\n            aug = G.Aug(t, perm)\\n            ctx = [j for j in range(n) if j != hold]\\n            rng.shuffle(ctx)\\n            order = ctx + [hold]\\n            seq = [(aug.fwd(pairs[j][0]), aug.fwd(pairs[j][1])) for j in order]\\n            ids, labels = self._encode(seq)\\n            while len(ids) > self.max_tokens and len(seq) > 1:\\n                longest = max(range(len(seq) - 1), key=lambda i: len(F.grid_ids(seq[i][0])) +\\n                              len(F.grid_ids(seq[i][1])))\\n                seq.pop(longest)\\n                ids, labels = self._encode(seq)\\n            yield ids, labels\\n\\n    @staticmethod\\n    def _encode(seq):\\n        \\"\\"\\"Chat tokens of the pairs; labels on every output after the first (on the only one if it is alone).\\"\\"\\"\\n        ids, labels = [], []\\n        for idx, (i, o) in enumerate(seq):\\n            u = F.user_turn(i)\\n            ids += u\\n            labels += [-100] * len(u)\\n            head = [F.IMS, F.ASSIST, F.NL]\\n            body = F.grid_ids(o) + [F.IME]\\n            ids += head + body + [F.NL]\\n            labels += [-100] * len(head) + (body if idx > 0 or len(seq) == 1 else [-100] * len(body)) + [-100]\\n        return ids, labels\\n\\n    def train(self, pairs, steps, seed, deadline=None, log_every=10):\\n        \\"\\"\\"Adapt a fresh adapter on the demonstrations. Returns statistics (loss curve, timing, memory).\\"\\"\\"\\n        self.reset(seed)\\n        opt = torch.optim.AdamW(self.params, lr=self.lr, weight_decay=0.0, betas=(0.9, 0.999))\\n        opt.zero_grad(set_to_none=True)\\n        warm = max(1, steps // 10)\\n\\n        def lr_at(s):\\n            if s < warm:\\n                return (s + 1) / warm\\n            return max(0.0, (steps - s) / max(1, steps - warm))\\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)\\n        dev = self.lm.device\\n        stream = self.examples(pairs, seed)\\n        self.model.train()\\n        t0, tokens, curve, done = time.time(), 0, [], 0\\n        use_amp = self.lm.dtype == torch.bfloat16\\n        try:\\n            for s in range(steps):\\n                if deadline is not None and time.time() > deadline:\\n                    break\\n                ids, labels = next(stream)\\n                x = torch.tensor([ids], device=dev)\\n                y = torch.tensor([labels], device=dev)\\n                with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=use_amp):\\n                    logits = self.model(input_ids=x, use_cache=False).logits[0, :-1].float()\\n                loss = torch.nn.functional.cross_entropy(logits, y[0, 1:], ignore_index=-100)\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(self.params, 1.0)\\n                opt.step()\\n                sched.step()\\n                opt.zero_grad(set_to_none=True)\\n                tokens += len(ids)\\n                done += 1\\n                if s % log_every == 0 or s == steps - 1:\\n                    curve.append((s, round(float(loss.detach()), 4)))\\n        finally:\\n            self.model.eval()\\n            del opt\\n        return {\\"steps\\": done, \\"tokens\\": tokens, \\"train_s\\": round(time.time() - t0, 2), \\"loss_curve\\": curve,\\n                \\"rank\\": self.rank, \\"alpha\\": self.alpha, \\"lr\\": self.lr}\\n"}')
for name, text in FILES.items():
    p = os.path.join(CODE_DIR, name)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w', encoding='utf-8').write(text)
print('code written', CODE_SHA, len(FILES))

open(os.path.join(CODE_DIR, 'deployment_guard.py'), 'w').write('"""Stdlib parent guard: fallback before model discovery, checkpoint salvage, wall reserve.\n\nOnly the newly spawned child session is supervised. No model, answer file,\nsignal-timer nesting, or neural selection code runs in this parent.\n"""\nfrom copy import deepcopy\nfrom dataclasses import dataclass\nimport fcntl\nimport glob\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport signal\nimport subprocess\nimport time\nimport threading\n\n\n_TERMINATION_SCOPE_ACTIVE = False\n\n\nclass TerminationRequest:\n    """Scoped cooperative SIGTERM, not an asynchronous exception or exit replay.\n\n    The previous handler is restored on exit but is not called for a signal\n    received here. No masks/preexec hooks are inherited by the child. A request\n    inside Popen waits until its return assigns the owned process before cleanup.\n    SIGINT retains the caller\'s existing behavior. Main-thread, nonnested use.\n    """\n    def __init__(self):\n        self.signum = None\n\n    @property\n    def requested(self):\n        return self.signum is not None\n\n    def _record(self, signum, frame):\n        # No locks, I/O, raising, or previous-handler calls in the signal handler.\n        if self.signum is None:\n            self.signum = int(signum)\n\n    def __enter__(self):\n        global _TERMINATION_SCOPE_ACTIVE\n        if threading.current_thread() is not threading.main_thread():\n            raise RuntimeError("guardian SIGTERM scope requires the main thread")\n        if _TERMINATION_SCOPE_ACTIVE:\n            raise RuntimeError("nested guardian SIGTERM scopes are not supported")\n        self.signum = None\n        self.previous = signal.getsignal(signal.SIGTERM)\n        signal.signal(signal.SIGTERM, self._record)\n        _TERMINATION_SCOPE_ACTIVE = True\n        return self\n\n    def __exit__(self, exc_type, exc_value, traceback):\n        global _TERMINATION_SCOPE_ACTIVE\n        try:\n            signal.signal(signal.SIGTERM, self.previous)\n        finally:\n            _TERMINATION_SCOPE_ACTIVE = False\n        return False\n\n\ndef valid_grid(g):\n    return (type(g) is list and 1 <= len(g) <= 30 and type(g[0]) is list\n            and 1 <= len(g[0]) <= 30 and all(type(r) is list and len(r) == len(g[0])\n            and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g))\n\n\ndef valid_pair(pair):\n    return (type(pair) is dict and set(pair) == {"attempt_1", "attempt_2"}\n            and all(valid_grid(v) for v in pair.values()))\n\n\ndef fallback(challenges):\n    sub = {}\n    for tid, task in challenges.items():\n        if not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", tid):\n            raise ValueError("unsafe task identifier")\n        if not isinstance(task.get("test"), list) or not task["test"]:\n            raise ValueError("missing test inputs")\n        sub[tid] = []\n        for item in task["test"]:\n            g = item.get("input")\n            if not valid_grid(g):\n                raise ValueError("invalid challenge input")\n            sub[tid].append({"attempt_1": deepcopy(g), "attempt_2": deepcopy(g)})\n    if not sub:\n        raise ValueError("empty challenges")\n    return sub\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + f".tmp.{os.getpid()}")\n    try:\n        with temp.open("w") as f:\n            json.dump(value, f, allow_nan=False)\n            f.flush()\n            os.fsync(f.fileno())\n        os.replace(temp, path)\n    finally:\n        temp.unlink(missing_ok=True)\n\n\ndef read_json(path):\n    try:\n        path = Path(path)\n        if path.stat().st_size > 64 * 1024 * 1024:\n            return None\n        return json.loads(path.read_text())\n    except (OSError, ValueError, RecursionError):\n        return None\n\n\ndef find_challenges(kaggle_root, rehearse=False):\n    filename = "arc-agi_evaluation_challenges.json" if rehearse else "arc-agi_test_challenges.json"\n    for directory in ("input/competitions/arc-prize-2026-arc-agi-2", "input/arc-prize-2026-arc-agi-2"):\n        path = Path(kaggle_root) / directory / filename\n        if path.is_file():\n            return path\n    hits = sorted(glob.glob(str(Path(kaggle_root) / "input" / "**" / filename), recursive=True))\n    if not hits:\n        raise FileNotFoundError("competition challenge file not found")\n    return Path(hits[0])\n\n\ndef salvage(challenges, child_submission, run_dir):\n    """Validate each query independently. Checkpoints contain already-ranked pairs.\n\n    Precedence: completed task checkpoint, in-flight query checkpoint, child\n    submission, fallback. A malformed query cannot erase valid other queries.\n    """\n    result = fallback(challenges)\n    child = read_json(child_submission)\n    if type(child) is dict:\n        for tid, entries in result.items():\n            incoming = child.get(tid)\n            if type(incoming) is list:\n                for j in range(min(len(entries), len(incoming))):\n                    if valid_pair(incoming[j]):\n                        entries[j] = incoming[j]\n    recovered = 0\n    for tid, entries in result.items():\n        for folder in ("progress", "pools"):\n            record = read_json(Path(run_dir) / folder / (tid + ".json"))\n            if type(record) is not dict or record.get("task") != tid:\n                continue\n            selected = record.get("selected")\n            if type(selected) is not list:\n                continue\n            for j in range(min(len(entries), len(selected))):\n                if valid_pair(selected[j]):\n                    entries[j] = selected[j]\n        recovered += sum(result[tid][j] != fallback({tid: challenges[tid]})[tid][j]\n                         for j in range(len(entries)))\n    return result, recovered\n\n\n@dataclass(frozen=True)\nclass Budget:\n    start: float\n    total_s: float\n    serialization_s: float\n    shutdown_s: float\n\n    def __post_init__(self):\n        if (any(not math.isfinite(x) or x < 0 for x in\n                (self.start, self.total_s, self.serialization_s, self.shutdown_s))\n                or self.total_s <= self.serialization_s + self.shutdown_s):\n            raise ValueError("invalid parent runtime budget")\n\n    def work_left(self, now=None):\n        now = time.monotonic() if now is None else now\n        return max(0.0, self.start + self.total_s - self.serialization_s - self.shutdown_s - now)\n\n    def task_seconds(self, outstanding, workers, task_cap=900, now=None):\n        # Planning helper; the outer process guard remains the global wall bound.\n        left = self.work_left(now)\n        return max(0.0, min(task_cap, left, left * workers / max(1, outstanding)))\n\n\ndef stop_owned_session(process, grace_s):\n    """Terminate only the session created by this Popen(start_new_session=True).\n\n    The session is exclusively owned for this run. No name matching or system\n    process search is used. Process groups do not include the parent/notebook.\n    """\n    try:\n        if process.poll() is None and os.getpgid(process.pid) != process.pid:\n            raise RuntimeError("owned child lost its separate process group")\n    except ProcessLookupError:\n        # The leader can exit after poll(); its descendants still occupy our\n        # separately created group and must reach the cleanup below.\n        pass\n    # A session remains ours while any of its workers survive the leader. Its\n    # identifier cannot be allocated to a new process while those members live.\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        process.wait(timeout=max(1, grace_s))\n        return\n    until = time.monotonic() + grace_s\n    while time.monotonic() < until:\n        process.poll()  # reap a finished leader; workers can outlive it\n        try:\n            os.killpg(process.pid, 0)\n        except ProcessLookupError:\n            return\n        time.sleep(min(.02, max(0, until - time.monotonic())))\n    try:\n        os.killpg(process.pid, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    process.wait(timeout=max(1, grace_s))\n\n\ndef supervise(challenges, submission, run_dir, command, *, budget, env=None,\n              cwd=None, poll_s=1.0, terminate_grace_s=5.0):\n    """Prewrite fallback, supervise a fresh session, preserve checkpoints at exit.\n\n    This is a local tested safety layer, not an OS or Kaggle runtime certificate.\n    If this parent is killed, its last atomically written submission still exists.\n    """\n    submission, run_dir = Path(submission), Path(run_dir)\n    submission.parent.mkdir(parents=True, exist_ok=True)\n    lease_path = submission.parent / ".submission_guard.lock"\n    process = None\n    start = time.monotonic()\n    with TerminationRequest() as termination, lease_path.open("a+") as lease:\n        fcntl.flock(lease, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        if run_dir.exists():\n            raise FileExistsError("fresh exclusively owned run directory required")\n        run_dir.mkdir(parents=True)\n        atomic_json(submission, fallback(challenges))  # before any model/PEFT imports\n        report = {"status": "FALLBACK_WRITTEN", "child_pid": None,\n                  "run_dir": str(run_dir), "recovered_queries": 0}\n        child_env = dict(os.environ if env is None else env)\n        child_env["SF_RUN_DIR"] = str(run_dir)\n        child_env["SF_CHILD_SUBMISSION"] = str(run_dir / "child_submission.json")\n        original = float(child_env.get("SF_DEADLINE_S", "41400"))\n        child_env["SF_DEADLINE_S"] = str(min(original, budget.work_left()))\n        child_env["PYTHONDONTWRITEBYTECODE"] = "1"\n        try:\n            if termination.requested:\n                report["status"] = "PARENT_SIGTERM"\n            elif budget.work_left() <= 0:\n                report["status"] = "PREPARATION_EXHAUSTED_BUDGET"\n            else:\n                with (run_dir / "child.log").open("wb") as log:\n                    process = subprocess.Popen(command, env=child_env, cwd=cwd,\n                                               stdout=log, stderr=subprocess.STDOUT,\n                                               start_new_session=True)\n                    report.update(status="RUNNING", child_pid=process.pid)\n                    while process.poll() is None and not termination.requested:\n                        sub, recovered = salvage(challenges, submission, run_dir)\n                        atomic_json(submission, sub)\n                        report["recovered_queries"] = recovered\n                        if termination.requested:\n                            break\n                        left = budget.work_left()\n                        if left <= 0:\n                            report["status"] = "PARENT_WALL_LIMIT"\n                            break\n                        time.sleep(min(poll_s, left))\n                    if termination.requested:\n                        report["status"] = "PARENT_SIGTERM"\n                    elif report["status"] == "RUNNING":\n                        report["status"] = "CHILD_EXIT"\n        finally:\n            if process is not None:\n                try:\n                    stop_owned_session(process, min(terminate_grace_s, max(.01, budget.shutdown_s)))\n                except OSError as error:\n                    report["shutdown_error"] = type(error).__name__\n                report["child_returncode"] = process.poll()\n            if termination.requested:\n                report.update(status="PARENT_SIGTERM", termination_signal=termination.signum)\n            sub, recovered = salvage(challenges, submission, run_dir)\n            atomic_json(submission, sub)\n            report.update(recovered_queries=recovered, wall_s=time.monotonic() - start,\n                          remaining_serialization_s=budget.start + budget.total_s - time.monotonic(),\n                          schema_valid=True)\n            if termination.requested:\n                report.update(status="PARENT_SIGTERM", termination_signal=termination.signum)\n            atomic_json(run_dir / "guard_status.json", report)\n        return report\n')


In [ ]:

# CORA side processes (configurations B and H), part 1 of 2: start one CAST side process per constructor package on
# CPU, next to the neural run below. Fail-safe: any failure in this cell only means that the neural output is submitted.
import hashlib, json, os, signal, subprocess, sys, threading, time
B_ENV = {'SF_CAST_END_RESERVE_S': '1800', 'SF_CAST_GRACE_S': '10', 'SF_CAST_NICE': '10', 'SF_CAST_TOTAL_S': '10800'}
B_PACKAGES = [{'name': 'cast_selective_v2', 'entry': 'evaluate', 'mode': 'evidence', 'policy': 'tier1:B3', 'version': 'cast-selective-v2-1', 'cpu_s': '15.0', 'wall_s': '30.0'}]
for _k, _v in B_ENV.items():
    os.environ[_k] = _v
CAST_B = {"procs": [], "packages": [], "error": None, "deadline": None, "t0": time.time(), "ch_path": None}
try:
    _files = json.loads('{"cast_selective_v2/__init__.py": "\\"\\"\\"cast_selective_v2: CAST v1 core modules with genuine leave-one-out evidence and neural-first insertion.\\"\\"\\"\\nfrom .api import POLICIES, VERSION, evaluate, insert_attempt2\\n\\n__all__ = [\\"evaluate\\", \\"insert_attempt2\\", \\"POLICIES\\", \\"VERSION\\"]\\n", "cast_selective_v2/_nd.py": "\\"\\"\\"The four image operations CAST needs, in pure numpy, with scipy used as an accelerator when it imports.\\n\\n    label(mask, structure)          connected components (labels 1..n), 4- or 8-connectivity from a 3x3 structure\\n    binary_dilation(mask, structure) dilation by a 3x3 structure, cells outside the grid count as False\\n    binary_fill_holes(mask)         mask plus every background cell not 4-connected to the border\\n    components_from_edges(n, a, b)  connected components of an undirected graph given as edge arrays\\nCallers never depend on label numbering (objects are sorted afterwards), so both backends give identical results;\\ntests/test_pkg.py checks the two backends against each other. Set CAST_FORCE_NUMPY=1 to disable scipy.\\n\\"\\"\\"\\nimport os\\n\\nimport numpy as np\\n\\n_SCIPY = None\\nif os.environ.get(\\"CAST_FORCE_NUMPY\\") != \\"1\\":\\n    try:                                                     # optional accelerator\\n        from scipy import ndimage as _sn\\n        from scipy.sparse import coo_matrix as _coo\\n        from scipy.sparse.csgraph import connected_components as _cc\\n        _SCIPY = True\\n    except Exception:                                        # noqa: BLE001\\n        _SCIPY = None\\n\\n\\ndef _offsets(structure):\\n    s = np.asarray(structure, dtype=bool)\\n    return [(dr - 1, dc - 1) for dr in range(3) for dc in range(3) if s[dr, dc] and (dr, dc) != (1, 1)]\\n\\n\\ndef label(mask, structure=None):\\n    mask = np.asarray(mask, dtype=bool)\\n    if structure is None:\\n        structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\\n    if _SCIPY:\\n        return _sn.label(mask, structure=structure)\\n    H, W = mask.shape\\n    lab = np.zeros((H, W), dtype=np.int32)\\n    offs = _offsets(structure)\\n    n = 0\\n    for r0 in range(H):\\n        for c0 in range(W):\\n            if not mask[r0, c0] or lab[r0, c0]:\\n                continue\\n            n += 1\\n            lab[r0, c0] = n\\n            stack = [(r0, c0)]\\n            while stack:\\n                r, c = stack.pop()\\n                for dr, dc in offs:\\n                    rr, cc = r + dr, c + dc\\n                    if 0 <= rr < H and 0 <= cc < W and mask[rr, cc] and not lab[rr, cc]:\\n                        lab[rr, cc] = n\\n                        stack.append((rr, cc))\\n    return lab, n\\n\\n\\ndef binary_dilation(mask, structure=None):\\n    mask = np.asarray(mask, dtype=bool)\\n    if structure is None:\\n        structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\\n    if _SCIPY:\\n        return _sn.binary_dilation(mask, structure=structure)\\n    out = mask.copy()\\n    H, W = mask.shape\\n    for dr, dc in _offsets(structure):\\n        sh = np.zeros_like(mask)\\n        r0, r1 = max(0, dr), min(H, H + dr)\\n        c0, c1 = max(0, dc), min(W, W + dc)\\n        sh[r0:r1, c0:c1] = mask[r0 - dr:r1 - dr, c0 - dc:c1 - dc]\\n        out |= sh\\n    return out\\n\\n\\ndef binary_fill_holes(mask):\\n    mask = np.asarray(mask, dtype=bool)\\n    if _SCIPY:\\n        return _sn.binary_fill_holes(mask)\\n    comp = ~mask\\n    lab, n = label(comp)\\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist()) - {0}\\n    holes = comp & ~np.isin(lab, list(border))\\n    return mask | holes\\n\\n\\ndef components_from_edges(n, rows, cols):\\n    rows, cols = np.asarray(rows), np.asarray(cols)\\n    if _SCIPY:\\n        g = _coo((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(n, n))\\n        return _cc(g, directed=False)\\n    parent = np.arange(n)\\n\\n    def find(x):\\n        while parent[x] != x:\\n            parent[x] = parent[parent[x]]\\n            x = parent[x]\\n        return x\\n    for a, b in zip(rows.tolist(), cols.tolist()):\\n        ra, rb = find(a), find(b)\\n        if ra != rb:\\n            parent[max(ra, rb)] = min(ra, rb)\\n    roots = np.array([find(i) for i in range(n)])\\n    _, lab = np.unique(roots, return_inverse=True)\\n    return int(lab.max()) + 1 if n else 0, lab\\n", "cast_selective_v2/absdom.py": "\\"\\"\\"M3 abstract domain: a reduced product of cheap domains over sets of grids.\\n\\nAn abstract value `Abs` describes a SET of grids (its concretization gamma):\\n    shapes    finite set of (h, w) the grid may have, or None = any shape with h >= shape_lo[0], w >= shape_lo[1]\\n    cells     (only when exactly one shape is possible) an (h, w) array of 10-bit masks: the colours cell p may take.\\n              This one domain subsumes the write footprint (cells whose mask is not the singleton of the reference\\n              grid\'s value may change; all others are protected) and the per-cell palette.\\n    pal       may-palette: every colour that occurs is in pal\\n    need      must-palette: every colour in need occurs\\n    lo, hi    per-colour cell-count intervals\\ngamma(A) = {y : shape(y) in shapes (or >= shape_lo), y[p] in cells[p] for all p, colours(y) <= pal,\\n            need <= colours(y), lo[k] <= count_k(y) <= hi[k]}.\\n`bottom` is the empty set. join over-approximates the union; meet is exact on each component (the product is\\nreduced by `normalize`, which only removes grids that cannot be in gamma).\\n\\nSoundness obligation of every operator contract (cast/ops.py): for an abstract input A and parameter theta,\\n    {op(theta, z) : z in gamma(A), op defined on z} is a subset of gamma(fwd(A, theta)),\\nand for a target y, {w : op(theta, w) = y} is a subset of gamma(bwd(y, theta)). Tests: tests/test_contracts_*.py.\\n\\"\\"\\"\\nimport numpy as np\\n\\nALL = (1 << 10) - 1\\nMAXC = 900\\nBITS = np.array([1 << k for k in range(10)], dtype=np.uint16)\\nPOPC = np.array([bin(i).count(\\"1\\") for i in range(1 << 10)], dtype=np.int64)\\n\\n\\ndef bit(c):\\n    return 1 << int(c)\\n\\n\\ndef mask_of(colours):\\n    m = 0\\n    for c in colours:\\n        m |= 1 << int(c)\\n    return m\\n\\n\\ndef colours_of(m):\\n    return [k for k in range(10) if m >> k & 1]\\n\\n\\nclass Abs:\\n    __slots__ = (\\"shapes\\", \\"shape_lo\\", \\"cells\\", \\"pal\\", \\"need\\", \\"lo\\", \\"hi\\", \\"bottom\\")\\n\\n    def __init__(self, shapes=None, shape_lo=(1, 1), cells=None, pal=ALL, need=0, lo=None, hi=None, bottom=False):\\n        self.shapes = None if shapes is None else frozenset(shapes)\\n        self.shape_lo = tuple(shape_lo)\\n        self.cells = cells\\n        self.pal = int(pal)\\n        self.need = int(need)\\n        self.lo = np.zeros(10, dtype=np.int64) if lo is None else np.asarray(lo, dtype=np.int64).copy()\\n        self.hi = np.full(10, MAXC, dtype=np.int64) if hi is None else np.asarray(hi, dtype=np.int64).copy()\\n        self.bottom = bottom\\n        if not bottom:\\n            normalize(self)\\n\\n    def __repr__(self):\\n        if self.bottom:\\n            return \\"Abs(BOTTOM)\\"\\n        sh = \\"any>=%s\\" % (self.shape_lo,) if self.shapes is None else sorted(self.shapes)[:4]\\n        return (f\\"Abs(shapes={sh}{\'...\' if self.shapes and len(self.shapes) > 4 else \'\'}, \\"\\n                f\\"cells={\'yes\' if self.cells is not None else \'no\'}, pal={colours_of(self.pal)}, \\"\\n                f\\"need={colours_of(self.need)})\\")\\n\\n    def copy(self):\\n        if self.bottom:\\n            return BOTTOM()\\n        return Abs(self.shapes, self.shape_lo, None if self.cells is None else self.cells.copy(), self.pal,\\n                   self.need, self.lo, self.hi)\\n\\n    @property\\n    def single_shape(self):\\n        if self.shapes is not None and len(self.shapes) == 1:\\n            return next(iter(self.shapes))\\n        return None\\n\\n    def footprint(self, ref):\\n        \\"\\"\\"Cells that may differ from the reference grid `ref` (None if not aligned).\\"\\"\\"\\n        if self.cells is None or self.cells.shape != np.asarray(ref).shape:\\n            return None\\n        return self.cells != BITS[np.asarray(ref)]\\n\\n\\ndef BOTTOM():\\n    a = Abs.__new__(Abs)\\n    a.shapes, a.shape_lo, a.cells, a.pal, a.need = frozenset(), (1, 1), None, 0, 0\\n    a.lo, a.hi, a.bottom = np.zeros(10, dtype=np.int64), np.zeros(10, dtype=np.int64), True\\n    return a\\n\\n\\ndef TOP():\\n    return Abs()\\n\\n\\nLOG2 = np.zeros(1 << 10, dtype=np.int64)\\nfor _k in range(10):\\n    LOG2[1 << _k] = _k\\nKS = np.arange(10)\\n\\n\\ndef _bits(m):\\n    return (int(m) >> KS) & 1\\n\\n\\ndef normalize(a):\\n    \\"\\"\\"Tighten the reduced product in place; set bottom when gamma is provably empty.\\"\\"\\"\\n    if a.shapes is not None:\\n        a.shapes = frozenset(s for s in a.shapes if s[0] >= a.shape_lo[0] and s[1] >= a.shape_lo[1]\\n                             and 1 <= s[0] <= 30 and 1 <= s[1] <= 30)\\n        if not a.shapes:\\n            return _bot(a)\\n        if len(a.shapes) > 1:\\n            a.cells = None\\n    if a.cells is not None:\\n        if a.shapes is None or len(a.shapes) != 1 or a.cells.shape != next(iter(a.shapes)):\\n            a.cells = None\\n    if a.cells is not None:\\n        a.cells = (a.cells & np.uint16(a.pal)).astype(np.uint16)\\n        flat = a.cells.ravel().astype(np.int64)\\n        if (flat == 0).any():\\n            return _bot(a)\\n        a.pal &= int(np.bitwise_or.reduce(flat))\\n        single = POPC[flat] == 1\\n        fixed = np.bincount(LOG2[flat[single]], minlength=10)[:10]\\n        may = ((flat[:, None] >> KS) & 1).sum(0)\\n        a.lo = np.maximum(a.lo, fixed)\\n        a.hi = np.minimum(a.hi, may)\\n    a.hi = np.where(_bits(a.pal) == 1, a.hi, 0)\\n    a.lo = np.where(_bits(a.need) == 1, np.maximum(a.lo, 1), a.lo)\\n    pos = np.nonzero(a.lo > 0)[0]\\n    for k in pos:\\n        a.need |= 1 << int(k)\\n    if a.need & ~a.pal or (a.lo > a.hi).any():\\n        return _bot(a)\\n    if a.shapes is not None:\\n        nmax = max(h * w for h, w in a.shapes)\\n        nmin = min(h * w for h, w in a.shapes)\\n        if a.lo.sum() > nmax or a.hi.sum() < nmin:\\n            return _bot(a)\\n    return a\\n\\n\\ndef _bot(a):\\n    b = BOTTOM()\\n    for s in Abs.__slots__:\\n        setattr(a, s, getattr(b, s))\\n    return a\\n\\n\\nBITW = np.array([1 << k for k in range(10)], dtype=np.int64)\\n\\n\\ndef alpha(g):\\n    \\"\\"\\"Exact abstraction of one concrete grid (already reduced; built without normalize).\\"\\"\\"\\n    g = np.asarray(g, dtype=np.int64)\\n    cnt = np.bincount(g.ravel(), minlength=10)[:10]\\n    m = int(BITW[cnt > 0].sum())\\n    a = Abs.__new__(Abs)\\n    a.shapes, a.shape_lo, a.cells = frozenset([g.shape]), (1, 1), BITS[g]\\n    a.pal = a.need = m\\n    a.lo, a.hi, a.bottom = cnt.copy(), cnt.copy(), False\\n    return a\\n\\n\\ndef contains(a, y):\\n    \\"\\"\\"y in gamma(a).\\"\\"\\"\\n    if a.bottom:\\n        return False\\n    y = np.asarray(y, dtype=np.int64)\\n    if a.shapes is not None and y.shape not in a.shapes:\\n        return False\\n    if a.shapes is None and (y.shape[0] < a.shape_lo[0] or y.shape[1] < a.shape_lo[1]):\\n        return False\\n    cnt = np.bincount(y.ravel(), minlength=10)[:10]\\n    cy = int(BITW[cnt > 0].sum())\\n    if cy & ~a.pal or a.need & ~cy:\\n        return False\\n    if (cnt < a.lo).any() or (cnt > a.hi).any():\\n        return False\\n    if a.cells is not None and not ((a.cells & BITS[y]) != 0).all():\\n        return False\\n    return True\\n\\n\\ndef join(a, b):\\n    if a.bottom:\\n        return b.copy()\\n    if b.bottom:\\n        return a.copy()\\n    if a.shapes is None or b.shapes is None:\\n        shapes = None\\n        lo_a = a.shape_lo if a.shapes is None else (min(s[0] for s in a.shapes), min(s[1] for s in a.shapes))\\n        lo_b = b.shape_lo if b.shapes is None else (min(s[0] for s in b.shapes), min(s[1] for s in b.shapes))\\n        shape_lo = (min(lo_a[0], lo_b[0]), min(lo_a[1], lo_b[1]))\\n    else:\\n        shapes = a.shapes | b.shapes\\n        shape_lo = (1, 1)\\n    cells = None\\n    if (a.cells is not None and b.cells is not None and a.cells.shape == b.cells.shape and shapes is not None\\n            and len(shapes) == 1):\\n        cells = a.cells | b.cells\\n    return Abs(shapes, shape_lo, cells, a.pal | b.pal, a.need & b.need, np.minimum(a.lo, b.lo),\\n               np.maximum(a.hi, b.hi))\\n\\n\\ndef meet(a, b):\\n    if a.bottom or b.bottom:\\n        return BOTTOM()\\n    if a.shapes is None and b.shapes is None:\\n        shapes = None\\n    elif a.shapes is None:\\n        shapes = b.shapes\\n    elif b.shapes is None:\\n        shapes = a.shapes\\n    else:\\n        shapes = a.shapes & b.shapes\\n    shape_lo = (max(a.shape_lo[0], b.shape_lo[0]), max(a.shape_lo[1], b.shape_lo[1]))\\n    cells = None\\n    if a.cells is not None and b.cells is not None:\\n        if a.cells.shape != b.cells.shape:\\n            return BOTTOM()\\n        cells = a.cells & b.cells\\n    elif a.cells is not None:\\n        cells = a.cells.copy()\\n    elif b.cells is not None:\\n        cells = b.cells.copy()\\n    return Abs(shapes, shape_lo, cells, a.pal & b.pal, a.need | b.need, np.maximum(a.lo, b.lo),\\n               np.minimum(a.hi, b.hi))\\n\\n\\ndef join_all(items):\\n    out = BOTTOM()\\n    for x in items:\\n        out = join(out, x)\\n    return out\\n\\n\\ndef is_concrete(a):\\n    return (not a.bottom and a.cells is not None and bool((POPC[a.cells] == 1).all()))\\n\\n\\ndef concretize_one(a):\\n    \\"\\"\\"The single grid of a concrete abstract value.\\"\\"\\"\\n    assert is_concrete(a)\\n    return np.log2(a.cells.astype(np.float64)).round().astype(np.int64)\\n\\n\\ndef sample(a, rng, tries=50):\\n    \\"\\"\\"A random member of gamma(a) when cells are known (rejection on the count/need constraints); else None.\\"\\"\\"\\n    if a.bottom or a.cells is None:\\n        return None\\n    opts = [[k for k in range(10) if m >> k & 1] for m in a.cells.ravel()]\\n    for _ in range(tries):\\n        y = np.array([rng.choice(o) for o in opts], dtype=np.int64).reshape(a.cells.shape)\\n        if contains(a, y):\\n            return y\\n    return None\\n", "cast_selective_v2/api.py": "\\"\\"\\"cast_selective_v2: CAST v1 core modules with genuine leave-one-out, disagreement and stability evidence, explicit\\nstatuses, per-call state reset, and a neural-first attempt-2 insertion rule. Does NOT inherit v1\'s confirmation claim.\\n\\n    from cast_selective_v2 import evaluate, insert_attempt2\\n    rep = evaluate(task)                                  # evidence and per-policy acceptance for every query\\n    attempts = insert_attempt2(task, neural, policy=\\"tier0\\")   # neural: [[a1, a2], ...] per query; a1 never changes\\n\\nModules (unchanged from cast_v1, same order): M4 symmetry completion (complete predictions only), M2 typed guarded\\nrewriting, L0 contract-guided search (depth 2, 1,000 executions).\\nEvidence per module, computed only for a module with a complete fitting candidate:\\n  folds          for each demonstration i: the module is re-run from scratch on the other demonstrations (state\\n                 cleared; the held-out output never enters generation, ordering, selection or caches), its SELECTED\\n                 (top-1) prediction on the held-out input is compared with the held-out output, and its predictions on\\n                 the real queries are recorded\\n  genuine_loo    every fold\'s top-1 prediction equals the held-out output (n_demos >= 2)\\n  v1_loo         the cast_v1 rule (held-out output among the fold\'s top 2) - kept for the B1 comparison only\\n  single_demo_folds   n_demos == 2 (each fold synthesised from ONE demonstration: weak evidence, labelled)\\n  query_disagreement  fitting programs of the full run disagree on a query (version-space ambiguity); B3 uses the\\n                      UNION over every module\'s fitting programs (union_disagreement), so a unanimous module is not\\n                      accepted while another module\'s fitting programs predict something else\\n  fold_agreement      every fold predicts the same query outputs as the full run\\n  restart_stable      a restart with the reversed enumeration order (views, parameters, instances) selects the same\\n                      query behaviour\\nPolicies (per query): B1 = complete and v1_loo; B2 = complete and genuine_loo; B3 = B2 and no query disagreement and\\nfold agreement and restart stability. Fewer than two demonstrations: INSUFFICIENT_VALIDATION_EVIDENCE (never accepted).\\nInsertion: attempt 1 is always the neural answer. policy \\"tier0\\": attempt 2 is replaced only where the neural\\nattempt 2 is missing or identical to attempt 1 (dominance-safe under per-query best-of-two scoring), by the first\\nmodule\'s complete fitting candidate if it differs from attempt 1; \\"tier1:B1|B2|B3\\": attempt 2 is replaced wherever\\nthat policy accepts a candidate that differs from attempt 1 (NOT safe: it can lose a correct neural attempt 2).\\nAny timeout, error, malformed input or off-main-thread call leaves the neural attempts unchanged.\\n\\"\\"\\"\\nimport signal\\nimport threading\\nimport time\\n\\nimport numpy as np\\n\\nfrom . import absdom as D\\nfrom . import candidates as CA\\nfrom . import grids as G\\nfrom . import ops as O\\nfrom . import rewrite as RW\\nfrom . import search as SR\\nfrom . import symmetry as S\\n\\nVERSION = \\"cast-selective-v2-1\\"\\nORDER = (\\"m4\\", \\"m2\\", \\"l0\\")\\nPOLICIES = (\\"B1\\", \\"B2\\", \\"B3\\")\\n\\n\\nclass _Timeout(BaseException):\\n    pass\\n\\n\\nclass _FoldTimeout(BaseException):\\n    pass\\n\\n\\ndef _clear():\\n    RW._SCENES.clear()\\n    S._MAP_CACHE.clear()\\n\\n\\n# ------------------------------------------------------------------ modules (cast_v1 semantics; optional reversed order)\\ndef _m4(task, reverse=False):\\n    op = S.SymComplete(0.5, 0.5)\\n    demos = [(np.asarray(x), np.asarray(y)) for x, y in task[\\"train\\"]]\\n    qs = [np.asarray(q) for q in task[\\"test_inputs\\"]]\\n    ctx = O.TaskCtx(task[\\"train\\"], task[\\"test_inputs\\"])\\n    thetas = op.domain(ctx)\\n    if reverse:\\n        thetas = list(reversed(thetas))\\n    fits = []\\n    for th in thetas:\\n        if any(not D.contains(op.fwd(D.alpha(x), th), y) for x, y in demos):\\n            continue\\n        ok = True\\n        for x, y in demos:\\n            out = op.apply(x, th)\\n            if out is None or out.shape != y.shape or not np.array_equal(out, y):\\n                ok = False\\n                break\\n        if not ok:\\n            continue\\n        preds, complete = [], []\\n        for q in qs:\\n            out = op.apply(q, th)\\n            preds.append(out)\\n            complete.append(out is not None and (op.last_trace or {}).get(\\"status\\") == \\"COMPLETE\\")\\n        fits.append({\\"program\\": f\\"(symcomplete mask={th[0]} crop={th[1]})\\", \\"preds\\": preds, \\"complete\\": complete,\\n                     \\"cost\\": 1.0, \\"order\\": len(fits), \\"loo\\": 0.0})\\n    return [f for f in fits if all(f[\\"complete\\"])] or [f for f in fits if any(f[\\"complete\\"])]\\n\\n\\ndef _m2(task, reverse=False):\\n    return RW.search(task, views=tuple(reversed(RW.VIEWS)) if reverse else RW.VIEWS)\\n\\n\\ndef _l0(task, reverse=False):\\n    ctx = O.TaskCtx(task[\\"train\\"], task[\\"test_inputs\\"])\\n    s = SR.Searcher(ctx, O.L0, depth=2, mode=\\"fwdbwd\\", max_exec=1000, oe=True, max_fits=32)\\n    if reverse:\\n        s.instances = list(reversed(s.instances))\\n    r = s.run()\\n    return [{\\"program\\": \\" ; \\".join(f\\"{n}{th!r}\\" for n, th in f[\\"program\\"]),\\n             \\"preds\\": [None if p is None else np.asarray(p) for p in f[\\"preds\\"]],\\n             \\"complete\\": [p is not None for p in f[\\"preds\\"]], \\"cost\\": float(len(f[\\"program\\"])), \\"order\\": i}\\n            for i, f in enumerate(r[\\"fits\\"])]\\n\\n\\nMODULES = {\\"m4\\": _m4, \\"m2\\": _m2, \\"l0\\": _l0}\\n\\n\\ndef _ranked(module, task, reverse=False):\\n    fits = MODULES[module](task, reverse)\\n    demos = [(np.asarray(x), np.asarray(y)) for x, y in task[\\"train\\"]]\\n    qs = [np.asarray(q) for q in task[\\"test_inputs\\"]]\\n    groups = CA.rank(fits, {\\"train\\": demos, \\"test_inputs\\": qs})\\n    return groups, CA.attempts(groups, len(qs)), fits\\n\\n\\ndef _key(g):\\n    return None if g is None else (tuple(np.asarray(g).shape), np.asarray(g).astype(np.int64).tobytes())\\n\\n\\ndef _valid_answer(g, q, changes):\\n    if g is None or not G.valid_grid(np.asarray(g)):\\n        return False\\n    g, q = np.asarray(g), np.asarray(q)\\n    return not (changes and g.shape == q.shape and np.array_equal(g, q))\\n\\n\\nclass _FoldCap:\\n    \\"\\"\\"A nested CPU budget for one fold, carved out of the call\'s remaining CPU budget (ITIMER_VIRTUAL).\\"\\"\\"\\n\\n    def __init__(self, fold_cpu_s):\\n        self.fold_cpu_s = fold_cpu_s\\n\\n    def __enter__(self):\\n        self.remaining = signal.getitimer(signal.ITIMER_VIRTUAL)[0]\\n        if self.fold_cpu_s is not None:\\n            self.old = signal.signal(signal.SIGVTALRM, self._boom)\\n            budget = min(self.remaining, self.fold_cpu_s) if self.remaining > 0 else self.fold_cpu_s\\n            self.t0 = time.process_time()\\n            signal.setitimer(signal.ITIMER_VIRTUAL, budget)\\n        return self\\n\\n    @staticmethod\\n    def _boom(signum, frame):                                  # noqa: ARG004\\n        raise _FoldTimeout()\\n\\n    def __exit__(self, *exc):\\n        if self.fold_cpu_s is not None:\\n            used = time.process_time() - self.t0\\n            signal.setitimer(signal.ITIMER_VIRTUAL, 0)\\n            signal.signal(signal.SIGVTALRM, self.old)\\n            if self.remaining > 0:\\n                left = self.remaining - used\\n                if left <= 0:\\n                    raise _Timeout()\\n                signal.setitimer(signal.ITIMER_VIRTUAL, left)\\n        return False\\n\\n\\ndef _module_evidence(module, t, changes, fold_cpu_s):\\n    groups, atts, fits = _ranked(module, t)\\n    nq = len(t[\\"test_inputs\\"])\\n    ev = {\\"module\\": module, \\"n_fits\\": len(fits), \\"n_behaviours\\": len(groups), \\"status\\": \\"NO_CANDIDATE\\",\\n          \\"candidate\\": [None] * nq, \\"program\\": groups[0][\\"members\\"][0][\\"program\\"] if groups else None}\\n    if not groups:\\n        return ev\\n    top = groups[0][\\"preds\\"]\\n    comp = groups[0][\\"complete\\"]\\n    cand = [top[j] if (j < len(top) and comp and _valid_answer(top[j], t[\\"test_inputs\\"][j], changes)) else None\\n            for j in range(nq)]\\n    ev[\\"candidate\\"] = [None if g is None else np.asarray(g).astype(int).tolist() for g in cand]\\n    if not any(c is not None for c in cand):\\n        ev[\\"status\\"] = \\"INCOMPLETE\\"\\n        return ev\\n    ev[\\"pred_keys\\"] = [sorted({_key(g[\\"preds\\"][j])[1].hex() + str(_key(g[\\"preds\\"][j])[0]) for g in groups\\n                               if j < len(g[\\"preds\\"]) and g[\\"preds\\"][j] is not None}) for j in range(nq)]\\n    ev[\\"query_disagreement\\"] = [len(k) > 1 for k in ev[\\"pred_keys\\"]]\\n    n = len(t[\\"train\\"])\\n    ev[\\"n_demos\\"] = n\\n    if n < 2:\\n        ev[\\"status\\"] = \\"INSUFFICIENT_VALIDATION_EVIDENCE\\"\\n        return ev\\n    folds = []\\n    for i in range(n):\\n        rest = [d for k, d in enumerate(t[\\"train\\"]) if k != i]\\n        sub = {\\"task_id\\": \\"fold\\", \\"train\\": rest, \\"test_inputs\\": [t[\\"train\\"][i][0]] + list(t[\\"test_inputs\\"])}\\n        _clear()                                               # nothing cached across folds\\n        try:\\n            with _FoldCap(fold_cpu_s):\\n                _, fa, _ = _ranked(module, sub)\\n        except _FoldTimeout:\\n            ev[\\"status\\"] = \\"TIMEOUT_IN_FOLD\\"\\n            ev[\\"folds\\"] = folds\\n            return ev\\n        y = np.asarray(t[\\"train\\"][i][1])\\n        top1 = fa[0][0] if fa[0] else None\\n        eq = lambda a: a is not None and np.asarray(a).shape == y.shape and np.array_equal(np.asarray(a), y)  # noqa: E731\\n        fq = [fa[1 + j][0] if fa[1 + j] else None for j in range(nq)]\\n        folds.append({\\"held_out\\": i, \\"top1_correct\\": eq(top1), \\"top2_correct\\": any(eq(a) for a in fa[0][:2]),\\n                      \\"query_agrees\\": [_key(fq[j]) == _key(cand[j]) for j in range(nq)]})\\n    ev[\\"folds\\"] = folds\\n    ev[\\"v1_loo\\"] = all(f[\\"top2_correct\\"] for f in folds)\\n    ev[\\"genuine_loo\\"] = all(f[\\"top1_correct\\"] for f in folds)\\n    ev[\\"single_demo_folds\\"] = n == 2\\n    ev[\\"fold_agreement\\"] = [all(f[\\"query_agrees\\"][j] for f in folds) for j in range(nq)]\\n    _clear()\\n    g2, _, _ = _ranked(module, t, reverse=True)\\n    ev[\\"restart_stable\\"] = bool(g2) and [_key(p) for p in g2[0][\\"preds\\"]] == [_key(p) for p in top]\\n    ev[\\"status\\"] = \\"EVIDENCE\\"\\n    return ev\\n\\n\\ndef _accept(ev, j, policy):\\n    if ev.get(\\"status\\") != \\"EVIDENCE\\" or ev[\\"candidate\\"][j] is None:\\n        return False\\n    if policy == \\"B1\\":\\n        return ev[\\"v1_loo\\"]\\n    if policy == \\"B2\\":\\n        return ev[\\"genuine_loo\\"]\\n    if policy == \\"B3\\":\\n        return (ev[\\"genuine_loo\\"] and not ev[\\"union_disagreement\\"][j] and ev[\\"fold_agreement\\"][j]\\n                and ev[\\"restart_stable\\"])\\n    raise KeyError(policy)\\n\\n\\ndef _norm(task):\\n    train = [(p[\\"input\\"], p[\\"output\\"]) if isinstance(p, dict) else (p[0], p[1]) for p in task[\\"train\\"]]\\n    tests = task.get(\\"test_inputs\\")\\n    if tests is None:\\n        tests = [x[\\"input\\"] if isinstance(x, dict) else x for x in task[\\"test\\"]]\\n    for g in [a for a, _ in train] + [b for _, b in train] + list(tests):\\n        if not G.valid_grid(np.asarray(g)):\\n            raise ValueError(\\"invalid grid\\")\\n    return {\\"task_id\\": \\"task\\", \\"train\\": train, \\"test_inputs\\": list(tests)}\\n\\n\\nclass _Caps:\\n    def __init__(self, cpu_s, wall_s):\\n        self.cpu_s, self.wall_s = cpu_s, wall_s\\n\\n    def __enter__(self):\\n        def boom(signum, frame):                               # noqa: ARG001\\n            raise _Timeout()\\n        self.old = (signal.signal(signal.SIGVTALRM, boom), signal.signal(signal.SIGALRM, boom))\\n        self.old_t = (signal.setitimer(signal.ITIMER_VIRTUAL, self.cpu_s),\\n                      signal.setitimer(signal.ITIMER_REAL, self.wall_s))\\n        return self\\n\\n    def __exit__(self, *exc):\\n        signal.setitimer(signal.ITIMER_VIRTUAL, 0)\\n        signal.setitimer(signal.ITIMER_REAL, 0)\\n        signal.signal(signal.SIGVTALRM, self.old[0])\\n        signal.signal(signal.SIGALRM, self.old[1])\\n        for which, (val, interval) in ((signal.ITIMER_VIRTUAL, self.old_t[0]), (signal.ITIMER_REAL, self.old_t[1])):\\n            if val > 0:\\n                signal.setitimer(which, val, interval)\\n        return False\\n\\n\\ndef evaluate(task, cpu_s=15.0, wall_s=30.0, fold_cpu_s=None):\\n    \\"\\"\\"Evidence for every module and query; never raises. rep[\\"queries\\"][j] = {\\"candidate\\", \\"candidate_module\\",\\n    \\"accepted\\": {policy: grid or None}, \\"accepted_module\\": {policy: module}}.\\"\\"\\"\\n    n_test = len(task.get(\\"test\\", task.get(\\"test_inputs\\", [])) or [])\\n    rep = {\\"version\\": VERSION, \\"status\\": \\"OK\\", \\"n_demos\\": None, \\"modules\\": {},\\n           \\"queries\\": [{\\"candidate\\": None, \\"candidate_module\\": None, \\"accepted\\": {p: None for p in POLICIES},\\n                        \\"accepted_module\\": {p: None for p in POLICIES}} for _ in range(n_test)]}\\n    t0, w0 = time.process_time(), time.time()\\n    if threading.current_thread() is not threading.main_thread():\\n        rep[\\"status\\"] = \\"NO_MAIN_THREAD\\"\\n        return rep\\n    _clear()\\n    try:\\n        with _Caps(cpu_s, wall_s):\\n            t = _norm(task)\\n            rep[\\"n_demos\\"] = len(t[\\"train\\"])\\n            changes = all(np.asarray(x).shape != np.asarray(y).shape or not np.array_equal(x, y) for x, y in t[\\"train\\"])\\n            for module in ORDER:\\n                _clear()\\n                ev = _module_evidence(module, t, changes, fold_cpu_s)\\n                rep[\\"modules\\"][module] = ev\\n                if ev[\\"status\\"] == \\"TIMEOUT_IN_FOLD\\":\\n                    raise _FoldTimeout()\\n            union = [set() for _ in range(n_test)]\\n            for ev in rep[\\"modules\\"].values():\\n                for j, ks in enumerate(ev.get(\\"pred_keys\\") or []):\\n                    union[j] |= set(ks)\\n            rep[\\"union_disagreement\\"] = [len(u) > 1 for u in union]\\n            for module in ORDER:\\n                ev = rep[\\"modules\\"][module]\\n                ev[\\"union_disagreement\\"] = rep[\\"union_disagreement\\"]\\n                for j in range(n_test):\\n                    q = rep[\\"queries\\"][j]\\n                    if q[\\"candidate\\"] is None and ev[\\"candidate\\"][j] is not None:\\n                        q[\\"candidate\\"], q[\\"candidate_module\\"] = ev[\\"candidate\\"][j], module\\n                    for p in POLICIES:\\n                        if q[\\"accepted\\"][p] is None and _accept(ev, j, p):\\n                            q[\\"accepted\\"][p], q[\\"accepted_module\\"][p] = ev[\\"candidate\\"][j], module\\n    except _FoldTimeout:\\n        rep[\\"status\\"] = \\"TIMEOUT_IN_FOLD\\"\\n    except _Timeout:\\n        rep[\\"status\\"] = \\"TIMEOUT\\"\\n    except Exception as e:                                     # noqa: BLE001  fail-safe by contract\\n        rep[\\"status\\"] = f\\"ERROR:{type(e).__name__}\\"\\n    finally:\\n        _clear()\\n    if rep[\\"status\\"] != \\"OK\\":                                  # abstain everywhere on any failure\\n        for q in rep[\\"queries\\"]:\\n            q.update(candidate=None, candidate_module=None, accepted={p: None for p in POLICIES},\\n                     accepted_module={p: None for p in POLICIES})\\n    rep[\\"cpu_s\\"] = round(time.process_time() - t0, 3)\\n    rep[\\"wall_s\\"] = round(time.time() - w0, 3)\\n    return rep\\n\\n\\ndef insert_attempt2(task, neural, policy=\\"tier0\\", report=None, **caps):\\n    \\"\\"\\"Final [[a1, a2], ...] per query. a1 is always neural[j][0]; on any failure the neural attempts are returned.\\"\\"\\"\\n    try:\\n        out = [list(x) + [None] * (2 - len(x)) if x else [None, None] for x in neural]\\n        rep = report if report is not None else evaluate(task, **caps)\\n        if rep[\\"status\\"] != \\"OK\\":\\n            return out\\n        for j, q in enumerate(rep[\\"queries\\"]):\\n            if j >= len(out):\\n                break\\n            a1, a2 = out[j][0], out[j][1]\\n            if policy == \\"tier0\\":\\n                g = q[\\"candidate\\"]\\n                dup = a2 is None or (a1 is not None and _key(a1) == _key(a2))\\n                if dup and g is not None and _key(g) != _key(a1):\\n                    out[j][1] = g\\n            elif policy.startswith(\\"tier1:\\"):\\n                g = q[\\"accepted\\"][policy.split(\\":\\")[1]]\\n                if g is not None and _key(g) != _key(a1):\\n                    out[j][1] = g\\n            else:\\n                raise KeyError(policy)\\n        return out\\n    except Exception:                                          # noqa: BLE001\\n        return [list(x) if x else [None, None] for x in neural]\\n", "cast_selective_v2/candidates.py": "\\"\\"\\"Fitting programs -> ranked behaviours -> two attempts per query, and export in the score-first branch\'s format.\\n\\nSelection rule (pre-declared for every CAST module; frozen 27 Oct before any evaluation run):\\n  1. Only programs that reproduce EVERY demonstration are candidates. Group them by their predictions on the queries\\n     (two programs that fit but disagree on a query stay separate behaviours).\\n  2. Non-vacuity: a query prediction equal to that query\'s input is dropped when every demonstration changes its\\n     input (the synthesis review\'s identity-attempt defect).\\n  3. Rank behaviours by: complete (no undetermined cell on any query) first; then leave-one-demonstration-out\\n     support (higher first); then group size (more agreeing programs first); then program cost (lower first); then\\n     enumeration order. Deterministic; no randomness; no answer access.\\n  4. Attempts per query: the first two distinct valid grids in that order.\\nExport (`to_program_candidates`) follows kaggle/hybrid_2026/llm_writer/program_candidates.py:\\n  {\\"per_test\\": [[{\\"grid\\", \\"tier\\", \\"score\\", \\"certified\\", \\"loo\\", \\"group_size\\", \\"program\\", \\"candidate\\"}, ...], ...],\\n   \\"n_programs\\", \\"n_fits\\", \\"n_groups\\", \\"failures\\"}; tier is \\"CAST_FIT\\" or \\"CAST_PARTIAL\\" (incomplete completion).\\nWhether the score-first entry imports these is the author\'s decision through the SUBMISSION_READY gate.\\n\\"\\"\\"\\nimport numpy as np\\n\\nfrom . import grids as G\\n\\n\\ndef _key(p):\\n    return None if p is None else (tuple(np.asarray(p).shape), np.asarray(p).tobytes())\\n\\n\\ndef rank(fits, task):\\n    \\"\\"\\"fits: [{\\"program\\", \\"preds\\", \\"complete\\": [bool per query], \\"loo\\": float, \\"cost\\": float, \\"order\\": int}]\\"\\"\\"\\n    every_changes = all(np.asarray(x).shape != np.asarray(y).shape or not np.array_equal(x, y)\\n                        for x, y in task[\\"train\\"])\\n    groups = {}\\n    for f in fits:\\n        preds = []\\n        for q, p in zip(task[\\"test_inputs\\"], f[\\"preds\\"]):\\n            if p is None or not G.valid_grid(p):\\n                preds.append(None)\\n                continue\\n            if every_changes and np.asarray(p).shape == np.asarray(q).shape and np.array_equal(p, q):\\n                preds.append(None)                    # vacuous\\n                continue\\n            preds.append(p)\\n        k = tuple(_key(p) for p in preds)\\n        if all(x is None for x in k):\\n            continue\\n        g = groups.setdefault(k, {\\"preds\\": preds, \\"members\\": [], \\"complete\\": True, \\"loo\\": 0.0, \\"cost\\": 1e9,\\n                                  \\"order\\": 1e9})\\n        g[\\"members\\"].append(f)\\n        g[\\"complete\\"] = g[\\"complete\\"] and all(f.get(\\"complete\\", [True] * len(preds)))\\n        g[\\"loo\\"] = max(g[\\"loo\\"], f.get(\\"loo\\", 0.0))\\n        g[\\"cost\\"] = min(g[\\"cost\\"], f.get(\\"cost\\", 1.0))\\n        g[\\"order\\"] = min(g[\\"order\\"], f.get(\\"order\\", 0))\\n    out = sorted(groups.values(), key=lambda g: (not g[\\"complete\\"], -g[\\"loo\\"], -len(g[\\"members\\"]), g[\\"cost\\"],\\n                                                  g[\\"order\\"]))\\n    return out\\n\\n\\ndef attempts(groups, n_queries):\\n    per = []\\n    for j in range(n_queries):\\n        atts, seen = [], set()\\n        for g in groups:\\n            p = g[\\"preds\\"][j] if j < len(g[\\"preds\\"]) else None\\n            k = _key(p)\\n            if p is None or k in seen:\\n                continue\\n            seen.add(k)\\n            atts.append(np.asarray(p).tolist())\\n            if len(atts) == 2:\\n                break\\n        per.append(atts)\\n    return per\\n\\n\\ndef to_program_candidates(groups, n_queries, n_programs, failures=None, limit=2):\\n    per = [[] for _ in range(n_queries)]\\n    for gi, g in enumerate(groups):\\n        best = g[\\"members\\"][0]\\n        for j in range(n_queries):\\n            p = g[\\"preds\\"][j] if j < len(g[\\"preds\\"]) else None\\n            if p is None or len(per[j]) >= limit or any(np.asarray(p).tolist() == x[\\"grid\\"] for x in per[j]):\\n                continue\\n            per[j].append({\\"grid\\": np.asarray(p).tolist(), \\"tier\\": \\"CAST_FIT\\" if g[\\"complete\\"] else \\"CAST_PARTIAL\\",\\n                           \\"score\\": round(float(len(per[j]) == 0) + g[\\"loo\\"], 3), \\"certified\\": False,\\n                           \\"loo\\": round(float(g[\\"loo\\"]), 3), \\"group_size\\": len(g[\\"members\\"]),\\n                           \\"program\\": best[\\"program\\"], \\"candidate\\": f\\"cast-{gi}\\"})\\n    return {\\"per_test\\": per, \\"n_programs\\": n_programs, \\"n_fits\\": sum(len(g[\\"members\\"]) for g in groups),\\n            \\"n_groups\\": len(groups), \\"failures\\": failures or {}}\\n", "cast_selective_v2/grids.py": "\\"\\"\\"Grid utilities and pixel-grounded views for CAST (numpy, deterministic, no background-0 assumption).\\n\\nConventions (declared once, used everywhere):\\n- A grid is a 2-D numpy int array with values 0..9. Coordinates are (row, col), row 0 at the top.\\n- Background of a grid = its majority colour; ties go to the smallest colour value. Never assumed to be 0.\\n- Connectivity: 4 = edge neighbours; 8 = edge or corner neighbours. A \\"mono\\" object is a connected set of cells of one\\n  colour; a \\"multi\\" object is a connected set of non-background cells of any colours.\\n- An enclosed region (hole) of a mask M is a 4-connected component of the complement of M that does not touch the\\n  grid border. With 8-connected objects, holes use 4-connectivity of the complement (the standard digital-topology\\n  pairing (8, 4)); with 4-connected objects the pairing is (4, 8). Both are offered; callers state which.\\nEvery entity keeps its source mask, its box and its cells, so rendering back to a grid is exact.\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nfrom . import _nd as ndimage\\n\\nS4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\\nS8 = np.ones((3, 3), dtype=bool)\\n\\n\\ndef A(g):\\n    return np.asarray(g, dtype=np.int64)\\n\\n\\ndef bg_of(g):\\n    \\"\\"\\"Majority colour; ties -> smallest colour value.\\"\\"\\"\\n    cnt = np.bincount(A(g).ravel(), minlength=10)\\n    return int(np.argmax(cnt))\\n\\n\\ndef palette(g):\\n    return frozenset(int(v) for v in np.unique(A(g)))\\n\\n\\ndef counts(g):\\n    return np.bincount(A(g).ravel(), minlength=10)[:10]\\n\\n\\ndef label(mask, conn=4):\\n    lab, n = ndimage.label(mask, structure=S4 if conn == 4 else S8)\\n    return lab, n\\n\\n\\n@dataclass\\nclass Obj:\\n    \\"\\"\\"A connected entity: its mask on the source grid, box (r0, c0, r1, c1) inclusive, colours and cell values.\\"\\"\\"\\n    mask: np.ndarray\\n    box: tuple\\n    colours: frozenset\\n    size: int\\n    idx: int = 0\\n    tags: dict = field(default_factory=dict)\\n\\n    @property\\n    def h(self):\\n        return self.box[2] - self.box[0] + 1\\n\\n    @property\\n    def w(self):\\n        return self.box[3] - self.box[1] + 1\\n\\n    def patch(self, g, fill=-1):\\n        \\"\\"\\"The object\'s box cut from g, cells outside the mask set to `fill`.\\"\\"\\"\\n        r0, c0, r1, c1 = self.box\\n        p = A(g)[r0:r1 + 1, c0:c1 + 1].copy()\\n        p[~self.mask[r0:r1 + 1, c0:c1 + 1]] = fill\\n        return p\\n\\n    def shape_key(self):\\n        r0, c0, r1, c1 = self.box\\n        return self.mask[r0:r1 + 1, c0:c1 + 1].tobytes() + bytes([self.h, self.w])\\n\\n    @property\\n    def colour(self):\\n        return min(self.colours) if len(self.colours) == 1 else -1\\n\\n\\ndef _box(mask):\\n    rows = np.where(mask.any(1))[0]\\n    cols = np.where(mask.any(0))[0]\\n    return (int(rows[0]), int(cols[0]), int(rows[-1]), int(cols[-1]))\\n\\n\\ndef objects(g, bg=None, conn=4, mono=True):\\n    \\"\\"\\"Objects of g under a declared view; bg defaults to the grid\'s majority colour.\\"\\"\\"\\n    g = A(g)\\n    if bg is None:\\n        bg = bg_of(g)\\n    out = []\\n    if mono:\\n        for c in sorted(set(np.unique(g).tolist()) - {bg}):\\n            lab, n = label(g == c, conn)\\n            for i in range(1, n + 1):\\n                m = lab == i\\n                out.append(Obj(m, _box(m), frozenset([c]), int(m.sum())))\\n    else:\\n        lab, n = label(g != bg, conn)\\n        for i in range(1, n + 1):\\n            m = lab == i\\n            out.append(Obj(m, _box(m), frozenset(int(v) for v in np.unique(g[m])), int(m.sum())))\\n    out.sort(key=lambda o: (o.box[0], o.box[1], o.colour, o.size))\\n    for i, o in enumerate(out):\\n        o.idx = i\\n    return out\\n\\n\\ndef holes_of_mask(mask, conn_bg=4):\\n    \\"\\"\\"Enclosed regions of the complement of mask (components not touching the border).\\"\\"\\"\\n    comp = ~mask\\n    lab, n = label(comp, conn_bg)\\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())\\n    return [lab == i for i in range(1, n + 1) if i not in border]\\n\\n\\ndef hole_count(obj, conn_bg=4):\\n    r0, c0, r1, c1 = obj.box\\n    m = np.pad(obj.mask[r0:r1 + 1, c0:c1 + 1], 1)\\n    return len(holes_of_mask(m, conn_bg))\\n\\n\\ndef enclosed_background(g, bg=None, conn_bg=4):\\n    \\"\\"\\"Background cells not connected (conn_bg) to the border through background cells.\\"\\"\\"\\n    g = A(g)\\n    if bg is None:\\n        bg = bg_of(g)\\n    lab, n = label(g == bg, conn_bg)\\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())\\n    keep = np.zeros(g.shape, dtype=bool)\\n    for i in range(1, n + 1):\\n        if i not in border:\\n            keep |= lab == i\\n    return keep\\n\\n\\ndef separators(g):\\n    \\"\\"\\"Full rows/columns of one non-background-or-background colour that cut the grid into panels.\\n    Returns (colour, row indices, col indices) for the colour with the most full lines, or None.\\"\\"\\"\\n    g = A(g)\\n    h, w = g.shape\\n    best = None\\n    for c in np.unique(g).tolist():\\n        rows = [r for r in range(h) if (g[r] == c).all()]\\n        cols = [k for k in range(w) if (g[:, k] == c).all()]\\n        if (rows or cols) and len(rows) < h and len(cols) < w:\\n            score = len(rows) + len(cols)\\n            if best is None or score > best[0]:\\n                best = (score, c, rows, cols)\\n    if best is None:\\n        return None\\n    return best[1], best[2], best[3]\\n\\n\\ndef panels(g):\\n    \\"\\"\\"Panels cut by the separator lines (row-major order), as (r0, c0, r1, c1) inclusive boxes; [] if none.\\"\\"\\"\\n    g = A(g)\\n    s = separators(g)\\n    if s is None:\\n        return []\\n    _, rows, cols = s\\n    h, w = g.shape\\n\\n    def spans(lines, n):\\n        out, start = [], 0\\n        for x in sorted(lines) + [n]:\\n            if x > start:\\n                out.append((start, x - 1))\\n            start = x + 1\\n        return out\\n    rs, cs = spans(rows, h), spans(cols, w)\\n    if len(rs) * len(cs) < 2:\\n        return []\\n    return [(a, b, c, d) for a, c in rs for b, d in cs]\\n\\n\\ndef crop(g, box):\\n    r0, c0, r1, c1 = box\\n    return A(g)[r0:r1 + 1, c0:c1 + 1].copy()\\n\\n\\ndef bbox_of(mask):\\n    if not mask.any():\\n        return None\\n    return _box(mask)\\n\\n\\ndef grid_key(g):\\n    g = A(g)\\n    return g.shape, g.tobytes()\\n\\n\\ndef valid_grid(g):\\n    if g is None:\\n        return False\\n    g = np.asarray(g)\\n    return g.ndim == 2 and 1 <= g.shape[0] <= 30 and 1 <= g.shape[1] <= 30 and g.min() >= 0 and g.max() <= 9\\n", "cast_selective_v2/ops.py": "\\"\\"\\"CAST operator library L0 with declared contracts (M3) and the task context that sets parameter domains.\\n\\nEvery operator class declares:\\n    domain(ctx)        finite parameter domain for a task (built from the demonstrations\' colours and shapes only:\\n                       an OBSERVED_DEMONSTRATION_CONSTRAINT that defines the grammar instance; never from queries\'\\n                       outputs, which the solver cannot read)\\n    apply(g, th)       concrete semantics; returns a grid or None (undefined); deterministic, bounded, no I/O\\n    fwd(a, th)         cheap forward contract (PROVED_OPERATOR_CONTRACT): gamma(fwd(a, th)) contains apply(z, th) for\\n                       every z in gamma(a) where apply is defined. It must not run the operator\'s expensive part.\\n    bwd(y, th)         predecessor relation: gamma(bwd(y, th)) contains every w with apply(w, th) == y\\n    cost               nominal execution cost unit (CPU is measured separately)\\nContracts are validated exhaustively on tiny grids and by randomized property tests (tests/test_contracts_*.py);\\nthe declared scope is the finite parameter domain and grids up to 30 x 30.\\nOverlap semantics: every L0 operator is a whole-grid function (one synchronous write of the full output).\\n\\"\\"\\"\\nimport numpy as np\\n\\nfrom . import absdom as D\\nfrom . import grids as G\\n\\nGEO = (\\"fliplr\\", \\"flipud\\", \\"rot90\\", \\"rot180\\", \\"rot270\\", \\"transpose\\", \\"antitranspose\\")\\n\\n\\ndef geo_apply(g, t):\\n    g = np.asarray(g)\\n    return {\\"fliplr\\": lambda x: x[:, ::-1], \\"flipud\\": lambda x: x[::-1, :], \\"rot90\\": lambda x: np.rot90(x, 1),\\n            \\"rot180\\": lambda x: np.rot90(x, 2), \\"rot270\\": lambda x: np.rot90(x, 3), \\"transpose\\": lambda x: x.T,\\n            \\"antitranspose\\": lambda x: np.rot90(x, 2).T}[t](g).copy()\\n\\n\\nGEO_INV = {\\"fliplr\\": \\"fliplr\\", \\"flipud\\": \\"flipud\\", \\"rot90\\": \\"rot270\\", \\"rot180\\": \\"rot180\\", \\"rot270\\": \\"rot90\\",\\n           \\"transpose\\": \\"transpose\\", \\"antitranspose\\": \\"antitranspose\\"}\\n\\n\\ndef geo_shape(s, t):\\n    return (s[1], s[0]) if t in (\\"rot90\\", \\"rot270\\", \\"transpose\\", \\"antitranspose\\") else s\\n\\n\\nclass TaskCtx:\\n    \\"\\"\\"What a grammar instance may depend on: demonstrations (inputs and outputs) and query INPUTS only.\\"\\"\\"\\n\\n    def __init__(self, train, test_inputs=()):\\n        self.train = [(np.asarray(x, dtype=np.int64), np.asarray(y, dtype=np.int64)) for x, y in train]\\n        self.test_inputs = [np.asarray(x, dtype=np.int64) for x in test_inputs]\\n        self.in_pal = sorted(set().union(*[G.palette(x) for x, _ in self.train]))\\n        self.out_pal = sorted(set().union(*[G.palette(y) for _, y in self.train]))\\n        self.all_pal = sorted(set(self.in_pal) | set(self.out_pal))\\n        self.same_shape = all(x.shape == y.shape for x, y in self.train)\\n\\n\\nclass Op:\\n    name = \\"op\\"\\n    cost = 1\\n    evidence = \\"PROVED_OPERATOR_CONTRACT\\"\\n    scope = \\"grids up to 30x30, the listed parameter domain\\"\\n    same_shape = True\\n\\n    def domain(self, ctx):\\n        return [()]\\n\\n    def apply(self, g, th):\\n        raise NotImplementedError\\n\\n    def fwd(self, a, th):\\n        return D.TOP()\\n\\n    def bwd(self, y, th):\\n        return D.TOP()\\n\\n    def label(self, th):\\n        return f\\"{self.name}{th}\\"\\n\\n\\ndef _same_shape_cells_or_top(a, newbits_fn):\\n    \\"\\"\\"Helper: per-cell transfer when cells are known; otherwise only shapes and palette are kept.\\"\\"\\"\\n    if a.bottom:\\n        return D.BOTTOM()\\n    if a.cells is None:\\n        return D.Abs(a.shapes, a.shape_lo, None, D.ALL)\\n    return D.Abs(a.shapes, a.shape_lo, newbits_fn(a.cells), D.ALL)\\n\\n\\nclass Identity(Op):\\n    name = \\"id\\"\\n\\n    def apply(self, g, th):\\n        return np.asarray(g).copy()\\n\\n    def fwd(self, a, th):\\n        return a.copy()\\n\\n    def bwd(self, y, th):\\n        return D.alpha(y)\\n\\n\\nclass Geo(Op):\\n    name = \\"geo\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(t,) for t in GEO]\\n\\n    def apply(self, g, th):\\n        return geo_apply(g, th[0])\\n\\n    def fwd(self, a, th):\\n        t = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        shapes = None if a.shapes is None else {geo_shape(s, t) for s in a.shapes}\\n        lo = geo_shape(a.shape_lo, t)\\n        cells = None if a.cells is None else geo_apply(a.cells, t)\\n        return D.Abs(shapes, lo, cells, a.pal, a.need, a.lo, a.hi)\\n\\n    def bwd(self, y, th):\\n        return D.alpha(geo_apply(y, GEO_INV[th[0]]))\\n\\n\\nclass Recolor(Op):\\n    \\"\\"\\"Every cell of colour a becomes b (a != b).\\"\\"\\"\\n    name = \\"recolor\\"\\n\\n    def domain(self, ctx):\\n        return [(a, b) for a in ctx.in_pal for b in ctx.out_pal if a != b]\\n\\n    def apply(self, g, th):\\n        a, b = th\\n        g = np.asarray(g).copy()\\n        g[g == a] = b\\n        return g\\n\\n    def fwd(self, x, th):\\n        a, b = th\\n        if x.bottom:\\n            return D.BOTTOM()\\n\\n        def mp(m):\\n            return (m & ~D.bit(a)) | (D.bit(b) if m >> a & 1 else 0)\\n        lo, hi = x.lo.copy(), x.hi.copy()\\n        lo[b], hi[b] = lo[b] + lo[a], hi[b] + hi[a]\\n        lo[a] = hi[a] = 0\\n        cells = None\\n        if x.cells is not None:\\n            has = (x.cells >> a) & 1\\n            cells = ((x.cells & ~np.uint16(D.bit(a))) | (has * np.uint16(D.bit(b)))).astype(np.uint16)\\n        return D.Abs(x.shapes, x.shape_lo, cells, mp(x.pal), mp(x.need), lo, hi)\\n\\n    def bwd(self, y, th):\\n        a, b = th\\n        y = np.asarray(y)\\n        if (y == a).any():\\n            return D.BOTTOM()\\n        cells = D.BITS[y].astype(np.uint16)\\n        cells[y == b] = D.bit(a) | D.bit(b)\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL)\\n\\n\\nSELECTORS = (\\"nonbg\\", \\"largest4\\", \\"smallest4\\", \\"largest8\\", \\"smallest8\\", \\"largest_multi8\\", \\"colour\\")\\n\\n\\ndef select_box(g, sel):\\n    g = np.asarray(g)\\n    bg = G.bg_of(g)\\n    if sel[0] == \\"nonbg\\":\\n        return G.bbox_of(g != bg)\\n    if sel[0] == \\"colour\\":\\n        return G.bbox_of(g == sel[1])\\n    conn = 8 if sel[0].endswith(\\"8\\") else 4\\n    objs = G.objects(g, bg, conn=conn, mono=not sel[0].startswith(\\"largest_multi\\"))\\n    if not objs:\\n        return None\\n    sizes = [o.size for o in objs]\\n    target = max(sizes) if sel[0].startswith(\\"largest\\") else min(sizes)\\n    cand = [o for o in objs if o.size == target]\\n    if len(cand) != 1:\\n        return None                                   # ambiguous extremum: undefined (no tie-break by position)\\n    return cand[0].box\\n\\n\\nclass CropBox(Op):\\n    \\"\\"\\"Crop to the box of a selected region: the non-background cells, one colour, or the unique largest/smallest\\n    object under a declared view.\\"\\"\\"\\n    name = \\"crop\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(s,) for s in SELECTORS if s != \\"colour\\"] + [(\\"colour\\", c) for c in ctx.in_pal]\\n\\n    def apply(self, g, th):\\n        box = select_box(g, th)\\n        if box is None:\\n            return None\\n        return G.crop(g, box)\\n\\n    def fwd(self, a, th):\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal, 0, None, a.hi)\\n        if D.is_concrete(a) and th[0] == \\"colour\\":            # cheap and exact: the box of one colour\\n            g = D.concretize_one(a)\\n            out = self.apply(g, th)\\n            return D.BOTTOM() if out is None else D.alpha(out)\\n        shapes = {(h, w) for H, W in a.shapes for h in range(1, H + 1) for w in range(1, W + 1)}\\n        return D.Abs(shapes, (1, 1), None, a.pal, 0, None, a.hi)\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        cnt = G.counts(y)\\n        need = D.mask_of(np.unique(y).tolist())\\n        return D.Abs(None, y.shape, None, D.ALL, need, cnt, None)\\n\\n\\nclass Scale(Op):\\n    name = \\"scale\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(k,) for k in (2, 3, 4, 5)]\\n\\n    def apply(self, g, th):\\n        k = th[0]\\n        g = np.asarray(g)\\n        if g.shape[0] * k > 30 or g.shape[1] * k > 30:\\n            return None\\n        return np.kron(g, np.ones((k, k), dtype=g.dtype))\\n\\n    def fwd(self, a, th):\\n        k = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        shapes = None if a.shapes is None else {(h * k, w * k) for h, w in a.shapes}\\n        cells = None if a.cells is None else np.kron(a.cells, np.ones((k, k), dtype=np.uint16)).astype(np.uint16)\\n        return D.Abs(shapes, (a.shape_lo[0] * k, a.shape_lo[1] * k), cells, a.pal, a.need, a.lo * k * k,\\n                     np.minimum(a.hi * k * k, D.MAXC))\\n\\n    def bwd(self, y, th):\\n        k = th[0]\\n        y = np.asarray(y)\\n        h, w = y.shape\\n        if h % k or w % k:\\n            return D.BOTTOM()\\n        small = y[::k, ::k]\\n        if not np.array_equal(np.kron(small, np.ones((k, k), dtype=y.dtype)), y):\\n            return D.BOTTOM()\\n        return D.alpha(small)\\n\\n\\nclass Downscale(Op):\\n    \\"\\"\\"Exact block downscale: defined only when the grid is constant on every k x k block.\\"\\"\\"\\n    name = \\"down\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(k,) for k in (2, 3, 4, 5)]\\n\\n    def apply(self, g, th):\\n        k = th[0]\\n        g = np.asarray(g)\\n        h, w = g.shape\\n        if h % k or w % k:\\n            return None\\n        small = g[::k, ::k]\\n        if not np.array_equal(np.kron(small, np.ones((k, k), dtype=g.dtype)), g):\\n            return None\\n        return small.copy()\\n\\n    def fwd(self, a, th):\\n        k = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal, a.need)\\n        shapes = {(h // k, w // k) for h, w in a.shapes if h % k == 0 and w % k == 0 and h >= k and w >= k}\\n        if not shapes:\\n            return D.BOTTOM()\\n        cells = None\\n        if a.cells is not None and a.cells.shape[0] % k == 0 and a.cells.shape[1] % k == 0:\\n            h, w = a.cells.shape\\n            cells = np.bitwise_and.reduce(a.cells.reshape(h // k, k, w // k, k).transpose(0, 2, 1, 3)\\n                                          .reshape(h // k, w // k, k * k), axis=2).astype(np.uint16)\\n        return D.Abs(shapes, (1, 1), cells, a.pal, a.need, (a.lo + k * k - 1) // (k * k), a.hi // (k * k))\\n\\n    def bwd(self, y, th):\\n        k = th[0]\\n        y = np.asarray(y)\\n        if y.shape[0] * k > 30 or y.shape[1] * k > 30:\\n            return D.BOTTOM()\\n        return D.alpha(np.kron(y, np.ones((k, k), dtype=y.dtype)))\\n\\n\\nclass Tile(Op):\\n    name = \\"tile\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(n, m) for n in range(1, 5) for m in range(1, 5) if (n, m) != (1, 1)]\\n\\n    def apply(self, g, th):\\n        n, m = th\\n        g = np.asarray(g)\\n        if g.shape[0] * n > 30 or g.shape[1] * m > 30:\\n            return None\\n        return np.tile(g, (n, m))\\n\\n    def fwd(self, a, th):\\n        n, m = th\\n        if a.bottom:\\n            return D.BOTTOM()\\n        shapes = None if a.shapes is None else {(h * n, w * m) for h, w in a.shapes}\\n        cells = None if a.cells is None else np.tile(a.cells, (n, m)).astype(np.uint16)\\n        return D.Abs(shapes, (a.shape_lo[0] * n, a.shape_lo[1] * m), cells, a.pal, a.need, a.lo * n * m,\\n                     np.minimum(a.hi * n * m, D.MAXC))\\n\\n    def bwd(self, y, th):\\n        n, m = th\\n        y = np.asarray(y)\\n        h, w = y.shape\\n        if h % n or w % m:\\n            return D.BOTTOM()\\n        u = y[:h // n, :w // m]\\n        if not np.array_equal(np.tile(u, (n, m)), y):\\n            return D.BOTTOM()\\n        return D.alpha(u)\\n\\n\\nclass Concat(Op):\\n    \\"\\"\\"g next to t(g): side \'h\' (left|right) or \'v\' (top/bottom), t a same-shape symmetry.\\"\\"\\"\\n    name = \\"concat\\"\\n    same_shape = False\\n    TS = (\\"id\\", \\"fliplr\\", \\"flipud\\", \\"rot180\\")\\n\\n    def domain(self, ctx):\\n        return [(s, t) for s in \\"hv\\" for t in self.TS]\\n\\n    @staticmethod\\n    def _t(g, t):\\n        return np.asarray(g).copy() if t == \\"id\\" else geo_apply(g, t)\\n\\n    def apply(self, g, th):\\n        s, t = th\\n        g = np.asarray(g)\\n        if (s == \\"h\\" and g.shape[1] * 2 > 30) or (s == \\"v\\" and g.shape[0] * 2 > 30):\\n            return None\\n        return np.concatenate([g, self._t(g, t)], axis=1 if s == \\"h\\" else 0)\\n\\n    def fwd(self, a, th):\\n        s, t = th\\n        if a.bottom:\\n            return D.BOTTOM()\\n        f = (lambda h, w: (h, 2 * w)) if s == \\"h\\" else (lambda h, w: (2 * h, w))\\n        shapes = None if a.shapes is None else {f(*sh) for sh in a.shapes}\\n        cells = None\\n        if a.cells is not None:\\n            cells = np.concatenate([a.cells, self._t(a.cells, t)], axis=1 if s == \\"h\\" else 0).astype(np.uint16)\\n        return D.Abs(shapes, f(*a.shape_lo), cells, a.pal, a.need, a.lo * 2, np.minimum(a.hi * 2, D.MAXC))\\n\\n    def bwd(self, y, th):\\n        s, t = th\\n        y = np.asarray(y)\\n        h, w = y.shape\\n        if (s == \\"h\\" and w % 2) or (s == \\"v\\" and h % 2):\\n            return D.BOTTOM()\\n        p, q = (y[:, :w // 2], y[:, w // 2:]) if s == \\"h\\" else (y[:h // 2], y[h // 2:])\\n        inv = q if t == \\"id\\" else geo_apply(q, GEO_INV[t])\\n        return D.meet(D.alpha(p), D.alpha(inv))\\n\\n\\nclass FillEnclosed(Op):\\n    \\"\\"\\"Background cells that cannot reach the border through background (4-connected) become colour c.\\"\\"\\"\\n    name = \\"fill_enclosed\\"\\n\\n    def domain(self, ctx):\\n        return [(c,) for c in ctx.out_pal]\\n\\n    def apply(self, g, th):\\n        g = np.asarray(g).copy()\\n        m = G.enclosed_background(g)\\n        if not m.any():\\n            return None\\n        g[m] = th[0]\\n        return g\\n\\n    def fwd(self, a, th):\\n        c = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.cells is None:\\n            return D.Abs(a.shapes, a.shape_lo, None, a.pal | D.bit(c))\\n        cells = a.cells.copy()\\n        inner = np.zeros(cells.shape, dtype=bool)\\n        inner[1:-1, 1:-1] = True                        # border cells can never be enclosed\\n        cells[inner] |= np.uint16(D.bit(c))\\n        return D.Abs(a.shapes, a.shape_lo, cells, a.pal | D.bit(c))\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        cells = D.BITS[y].astype(np.uint16)\\n        cells[y == th[0]] = D.ALL\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL)\\n\\n\\nclass Outline(Op):\\n    \\"\\"\\"Background cells 4- or 8-adjacent to a non-background cell become colour c.\\"\\"\\"\\n    name = \\"outline\\"\\n\\n    def domain(self, ctx):\\n        return [(c, k) for c in ctx.out_pal for k in (4, 8)]\\n\\n    def apply(self, g, th):\\n        c, k = th\\n        g = np.asarray(g)\\n        bg = G.bg_of(g)\\n        from . import _nd as ndimage\\n        fg = g != bg\\n        if not fg.any():\\n            return None\\n        near = ndimage.binary_dilation(fg, structure=G.S4 if k == 4 else G.S8) & ~fg\\n        out = g.copy()\\n        out[near] = c\\n        return out\\n\\n    def fwd(self, a, th):\\n        c = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.cells is None:\\n            return D.Abs(a.shapes, a.shape_lo, None, a.pal | D.bit(c))\\n        return D.Abs(a.shapes, a.shape_lo, (a.cells | np.uint16(D.bit(c))).astype(np.uint16), a.pal | D.bit(c))\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        cells = D.BITS[y].astype(np.uint16)\\n        cells[y == th[0]] = D.ALL\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL)\\n\\n\\nclass Gravity(Op):\\n    \\"\\"\\"Cell gravity: in every column (up/down) or row (left/right) the non-background cells slide to that side,\\n    keeping their order (one synchronous write of the full grid).\\"\\"\\"\\n    name = \\"gravity\\"\\n\\n    def domain(self, ctx):\\n        return [(d,) for d in (\\"down\\", \\"up\\", \\"left\\", \\"right\\")]\\n\\n    def apply(self, g, th):\\n        d = th[0]\\n        g = np.asarray(g)\\n        bg = G.bg_of(g)\\n        x = g if d in (\\"down\\", \\"up\\") else g.T\\n        out = np.full_like(x, bg)\\n        for j in range(x.shape[1]):\\n            col = x[:, j]\\n            fg = col[col != bg]\\n            if d in (\\"down\\", \\"right\\"):\\n                out[x.shape[0] - len(fg):, j] = fg\\n            else:\\n                out[:len(fg), j] = fg\\n        return out if d in (\\"down\\", \\"up\\") else out.T\\n\\n    def fwd(self, a, th):\\n        d = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.cells is None:\\n            return D.Abs(a.shapes, a.shape_lo, None, a.pal, a.need, a.lo, a.hi)\\n        x = a.cells if d in (\\"down\\", \\"up\\") else a.cells.T\\n        u = np.bitwise_or.reduce(x, axis=0)\\n        out = np.broadcast_to(u, x.shape).copy()\\n        out = out if d in (\\"down\\", \\"up\\") else out.T\\n        return D.Abs(a.shapes, a.shape_lo, out.astype(np.uint16), a.pal, a.need, a.lo, a.hi)\\n\\n    def bwd(self, y, th):\\n        d = th[0]\\n        y = np.asarray(y)\\n        x = y if d in (\\"down\\", \\"up\\") else y.T\\n        u = np.bitwise_or.reduce(D.BITS[x], axis=0)\\n        out = np.broadcast_to(u, x.shape).copy()\\n        out = out if d in (\\"down\\", \\"up\\") else out.T\\n        cnt = G.counts(y)\\n        return D.Abs({y.shape}, (1, 1), out.astype(np.uint16), D.ALL, D.mask_of(np.unique(y).tolist()), cnt, cnt)\\n\\n\\nL0 = [Geo(), Recolor(), CropBox(), Scale(), Downscale(), Tile(), Concat(), FillEnclosed(), Outline(), Gravity()]\\n\\n\\ndef by_name(name, lib=None):\\n    for op in (lib or L0):\\n        if op.name == name:\\n            return op\\n    raise KeyError(name)\\n", "cast_selective_v2/rewrite.py": "\\"\\"\\"M2: typed guarded rewriting over a grounded scene, and guarded output frames.\\n\\nScene (per grid, per view): background = majority colour; entities = objects under a declared view\\n(mono-4, mono-8, multi-8), each with its source mask, box, colours and features; relation tables computed from the\\nmasks (8-adjacency, enclosure of another object\'s cells inside the object\'s holes). Rendering is exact: an entity is\\nits cells.\\n\\nA rule is a typed guarded rewrite (NOT a double-pushout construction: no categorical interface is checked; this is an\\nordinary guarded graph rewrite with explicit semantics):\\n    pattern     one entity of the view\\n    guard       an atom over the entity\'s own features or its relations (or TRUE)\\n    preserved   every entity the rule does not match, and every cell outside the action\'s write set\\n    changed     the matched entity\'s write set (below)\\n    action      KEEP | DELETE | RECOLOR(c) | RECOLOR_REF(adjacent | enclosed) | FILL_HOLES(c) | OUTLINE(c) |\\n                MOVE(direction, until_blocked | to_border)\\n    schedule    SYNC for every action except MOVE: all matches and write values are computed on the INPUT scene,\\n                writes are applied together; two writes of different values to one cell make the program UNDEFINED.\\n                SEQ for MOVE: entities move one at a time, nearest to the target wall first, each reading the grid\\n                left by the previous moves (an entity stops at the border or before the first non-background cell).\\nA rule list is first-match: an entity takes the action of the first rule whose guard holds; no rule = KEEP.\\nWrite sets (the contract): DELETE/RECOLOR: the entity\'s cells; FILL_HOLES: its enclosed background cells; OUTLINE:\\nbackground cells 8-adjacent to it; MOVE: cells in its corridor along the direction. Canvas unchanged.\\nPrograms are found by construction from evidence (every demonstration object\'s observed fate proposes actions; guards\\nthat separate the objects of different fates are enumerated) and are ALWAYS verified by exact execution of the whole\\nrule list on every demonstration. Frame rules (CropEntity) crop to the box of the unique entity satisfying a guard.\\n\\"\\"\\"\\nimport itertools\\n\\nimport numpy as np\\nfrom . import _nd as ndimage\\n\\nfrom . import absdom as D\\nfrom . import grids as G\\nfrom .ops import Op\\n\\nVIEWS = ((\\"mono\\", 4), (\\"mono\\", 8), (\\"multi\\", 8))\\nMAX_ACTS = 8           # actions considered per view: the 8 that explain the most changed objects (declared bound)\\nMAX_OBJS = 120         # scenes with more objects over all demonstrations are skipped (declared bound)\\nTABLE_FEATURES = (\\"colour\\", \\"size\\", \\"holes\\", \\"shape\\", \\"height\\", \\"width\\", \\"n_colours\\")\\nDIRS = {\\"up\\": (-1, 0), \\"down\\": (1, 0), \\"left\\": (0, -1), \\"right\\": (0, 1)}\\n\\n\\n# ------------------------------------------------------------------ scene\\nclass Scene:\\n    def __init__(self, g, view):\\n        self.g = G.A(g)\\n        self.bg = G.bg_of(self.g)\\n        kind, conn = view\\n        self.view = view\\n        self.objs = G.objects(self.g, self.bg, conn=conn, mono=(kind == \\"mono\\"))\\n        H, W = self.g.shape\\n        lab = np.full(self.g.shape, -1)\\n        for o in self.objs:\\n            lab[o.mask] = o.idx\\n        self.lab = lab\\n        sizes = [o.size for o in self.objs]\\n        shapes = [o.shape_key() for o in self.objs]\\n        cols = [o.colour for o in self.objs]\\n        for o in self.objs:\\n            f = o.tags\\n            f[\\"colour\\"] = o.colour\\n            f[\\"size\\"] = o.size\\n            f[\\"holes\\"] = G.hole_count(o, 4)\\n            f[\\"has_hole\\"] = f[\\"holes\\"] > 0\\n            f[\\"border\\"] = bool(o.box[0] == 0 or o.box[1] == 0 or o.box[2] == H - 1 or o.box[3] == W - 1)\\n            f[\\"largest\\"] = o.size == max(sizes)\\n            f[\\"smallest\\"] = o.size == min(sizes)\\n            f[\\"size_unique\\"] = sizes.count(o.size) == 1\\n            f[\\"shape_unique\\"] = shapes.count(o.shape_key()) == 1\\n            f[\\"colour_unique\\"] = cols.count(o.colour) == 1\\n            f[\\"single_cell\\"] = o.size == 1\\n            f[\\"n_colours\\"] = len(o.colours)\\n            dil = ndimage.binary_dilation(o.mask, structure=G.S8) & ~o.mask\\n            nb = set(np.unique(lab[dil]).tolist()) - {-1}\\n            f[\\"adj_ids\\"] = sorted(nb)\\n            f[\\"adj_colours\\"] = sorted({self.objs[i].colour for i in nb})\\n            holes = self.hole_mask(o)\\n            inner = set(np.unique(lab[holes]).tolist()) - {-1} if holes.any() else set()\\n            f[\\"enc_ids\\"] = sorted(inner)\\n            f[\\"enc_colours\\"] = sorted({self.objs[i].colour for i in inner})\\n\\n    def hole_mask(self, o):\\n        r0, c0, r1, c1 = o.box\\n        sub = o.mask[r0:r1 + 1, c0:c1 + 1]\\n        filled = ndimage.binary_fill_holes(sub)\\n        m = np.zeros(self.g.shape, dtype=bool)\\n        m[r0:r1 + 1, c0:c1 + 1] = filled & ~sub\\n        return m\\n\\n\\n# ------------------------------------------------------------------ guards\\n_TRUE_ONLY = False\\n\\n\\ndef guard_space(scenes):\\n    \\"\\"\\"Finite guard atoms from the demonstration scenes (values seen in the demonstrations only).\\"\\"\\"\\n    out = [(\\"true\\",)]\\n    if _TRUE_ONLY:\\n        return out\\n    cols = sorted({o.tags[\\"colour\\"] for s in scenes for o in s.objs})\\n    out += [(\\"colour\\", c) for c in cols]\\n    out += [(\\"not_colour\\", c) for c in cols]\\n    for b in (\\"has_hole\\", \\"border\\", \\"largest\\", \\"smallest\\", \\"size_unique\\", \\"shape_unique\\", \\"colour_unique\\",\\n              \\"single_cell\\"):\\n        out += [(b, True), (b, False)]\\n    sizes = sorted({o.tags[\\"size\\"] for s in scenes for o in s.objs})\\n    out += [(\\"size\\", k) for k in sizes[:12]]\\n    hs = sorted({o.tags[\\"holes\\"] for s in scenes for o in s.objs})\\n    out += [(\\"holes\\", k) for k in hs if k <= 4]\\n    adj = sorted({c for s in scenes for o in s.objs for c in o.tags[\\"adj_colours\\"]})\\n    out += [(\\"adj_colour\\", c) for c in adj] + [(\\"adj_any\\", True), (\\"adj_any\\", False)]\\n    enc = sorted({c for s in scenes for o in s.objs for c in o.tags[\\"enc_colours\\"]})\\n    out += [(\\"enc_colour\\", c) for c in enc]\\n    return out\\n\\n\\ndef holds(o, gd):\\n    k = gd[0]\\n    f = o.tags\\n    if k == \\"true\\":\\n        return True\\n    if k == \\"colour\\":\\n        return f[\\"colour\\"] == gd[1]\\n    if k == \\"not_colour\\":\\n        return f[\\"colour\\"] != gd[1]\\n    if k in (\\"has_hole\\", \\"border\\", \\"largest\\", \\"smallest\\", \\"size_unique\\", \\"shape_unique\\", \\"colour_unique\\",\\n             \\"single_cell\\"):\\n        return f[k] == gd[1]\\n    if k == \\"size\\":\\n        return f[\\"size\\"] == gd[1]\\n    if k == \\"holes\\":\\n        return f[\\"holes\\"] == gd[1]\\n    if k == \\"adj_colour\\":\\n        return gd[1] in f[\\"adj_colours\\"]\\n    if k == \\"adj_any\\":\\n        return bool(f[\\"adj_ids\\"]) == gd[1]\\n    if k == \\"enc_colour\\":\\n        return gd[1] in f[\\"enc_colours\\"]\\n    raise KeyError(k)\\n\\n\\n# ------------------------------------------------------------------ actions (SYNC write sets and values)\\ndef writes(scene, o, act):\\n    \\"\\"\\"(mask, value array or scalar) written by a SYNC action on entity o; None = action undefined on o.\\"\\"\\"\\n    k = act[0]\\n    if k == \\"keep\\":\\n        return np.zeros(scene.g.shape, dtype=bool), 0\\n    if k == \\"delete\\":\\n        return o.mask, scene.bg\\n    if k == \\"recolor\\":\\n        return o.mask, act[1]\\n    if k == \\"recolor_ref\\":\\n        cs = o.tags[\\"adj_colours\\"] if act[1] == \\"adjacent\\" else o.tags[\\"enc_colours\\"]\\n        cs = [c for c in cs if c != o.tags[\\"colour\\"]]\\n        if len(cs) != 1:\\n            return None\\n        return o.mask, cs[0]\\n    if k == \\"fill_holes\\":\\n        h = scene.hole_mask(o) & (scene.g == scene.bg)\\n        if not h.any():\\n            return None\\n        return h, act[1]\\n    if k == \\"outline\\":\\n        m = ndimage.binary_dilation(o.mask, structure=G.S8) & (scene.g == scene.bg)\\n        return m, act[1]\\n    raise KeyError(k)\\n\\n\\ndef run_sync(scene, rules):\\n    out = scene.g.copy()\\n    wrote = np.zeros(scene.g.shape, dtype=bool)\\n    for o in scene.objs:\\n        act = next((a for gd, a in rules if holds(o, gd)), (\\"keep\\",))\\n        if act[0] == \\"keep\\":\\n            continue\\n        w = writes(scene, o, act)\\n        if w is None:\\n            return None\\n        m, v = w\\n        clash = wrote & m & (out != v)\\n        if clash.any():\\n            return None\\n        out[m] = v\\n        wrote |= m\\n    return out\\n\\n\\ndef run_moves(scene, rules):\\n    \\"\\"\\"SEQ schedule for MOVE rules (other entities keep their place).\\"\\"\\"\\n    g = scene.g.copy()\\n    plan = []\\n    for o in scene.objs:\\n        act = next((a for gd, a in rules if holds(o, gd)), (\\"keep\\",))\\n        if act[0] == \\"move\\":\\n            plan.append((o, act))\\n        elif act[0] != \\"keep\\":\\n            return None\\n    if not plan:\\n        return None\\n\\n    def key(item):\\n        o, act = item\\n        dr, dc = DIRS[act[1]]\\n        r0, c0, r1, c1 = o.box\\n        return -(r1 if dr > 0 else (-r0 if dr < 0 else (c1 if dc > 0 else -c0)))\\n    plan.sort(key=key)\\n    H, W = g.shape\\n    for o, act in plan:\\n        dr, dc = DIRS[act[1]]\\n        vals = g[o.mask]\\n        ys, xs = np.nonzero(o.mask)\\n        g[o.mask] = scene.bg\\n        step = 0\\n        while True:\\n            ny, nx = ys + (step + 1) * dr, xs + (step + 1) * dc\\n            if ny.min() < 0 or nx.min() < 0 or ny.max() >= H or nx.max() >= W:\\n                break\\n            if act[2] == \\"until_blocked\\" and (g[ny, nx] != scene.bg).any():\\n                break\\n            step += 1\\n        g[ys + step * dr, xs + step * dc] = vals\\n    return g\\n\\n\\n_SCENES = {}\\n\\n\\ndef scene_of(g, view):\\n    g = G.A(g)\\n    key = (g.shape, g.tobytes(), view)\\n    s = _SCENES.get(key)\\n    if s is None:\\n        if len(_SCENES) > 512:\\n            _SCENES.clear()\\n        s = Scene(g, view)\\n        _SCENES[key] = s\\n    return s\\n\\n\\ndef execute(g, view, rules):\\n    s = scene_of(g, view)\\n    if any(a[0] == \\"move\\" for _, a in rules):\\n        return run_moves(s, rules)\\n    return run_sync(s, rules)\\n\\n\\n# ------------------------------------------------------------------ construction from evidence\\ndef fates(scene, y):\\n    \\"\\"\\"Actions that explain each object of a demonstration in isolation (SYNC actions), by object index.\\"\\"\\"\\n    out = {}\\n    if y.shape != scene.g.shape:\\n        return None\\n    for o in scene.objs:\\n        ex = set()\\n        if (y[o.mask] == scene.g[o.mask]).all():\\n            ex.add((\\"keep\\",))\\n        vals = np.unique(y[o.mask])\\n        if len(vals) == 1:\\n            v = int(vals[0])\\n            if v == scene.bg:\\n                ex.add((\\"delete\\",))\\n            elif not (len(o.colours) == 1 and v == o.colour):\\n                ex.add((\\"recolor\\", v))\\n                for rel in (\\"adjacent\\", \\"enclosed\\"):\\n                    w = writes(scene, o, (\\"recolor_ref\\", rel))\\n                    if w is not None and w[1] == v:\\n                        ex.add((\\"recolor_ref\\", rel))\\n        h = scene.hole_mask(o) & (scene.g == scene.bg)\\n        if h.any():\\n            hv = np.unique(y[h])\\n            if len(hv) == 1 and int(hv[0]) != scene.bg:\\n                ex.add((\\"fill_holes\\", int(hv[0])))\\n        m = ndimage.binary_dilation(o.mask, structure=G.S8) & (scene.g == scene.bg)\\n        if m.any():\\n            mv = np.unique(y[m])\\n            if len(mv) == 1 and int(mv[0]) != scene.bg:\\n                ex.add((\\"outline\\", int(mv[0])))\\n        out[o.idx] = ex\\n    return out\\n\\n\\ndef construct(demos, view, max_lists=64):\\n    \\"\\"\\"Candidate rule lists (first-match, <= 2 rules + KEEP default, or 1 rule + another default) from evidence,\\n    each verified by exact execution on every demonstration. Returns verified rule lists.\\"\\"\\"\\n    if any(x.shape != y.shape for x, y in demos):\\n        return []\\n    scenes = [scene_of(x, view) for x, _ in demos]\\n    fs = [fates(s, y) for s, (_, y) in zip(scenes, demos)]\\n    cover = {}\\n    for f in fs:\\n        for e in f.values():\\n            if (\\"keep\\",) in e:\\n                continue\\n            for a in e:\\n                cover[a] = cover.get(a, 0) + 1\\n    acts = sorted(cover, key=lambda a: (-cover[a], repr(a)))[:MAX_ACTS]\\n    objs = [(i, o) for i, s in enumerate(scenes) for o in s.objs]\\n    if len(objs) > MAX_OBJS:\\n        return []\\n    guards = guard_space(scenes)\\n    # which guards are consistent with an action: every matching object is explained by it\\n    match = {gd: [holds(o, gd) for _, o in objs] for gd in guards}\\n    expl = {a: [a in fs[i][o.idx] for i, o in objs] for a in acts + [(\\"keep\\",)]}\\n    cands = []\\n    for a in acts:\\n        for gd in guards:\\n            m = match[gd]\\n            if not any(m) or not all(e for e, mm in zip(expl[a], m) if mm):\\n                continue\\n            rest = [k for k, mm in enumerate(m) if not mm]\\n            if all(expl[(\\"keep\\",)][k] for k in rest):\\n                cands.append([(gd, a)])\\n                continue\\n            for a2 in acts:\\n                if a2 == a:\\n                    continue\\n                if all(expl[a2][k] for k in rest):\\n                    cands.append([(gd, a), ((\\"true\\",), a2)])\\n                    continue\\n                for g2 in guards:\\n                    m2 = match[g2]\\n                    if not all((not m2[k]) or expl[a2][k] for k in rest):\\n                        continue\\n                    if all(m2[k] or expl[(\\"keep\\",)][k] for k in rest) and any(m2[k] for k in rest):\\n                        cands.append([(gd, a), (g2, a2)])\\n                        break\\n            if len(cands) > 4 * max_lists:\\n                break\\n    verified, seen = [], set()\\n    for rules in cands:\\n        key = repr(rules)\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        ok = True\\n        for x, y in demos:\\n            out = execute(x, view, rules)\\n            if out is None or not np.array_equal(out, y):\\n                ok = False\\n                break\\n        if ok:\\n            verified.append(rules)\\n            if len(verified) >= max_lists:\\n                break\\n    return verified\\n\\n\\ndef tfeat(o, f):\\n    if f == \\"shape\\":\\n        return o.shape_key()\\n    if f == \\"height\\":\\n        return o.h\\n    if f == \\"width\\":\\n        return o.w\\n    return o.tags[f]\\n\\n\\ndef run_table(scene, f, table):\\n    out = scene.g.copy()\\n    for o in scene.objs:\\n        k = tfeat(o, f)\\n        if k in table:\\n            out[o.mask] = table[k]\\n    return out\\n\\n\\ndef construct_tables(demos, view):\\n    \\"\\"\\"N-guard table rewrite: one categorical feature f; every object whose f-value is in the table is recoloured\\n    to the table\'s colour (guards f == v_i, actions RECOLOR(c_i)); all other objects KEEP. The table is a\\n    task-level constant learned from the demonstrations (an OBSERVED_DEMONSTRATION_CONSTRAINT), unlike M1\'s tables\\n    which are re-read from every input. Verified by exact execution.\\"\\"\\"\\n    if any(x.shape != y.shape for x, y in demos):\\n        return []\\n    out = []\\n    for f in TABLE_FEATURES:\\n        table, ok = {}, True\\n        for x, y in demos:\\n            s = scene_of(x, view)\\n            for o in s.objs:\\n                vals = np.unique(y[o.mask])\\n                k = tfeat(o, f)\\n                if len(vals) == 1 and not (len(o.colours) == 1 and int(vals[0]) == o.colour):\\n                    v = int(vals[0])\\n                    if table.get(k, v) != v:\\n                        ok = False\\n                        break\\n                    table[k] = v\\n            if not ok:\\n                break\\n        if not ok or not table:\\n            continue\\n        good = True\\n        for x, y in demos:\\n            if not np.array_equal(run_table(scene_of(x, view), f, table), y):\\n                good = False\\n                break\\n        if good:\\n            out.append((f, table))\\n    return out\\n\\n\\ndef construct_moves(demos, view, max_lists=16):\\n    \\"\\"\\"MOVE rule lists (SEQ): [(guard, move)] + KEEP, or [(guard, move d1), (true, move d2)]; verified by execution.\\"\\"\\"\\n    if any(x.shape != y.shape for x, y in demos):\\n        return []\\n    # declared precondition of the move constructor: every demonstration changes and keeps its colour counts\\n    if any(np.array_equal(x, y) or not np.array_equal(G.counts(x), G.counts(y)) for x, y in demos):\\n        return []\\n    scenes = [scene_of(x, view) for x, _ in demos]\\n    guards = guard_space(scenes)\\n    moves = [(\\"move\\", d, m) for d in DIRS for m in (\\"until_blocked\\", \\"to_border\\")]\\n    out = []\\n    for mv in moves:\\n        for gd in guards:\\n            for default in [None] + [m2 for m2 in moves if m2 != mv]:\\n                rules = [(gd, mv)] + ([((\\"true\\",), default)] if default else [])\\n                if gd == (\\"true\\",) and default:\\n                    continue\\n                ok = True\\n                for x, y in demos:\\n                    o = execute(x, view, rules)\\n                    if o is None or not np.array_equal(o, y):\\n                        ok = False\\n                        break\\n                if ok:\\n                    out.append(rules)\\n                    if len(out) >= max_lists:\\n                        return out\\n    return out\\n\\n\\n# ------------------------------------------------------------------ guarded output frames\\nclass CropEntity(Op):\\n    \\"\\"\\"Crop the input to the box of the UNIQUE entity (under a view) whose guard holds; undefined otherwise.\\"\\"\\"\\n    name = \\"crop_entity\\"\\n    same_shape = False\\n    scope = \\"grids up to 30x30; views mono-4, mono-8, multi-8; guard atoms from the demonstrations\\"\\n\\n    def domain(self, ctx):\\n        scenes = {v: [scene_of(x, v) for x, _ in ctx.train] for v in VIEWS}\\n        return [(v, gd) for v in VIEWS for gd in guard_space(scenes[v]) if gd != (\\"true\\",)]\\n\\n    def apply(self, g, th):\\n        view, gd = th\\n        s = scene_of(g, view)\\n        hits = [o for o in s.objs if holds(o, gd)]\\n        if len(hits) != 1:\\n            return None\\n        return G.crop(s.g, hits[0].box)\\n\\n    def fwd(self, a, th):\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal, 0, None, a.hi)\\n        shapes = {(h, w) for H, W in a.shapes for h in range(1, H + 1) for w in range(1, W + 1)}\\n        return D.Abs(shapes, (1, 1), None, a.pal, 0, None, a.hi)\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        return D.Abs(None, y.shape, None, D.ALL, D.mask_of(np.unique(y).tolist()), G.counts(y), None)\\n\\n\\ndef program_text(view, rules):\\n    return f\\"(rewrite view={view[0]}{view[1]} \\" + \\" \\".join(f\\"[{gd} -> {a}]\\" for gd, a in rules) + \\")\\"\\n\\n\\ndef search(task, max_lists=64, views=VIEWS, true_only=False, frames=True):\\n    \\"\\"\\"All verified rewrite programs (SYNC rule lists, SEQ moves, table rewrites, guarded crop frames) over the given\\n    views, with query predictions. true_only=True restricts every guard to TRUE (control: no guards).\\"\\"\\"\\n    demos = [(np.asarray(x, dtype=np.int64), np.asarray(y, dtype=np.int64)) for x, y in task[\\"train\\"]]\\n    qs = [np.asarray(q, dtype=np.int64) for q in task[\\"test_inputs\\"]]\\n    fits, execs = [], 0\\n    global _TRUE_ONLY\\n    _TRUE_ONLY = true_only\\n    for view in views:\\n        for rules in construct(demos, view, max_lists) + construct_moves(demos, view):\\n            preds = [execute(q, view, rules) for q in qs]\\n            fits.append({\\"program\\": program_text(view, rules), \\"preds\\": preds,\\n                         \\"complete\\": [p is not None for p in preds], \\"cost\\": float(len(rules)),\\n                         \\"order\\": len(fits), \\"view\\": view, \\"rules\\": repr(rules)})\\n        for f, table in construct_tables(demos, view):\\n            preds = [run_table(scene_of(q, view), f, table) for q in qs]\\n            if true_only:\\n                continue\\n            fits.append({\\"program\\": f\\"(table-rewrite view={view[0]}{view[1]} feature={f} n={len(table)})\\",\\n                         \\"preds\\": preds, \\"complete\\": [True] * len(preds), \\"cost\\": float(len(table)),\\n                         \\"order\\": len(fits), \\"view\\": view, \\"rules\\": f\\"table:{f}\\"})\\n    if frames and not true_only and all(np.asarray(x).shape != np.asarray(y).shape for x, y in task[\\"train\\"]):\\n        op = CropEntity()\\n        ctx = type(\\"C\\", (), {\\"train\\": demos})\\n        for th in op.domain(ctx):\\n            if th[0] not in views:\\n                continue\\n            ok = True\\n            for x, y in demos:\\n                o = op.apply(x, th)\\n                if o is None or o.shape != y.shape or not np.array_equal(o, y):\\n                    ok = False\\n                    break\\n            if ok:\\n                preds = [op.apply(q, th) for q in qs]\\n                fits.append({\\"program\\": f\\"(crop-entity view={th[0][0]}{th[0][1]} guard={th[1]})\\", \\"preds\\": preds,\\n                             \\"complete\\": [p is not None for p in preds], \\"cost\\": 1.0, \\"order\\": len(fits),\\n                             \\"view\\": th[0], \\"rules\\": repr(th)})\\n    _TRUE_ONLY = False\\n    return fits\\n\\n\\ndef footprint_ok(demos, view, rules):\\n    \\"\\"\\"M3 contract check for a SYNC rule list on concrete demonstrations: every changed cell must lie in the union of\\n    the matched entities\' write sets (computed without evaluating values).\\"\\"\\"\\n    for x, y in demos:\\n        s = scene_of(x, view)\\n        m = np.zeros(x.shape, dtype=bool)\\n        for o in s.objs:\\n            act = next((a for gd, a in rules if holds(o, gd)), (\\"keep\\",))\\n            if act[0] in (\\"keep\\",):\\n                continue\\n            if act[0] == \\"move\\":\\n                return True                                  # corridor footprint: no cheap rejection here\\n            w = writes(s, o, act)\\n            if w is not None:\\n                m |= w[0]\\n        if ((x != y) & ~m).any():\\n            return False\\n    return True\\n\\n\\n__all__ = [\\"Scene\\", \\"VIEWS\\", \\"construct\\", \\"construct_moves\\", \\"execute\\", \\"search\\", \\"CropEntity\\", \\"guard_space\\",\\n           \\"holds\\", \\"footprint_ok\\"]\\n_unused = itertools\\n", "cast_selective_v2/search.py": "\\"\\"\\"Bounded enumerative search over operator sequences with sound contract pruning (M3).\\n\\nA program is a sequence of operator instances (op, theta) applied left to right to the input grid. A partial program\\n(prefix) is executed concretely on every demonstration input; its remaining completions are the sequences of at\\nmost r further instances from the same grammar instance (stopping early is allowed).\\n\\nModes (the directive\'s mandatory controls; identical enumeration order and grammar):\\n    exact     no pruning: execute everything (exact execution only)\\n    typed     prune with the dimension domain only (ordinary typed constraints)\\n    fwd       sound forward analysis: hole(alpha(z), r) must contain the demonstrated output\\n    fwdbwd    fwd plus a backward predecessor check: meet(hole(alpha(z), r - 1), pre(y)) must be non-empty, where\\n              pre(y) is the join over the grammar of every operator\'s predecessor relation (and y itself)\\nA prune is recorded with its scope (prefix, remaining depth, grammar hash, demonstration index, domain, evidence\\nclass). A prune is never generalized beyond that scope. Search outcome statuses are kept distinct:\\n    FIT                      at least one program reproduces every demonstration\\n    NO_FIT_EXHAUSTED         every program of the bounded grammar was decided (executed or soundly pruned) and none\\n                             fits: \\"bounded-unrealizable in this grammar instance\\", NOT a proof that the task is\\n                             unexpressible\\n    BUDGET_EXHAUSTED         the execution or CPU budget ran out first (unknown)\\nObservational equivalence (optional): prefixes whose outputs on every demonstration AND query input equal an earlier\\nprefix\'s (at the same or larger remaining depth) are skipped; two programs that fit but disagree on a query are never\\nmerged.\\n\\"\\"\\"\\nimport hashlib\\nimport time\\n\\nimport numpy as np\\n\\nfrom . import absdom as D\\nfrom .ops import Identity\\n\\nMODES = (\\"exact\\", \\"typed\\", \\"fwd\\", \\"fwdbwd\\")\\n\\n\\ndef shapes_only(a):\\n    if a.bottom:\\n        return a\\n    return D.Abs(a.shapes, a.shape_lo, None, D.ALL)\\n\\n\\nclass Budget:\\n    def __init__(self, max_exec=None, max_cpu_s=None):\\n        self.max_exec, self.max_cpu_s = max_exec, max_cpu_s\\n        self.t0 = time.process_time()\\n        self.execs = 0\\n        self.transfers = 0\\n\\n    def out(self):\\n        if self.max_exec is not None and self.execs >= self.max_exec:\\n            return True\\n        if self.max_cpu_s is not None and time.process_time() - self.t0 >= self.max_cpu_s:\\n            return True\\n        return False\\n\\n    @property\\n    def cpu_s(self):\\n        return time.process_time() - self.t0\\n\\n\\ndef grammar_hash(instances):\\n    h = hashlib.sha256()\\n    for op, th in instances:\\n        h.update(f\\"{op.name}{th};\\".encode())\\n    return h.hexdigest()[:12]\\n\\n\\nclass Searcher:\\n    def __init__(self, ctx, lib, depth=2, mode=\\"fwdbwd\\", max_exec=None, max_cpu_s=None, oe=True,\\n                 keep_prunes=200, max_fits=64):\\n        assert mode in MODES\\n        self.ctx, self.lib, self.depth, self.mode, self.oe = ctx, lib, depth, mode, oe\\n        self.instances = [(op, th) for op in lib for th in op.domain(ctx)]\\n        self.ghash = grammar_hash(self.instances)\\n        self.budget = Budget(max_exec, max_cpu_s)\\n        self.fits, self.prunes, self.keep_prunes, self.max_fits = [], [], keep_prunes, max_fits\\n        self.n_pruned = {\\"subtree\\": 0, \\"subtree_bwd\\": 0, \\"leaf\\": 0}\\n        self.seen = {}\\n        self.ys = [y for _, y in ctx.train]\\n        self.queries = list(ctx.test_inputs)\\n        self._pre = None\\n        self._hcache = {}\\n        self.complete = True\\n\\n    # ---------------------------------------------------------------- abstract helpers\\n    def _proj(self, a):\\n        return shapes_only(a) if self.mode == \\"typed\\" else a\\n\\n    def hole(self, a, r):\\n        \\"\\"\\"Over-approximation of every completion of length <= r from abstract a (stop included).\\"\\"\\"\\n        cur = self._proj(a)\\n        for _ in range(r):\\n            nxt = cur\\n            for op, th in self.instances:\\n                self.budget.transfers += 1\\n                nxt = D.join(nxt, self._proj(op.fwd(cur, th)))\\n            if _same(nxt, cur):\\n                break\\n            cur = nxt\\n        return cur\\n\\n    def hole_summary(self, z, r):\\n        \\"\\"\\"Sound, cached over-approximation for subtree checks: the hole transfer started from the shape-and-palette\\n        abstraction of z (gamma contains alpha(z), so every completion\'s output is covered). Cached per\\n        (shape, palette, r); per-cell precision is kept for the leaf (instance) checks.\\"\\"\\"\\n        pal = int(D.BITW[np.bincount(z.ravel(), minlength=10)[:10] > 0].sum())\\n        key = (z.shape, pal, r, self.mode)\\n        hit = self._hcache.get(key)\\n        if hit is None:\\n            hit = self.hole(D.Abs({z.shape}, (1, 1), None, pal), r)\\n            self._hcache[key] = hit\\n        return hit\\n\\n    def pre(self, i):\\n        \\"\\"\\"Join of the predecessor relations of every grammar instance for demonstration output i, plus y_i.\\"\\"\\"\\n        if self._pre is None:\\n            self._pre = []\\n            for y in self.ys:\\n                acc = D.alpha(y)\\n                for op, th in self.instances:\\n                    self.budget.transfers += 1\\n                    acc = D.join(acc, op.bwd(y, th))\\n                self._pre.append(acc)\\n        return self._pre[i]\\n\\n    def _record(self, kind, prefix, r, i, why):\\n        self.n_pruned[kind] += 1\\n        if len(self.prunes) < self.keep_prunes:\\n            self.prunes.append({\\"kind\\": kind, \\"prefix\\": [f\\"{op.name}{th}\\" for op, th in prefix], \\"remaining\\": r,\\n                                \\"demo\\": i, \\"grammar\\": self.ghash, \\"domain\\": why,\\n                                \\"evidence\\": \\"PROVED_OPERATOR_CONTRACT\\"})\\n\\n    def subtree_dead(self, prefix, zs, r):\\n        if self.mode == \\"exact\\" or r == 0:\\n            return False\\n        for i, (z, y) in enumerate(zip(zs, self.ys)):\\n            if not D.contains(self.hole_summary(z, r), y):\\n                self._record(\\"subtree\\", prefix, r, i, \\"typed:dims\\" if self.mode == \\"typed\\" else \\"fwd\\")\\n                return True\\n        if self.mode == \\"fwdbwd\\":\\n            for i, (z, y) in enumerate(zip(zs, self.ys)):\\n                if np.array_equal(z, y) if z.shape == y.shape else False:\\n                    continue\\n                if D.meet(self.hole_summary(z, r - 1), self.pre(i)).bottom:\\n                    self._record(\\"subtree_bwd\\", prefix, r, i, \\"fwd-meet-bwd\\")\\n                    return True\\n        return False\\n\\n    def leaf_dead(self, prefix, zs, op, th):\\n        \\"\\"\\"Instance-level check before executing the last operator of a program (remaining depth 1).\\"\\"\\"\\n        if self.mode == \\"exact\\":\\n            return False\\n        for i, (z, y) in enumerate(zip(zs, self.ys)):\\n            self.budget.transfers += 1\\n            if not D.contains(self._proj(op.fwd(D.alpha(z), th)), y):\\n                self._record(\\"leaf\\", prefix + [(op, th)], 0, i, \\"instance contract\\")\\n                return True\\n        return False\\n\\n    # ---------------------------------------------------------------- search\\n    def run(self):\\n        zs = [x for x, _ in self.ctx.train]\\n        self._visit([], zs, list(self.queries), self.depth)\\n        status = \\"FIT\\" if self.fits else (\\"NO_FIT_EXHAUSTED\\" if self.complete else \\"BUDGET_EXHAUSTED\\")\\n        return {\\"status\\": status, \\"fits\\": self.fits, \\"execs\\": self.budget.execs, \\"transfers\\": self.budget.transfers,\\n                \\"cpu_s\\": round(self.budget.cpu_s, 4), \\"pruned\\": dict(self.n_pruned), \\"mode\\": self.mode,\\n                \\"depth\\": self.depth, \\"grammar\\": self.ghash, \\"n_instances\\": len(self.instances)}\\n\\n    def _visit(self, prefix, zs, qs, r):\\n        if self.budget.out():\\n            self.complete = False\\n            return\\n        if prefix and all(z.shape == y.shape and np.array_equal(z, y) for z, y in zip(zs, self.ys)):\\n            if len(self.fits) < self.max_fits:\\n                self.fits.append({\\"program\\": [(op.name, th) for op, th in prefix],\\n                                  \\"preds\\": [None if q is None else q.tolist() for q in qs]})\\n        if r == 0:\\n            return\\n        if prefix and self.subtree_dead(prefix, zs, r):\\n            return\\n        for op, th in self.instances:\\n            if self.budget.out():\\n                self.complete = False\\n                return\\n            if r == 1 and self.leaf_dead(prefix, zs, op, th):\\n                continue\\n            nz = []\\n            ok = True\\n            for z in zs:\\n                self.budget.execs += 1\\n                o = op.apply(z, th)\\n                if o is None:\\n                    ok = False\\n                    break\\n                nz.append(np.asarray(o, dtype=np.int64))\\n            if not ok:\\n                continue\\n            nq = []\\n            for q in qs:\\n                if q is None:\\n                    nq.append(None)\\n                    continue\\n                self.budget.execs += 1\\n                o = op.apply(q, th)\\n                nq.append(None if o is None else np.asarray(o, dtype=np.int64))\\n            if self.oe:\\n                key = tuple(z.shape + (z.tobytes(),) for z in nz) + tuple(\\n                    (None if q is None else q.shape + (q.tobytes(),)) for q in nq)\\n                if self.seen.get(key, -1) >= r - 1:\\n                    continue\\n                self.seen[key] = r - 1\\n            self._visit(prefix + [(op, th)], nz, nq, r - 1)\\n\\n\\ndef _same(a, b):\\n    if a.bottom != b.bottom:\\n        return False\\n    if a.bottom:\\n        return True\\n    if a.shapes != b.shapes or a.shape_lo != b.shape_lo or a.pal != b.pal or a.need != b.need:\\n        return False\\n    if (a.cells is None) != (b.cells is None):\\n        return False\\n    if a.cells is not None and not np.array_equal(a.cells, b.cells):\\n        return False\\n    return bool(np.array_equal(a.lo, b.lo) and np.array_equal(a.hi, b.hi))\\n\\n\\ndef stop_instance():\\n    return Identity(), ()\\n", "cast_selective_v2/symmetry.py": "\\"\\"\\"M4, first item: general symmetry completion (a whole-grid program class with a declared contract).\\n\\nSemantics of SymComplete(c, crop), for an input grid g of shape (h, w) and a mask colour c (a task-level parameter;\\nnever assumed to be 0 and never a background guess):\\n  1. Known cells K = {p : g[p] != c}; masked cells M = {p : g[p] == c}. Undefined if M is empty.\\n  2. Candidate symmetries are lattice isometries with ANY centre or axis (no centroid, no cell-count cap):\\n     reflections about every row axis and column axis at half-integer steps, 180-degree rotations about every\\n     half-integer centre, 90-degree rotations about every centre that maps the lattice to itself, and reflections\\n     about every main and anti diagonal offset.\\n  3. A candidate sigma is VALID on g when (i) at least theta_inside * h * w cells p have sigma(p) inside the grid,\\n     (ii) at least theta_known * |K| known cells p have sigma(p) known, and (iii) g[p] == g[sigma(p)] for every\\n     known p with sigma(p) known (exact agreement; background included, so it adapts to any background).\\n  4. Orbits: the connected components of the graph on cells with an edge p -- sigma(p) for every valid sigma.\\n     If one orbit holds two different known colours the operator is UNDEFINED on g (returns None). Otherwise every\\n     masked cell takes its orbit\'s known colour; masked cells in orbits without a known cell keep colour c\\n     (undetermined; counted in the trace). The result is independent of the order of symmetries.\\n  5. crop=False returns the completed grid; crop=True returns the completed grid restricted to the box of M.\\nThresholds theta_inside and theta_known are frozen on source-selection data (CAST_select), never on evaluation\\noutcomes. Defaults 0.5 / 0.5 follow the synthesis review\'s probe.\\n\\nContract (PROVED_OPERATOR_CONTRACT; argument: only cells of M are written, each with a colour copied from a known\\ncell, i.e. from palette(g) minus c; tests/test_symmetry.py validates it exhaustively on tiny grids):\\n  crop=False: shape unchanged; footprint subset of M; written colours subset of palette(g) - {c};\\n              count_c does not increase; count_k (k != c) does not decrease; total cell count unchanged.\\n  crop=True:  shape == shape of box(M); every output cell outside M keeps g\'s colour; cells of M take a colour in\\n              palette(g) - {c} or keep c.\\nPredecessor relation (crop=False): w[p] in {y[p], c} for every p (and w[p] == c wherever y[p] == c).\\n\\"\\"\\"\\nimport numpy as np\\nfrom . import _nd\\n\\nfrom . import absdom as D\\nfrom . import grids as G\\nfrom .ops import Op\\n\\nTHETA_INSIDE = 0.5\\nTHETA_KNOWN = 0.5\\n\\n\\ndef _maps(h, w, min_inside):\\n    \\"\\"\\"Candidate symmetries as (name, (Y\', X\')) index arrays; analytic overlap pre-filter.\\"\\"\\"\\n    Y, X = np.mgrid[0:h, 0:w]\\n    out = []\\n    for a in range(0, 2 * w - 1):                         # column axis at a / 2\\n        if h * (w - abs(a - (w - 1))) >= min_inside:\\n            out.append((f\\"flip_lr@{a / 2}\\", Y, a - X))\\n    for a in range(0, 2 * h - 1):\\n        if w * (h - abs(a - (h - 1))) >= min_inside:\\n            out.append((f\\"flip_ud@{a / 2}\\", a - Y, X))\\n    for a in range(0, 2 * h - 1):\\n        for b in range(0, 2 * w - 1):\\n            if (h - abs(a - (h - 1))) * (w - abs(b - (w - 1))) >= min_inside:\\n                out.append((f\\"rot180@{a / 2},{b / 2}\\", a - Y, b - X))\\n                if (a - b) % 2 == 0:\\n                    d, s = (a - b) // 2, (a + b) // 2\\n                    out.append((f\\"rot90@{a / 2},{b / 2}\\", X + d, s - Y))\\n    for k in range(-(h - 1), w):                           # (y, x) -> (x - k, y + k): reflection about y = x - k\\n        out.append((f\\"transpose@{k}\\", X - k, Y + k))\\n    for s in range(0, h + w - 1):                          # (y, x) -> (s - x, s - y)\\n        out.append((f\\"antitranspose@{s}\\", s - X, s - Y))\\n    return out\\n\\n\\n_MAP_CACHE = {}\\n\\n\\ndef maps_for(h, w, theta_inside=THETA_INSIDE):\\n    key = (h, w, theta_inside)\\n    if key not in _MAP_CACHE:\\n        if len(_MAP_CACHE) > 16:\\n            _MAP_CACHE.clear()\\n        min_inside = theta_inside * h * w\\n        res = []\\n        for name, YY, XX in _maps(h, w, min_inside):\\n            inside = (YY >= 0) & (YY < h) & (XX >= 0) & (XX < w)\\n            n_in = int(inside.sum())\\n            if n_in < min_inside or n_in == 0:\\n                continue\\n            src = np.flatnonzero(inside).astype(np.int32)\\n            dst = (YY[inside] * w + XX[inside]).astype(np.int32)\\n            if np.array_equal(src, dst):                    # the identity is not a symmetry candidate\\n                continue\\n            res.append((name, src, dst))\\n        _MAP_CACHE[key] = res\\n    return _MAP_CACHE[key]\\n\\n\\ndef valid_symmetries(g, c, theta_inside=THETA_INSIDE, theta_known=THETA_KNOWN):\\n    g = np.asarray(g)\\n    h, w = g.shape\\n    flat = g.ravel()\\n    known = flat != c\\n    nk = int(known.sum())\\n    if nk == 0:\\n        return []\\n    out = []\\n    for name, src, dst in maps_for(h, w, theta_inside):\\n        both = known[src] & known[dst]\\n        nb = int(both.sum())\\n        if nb < theta_known * nk:\\n            continue\\n        if np.array_equal(flat[src[both]], flat[dst[both]]):\\n            out.append((name, src, dst))\\n    return out\\n\\n\\ndef complete(g, c, theta_inside=THETA_INSIDE, theta_known=THETA_KNOWN):\\n    \\"\\"\\"(completed grid or None, trace dict).\\"\\"\\"\\n    g = np.asarray(g, dtype=np.int64)\\n    h, w = g.shape\\n    flat = g.ravel()\\n    masked = flat == c\\n    trace = {\\"mask_colour\\": int(c), \\"masked\\": int(masked.sum())}\\n    if not masked.any():\\n        trace[\\"status\\"] = \\"UNDEFINED_NO_MASK\\"\\n        return None, trace\\n    syms = valid_symmetries(g, c, theta_inside, theta_known)\\n    trace[\\"symmetries\\"] = [s[0] for s in syms]\\n    if not syms:\\n        trace[\\"status\\"] = \\"UNDEFINED_NO_SYMMETRY\\"\\n        return None, trace\\n    n = h * w\\n    rows = np.concatenate([s[1] for s in syms])\\n    cols = np.concatenate([s[2] for s in syms])\\n    ncomp, lab = _nd.components_from_edges(n, rows, cols)\\n    known = ~masked\\n    kl, kc = lab[known], flat[known]\\n    colour_of = np.full(ncomp, -1, dtype=np.int64)\\n    pairs = np.unique(np.stack([kl, kc], 1), axis=0) if len(kl) else np.zeros((0, 2), dtype=np.int64)\\n    if len(pairs) and len(np.unique(pairs[:, 0])) != len(pairs):\\n        trace[\\"status\\"] = \\"UNDEFINED_CONFLICT\\"\\n        return None, trace\\n    colour_of[pairs[:, 0]] = pairs[:, 1]\\n    out = flat.copy()\\n    fill = colour_of[lab]\\n    sel = masked & (fill >= 0)\\n    out[sel] = fill[sel]\\n    trace[\\"undetermined\\"] = int((masked & (fill < 0)).sum())\\n    trace[\\"status\\"] = \\"COMPLETE\\" if trace[\\"undetermined\\"] == 0 else \\"PARTIAL\\"\\n    return out.reshape(h, w), trace\\n\\n\\nclass SymComplete(Op):\\n    name = \\"symcomplete\\"\\n    cost = 20\\n    scope = (\\"grids up to 30x30; mask colour in the demonstrations\' input palette; thresholds frozen on \\"\\n             \\"CAST_select\\")\\n\\n    def __init__(self, theta_inside=THETA_INSIDE, theta_known=THETA_KNOWN):\\n        self.theta_inside, self.theta_known = theta_inside, theta_known\\n        self.last_trace = None\\n\\n    @property\\n    def same_shape(self):\\n        return False\\n\\n    def domain(self, ctx):\\n        return [(c, crop) for c in ctx.in_pal for crop in (False, True)]\\n\\n    def apply(self, g, th):\\n        c, crop = th\\n        out, tr = complete(g, c, self.theta_inside, self.theta_known)\\n        self.last_trace = tr\\n        if out is None:\\n            return None\\n        if crop:\\n            box = G.bbox_of(np.asarray(g) == c)\\n            return G.crop(out, box)\\n        return out\\n\\n    # ---- contract\\n    def _cells_same(self, a, c):\\n        others = a.pal & ~D.bit(c)\\n        has_c = ((a.cells >> c) & 1).astype(bool)\\n        cells = a.cells.copy()\\n        cells[has_c] |= np.uint16(others)\\n        return cells\\n\\n    def fwd(self, a, th):\\n        c, crop = th\\n        if a.bottom or not (a.pal >> c & 1):\\n            return D.BOTTOM()\\n        lo, hi = a.lo.copy(), a.hi.copy()\\n        lo_c, hi_c = lo[c], hi[c]\\n        lo[c] = 0\\n        for k in range(10):\\n            if k != c and (a.pal >> k & 1):\\n                hi[k] = hi[k] + hi_c\\n        others = a.pal & ~D.bit(c)\\n        if not crop:\\n            if a.cells is None:\\n                return D.Abs(a.shapes, a.shape_lo, None, a.pal, 0, lo, hi)\\n            return D.Abs(a.shapes, a.shape_lo, self._cells_same(a, c), a.pal, 0, lo, hi)\\n        if D.is_concrete(a):\\n            g = D.concretize_one(a)\\n            box = G.bbox_of(g == c)\\n            if box is None:\\n                return D.BOTTOM()\\n            cells = G.crop(self._cells_same(a, c), box).astype(np.uint16)\\n            return D.Abs({cells.shape}, (1, 1), cells, a.pal)\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal)\\n        shapes = {(hh, ww) for H, W in a.shapes for hh in range(1, H + 1) for ww in range(1, W + 1)}\\n        return D.Abs(shapes, (1, 1), None, a.pal | others)\\n\\n    def bwd(self, y, th):\\n        c, crop = th\\n        y = np.asarray(y)\\n        if crop:\\n            return D.Abs(None, y.shape, None, D.ALL, D.bit(c))\\n        cells = (D.BITS[y] | np.uint16(D.bit(c))).astype(np.uint16)\\n        cells[y == c] = D.bit(c)\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL, D.bit(c))\\n", "sfb/__init__.py": "\\"\\"\\"Configuration B\'s side code: the CAST side process (cast_side) and the attempt-2 merge (merge).\\"\\"\\"\\n", "sfb/cast_side.py": "\\"\\"\\"Configuration B: the CAST side process. It runs on CPU next to the neural GPU workers. It calls the CAST package\\nnamed in the build manifest once per puzzle, in a freshly forked child, and writes one result file per puzzle.\\n\\n  python -m sfb.cast_side --challenges C --out DIR --package NAME --entry FUNC --deadline UNIX_TS\\n      [--mode grids|evidence] [--task-ids a,b,...] [--cpu-s 15] [--wall-s 30] [--grace-s 10] [--nice 10]\\n\\nModes:\\n- grids: the entry returns one grid or None per test input, or a details dict with \\"grids\\" (cast_v1\'s\\n  cast_candidates and the constructor-package contract).\\n- evidence: the entry is cast_selective_v2\'s evaluate(task, cpu_s, wall_s). The child turns its report into plain\\n  JSON: status, and per query the first complete candidate and each policy\'s accepted grid. The parent re-validates\\n  every grid. The merge later applies the package\'s own insert_attempt2 to that report.\\n\\nWhat it guarantees (each item is tested in configs/tests/test_cfg_b.py):\\n- State resets between puzzles. CAST code runs only inside a child forked for that puzzle, and the parent never calls\\n  it. So nothing a call leaves behind reaches the next puzzle: module caches, globals, timers, signal handlers or\\n  memory.\\n- Caps are enforced at process level, not only configured.\\n  - The package\'s own caps are passed through: `cpu_s` and `wall_s`.\\n  - Backstop 1: the child gets RLIMIT_CPU of cpu_s + grace_s seconds, enforced by the kernel.\\n  - Backstop 2: the parent SIGKILLs the child at wall_s + grace_s seconds of wall time.\\n  - The child is always reaped (wait4), and its CPU time is recorded.\\n- Total budget: a puzzle starts only if it can finish (wall_s + grace_s) before --deadline. The notebook additionally\\n  kills this whole process group at the deadline.\\n- Output is re-validated here: one entry per test input, each None or a rectangular grid of 1..30 x 1..30 ints in\\n  0..9. Anything else becomes None and is counted as malformed. Result files are written atomically.\\n- Fail-safe: no exception leaves this process. A failure leaves that puzzle without a CAST answer, so it keeps\\n  configuration A\'s attempts.\\n- Only the puzzle\'s demonstrations and test inputs are passed to CAST (the task dict is rebuilt from those fields).\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport select\\nimport signal\\nimport sys\\nimport time\\n\\nfor _v in (\\"OMP_NUM_THREADS\\", \\"OPENBLAS_NUM_THREADS\\", \\"MKL_NUM_THREADS\\", \\"NUMEXPR_NUM_THREADS\\"):\\n    os.environ.setdefault(_v, \\"1\\")                    # one core for CAST; set before numpy is imported\\n\\nMAX_PAYLOAD = 4 << 20                                  # bytes; a larger reply is treated as malformed\\n\\n\\ndef valid_grid(g):\\n    \\"\\"\\"A valid ARC grid: a non-empty rectangular list of lists of ints 0..9, at most 30 x 30 (bools rejected).\\"\\"\\"\\n    if not isinstance(g, list) or not 1 <= len(g) <= 30:\\n        return False\\n    w = None\\n    for row in g:\\n        if not isinstance(row, list) or not 1 <= len(row) <= 30:\\n            return False\\n        if w is None:\\n            w = len(row)\\n        elif len(row) != w:\\n            return False\\n        for v in row:\\n            if type(v) is not int or not 0 <= v <= 9:\\n                return False\\n    return True\\n\\n\\ndef clean_task(t):\\n    \\"\\"\\"Only what the solver may see: demonstration pairs and test inputs.\\"\\"\\"\\n    return {\\"train\\": [{\\"input\\": p[\\"input\\"], \\"output\\": p[\\"output\\"]} for p in t[\\"train\\"]],\\n            \\"test\\": [{\\"input\\": x[\\"input\\"]} for x in t[\\"test\\"]]}\\n\\n\\ndef atomic_json(path, obj):\\n    tmp = f\\"{path}.tmp{os.getpid()}\\"\\n    with open(tmp, \\"w\\") as f:\\n        json.dump(obj, f)\\n        f.flush()\\n        os.fsync(f.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef _to_grid(g):\\n    \\"\\"\\"A grid as a list of lists of ints, None, or the marker \\"MALFORMED\\".\\"\\"\\"\\n    if g is None:\\n        return None\\n    try:\\n        import numpy as np\\n        a = np.asarray(g)\\n        if a.ndim != 2 or a.dtype.kind not in \\"iub\\":\\n            return \\"MALFORMED\\"\\n        return [[int(v) for v in row] for row in a.tolist()]\\n    except Exception:                                  # noqa: BLE001\\n        return \\"MALFORMED\\"\\n\\n\\ndef compact_evidence(rep):\\n    \\"\\"\\"The parts of an evaluate() report that the merge needs, as plain JSON (computed in the child).\\"\\"\\"\\n    if not isinstance(rep, dict):\\n        return {\\"status\\": \\"MALFORMED_REPORT\\", \\"queries\\": None}\\n    out = {\\"status\\": str(rep.get(\\"status\\"))[:60], \\"version\\": str(rep.get(\\"version\\"))[:60],\\n           \\"n_demos\\": rep.get(\\"n_demos\\") if isinstance(rep.get(\\"n_demos\\"), int) else None,\\n           \\"cpu_s\\": rep.get(\\"cpu_s\\") if isinstance(rep.get(\\"cpu_s\\"), (int, float)) else None,\\n           \\"wall_s\\": rep.get(\\"wall_s\\") if isinstance(rep.get(\\"wall_s\\"), (int, float)) else None, \\"queries\\": []}\\n    qs = rep.get(\\"queries\\")\\n    if not isinstance(qs, list):\\n        out.update(status=\\"MALFORMED_REPORT\\", queries=None)\\n        return out\\n    for q in qs:\\n        if not isinstance(q, dict):\\n            out[\\"queries\\"].append(None)\\n            continue\\n        acc = q.get(\\"accepted\\") if isinstance(q.get(\\"accepted\\"), dict) else {}\\n        accm = q.get(\\"accepted_module\\") if isinstance(q.get(\\"accepted_module\\"), dict) else {}\\n        cm = q.get(\\"candidate_module\\")\\n        out[\\"queries\\"].append({\\"candidate\\": _to_grid(q.get(\\"candidate\\")),\\n                               \\"candidate_module\\": None if cm is None else str(cm)[:20],\\n                               \\"accepted\\": {str(k)[:10]: _to_grid(v) for k, v in acc.items()},\\n                               \\"accepted_module\\": {str(k)[:10]: (None if v is None else str(v)[:20])\\n                                                   for k, v in accm.items()}})\\n    return out\\n\\n\\ndef _child(fn, task, kwargs, w, cpu_limit, mode=\\"grids\\"):\\n    \\"\\"\\"Runs in the forked child: never returns.\\"\\"\\"\\n    code = 0\\n    try:\\n        import resource\\n        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))      # a CPU-limit kill leaves no core file\\n        resource.setrlimit(resource.RLIMIT_CPU, (cpu_limit, cpu_limit + 5))\\n        res = fn(task, **kwargs)\\n        if mode == \\"evidence\\":\\n            res = compact_evidence(res)\\n        payload = json.dumps({\\"ok\\": True, \\"res\\": res}, default=str).encode()\\n    except BaseException as e:                         # noqa: BLE001  fail-safe by contract\\n        payload = json.dumps({\\"ok\\": False, \\"error\\": f\\"{type(e).__name__}: {str(e)[:200]}\\"}).encode()\\n        code = 3\\n    try:\\n        view = memoryview(payload)\\n        while view:\\n            n = os.write(w, view)\\n            view = view[n:]\\n    except BaseException:                              # noqa: BLE001\\n        code = 4\\n    os._exit(code)\\n\\n\\ndef run_one(fn, task, kwargs, wall_limit, cpu_limit, mode=\\"grids\\"):\\n    \\"\\"\\"Fork, run, collect. Returns (reply dict or None, child record).\\"\\"\\"\\n    r, w = os.pipe()\\n    t0 = time.time()\\n    pid = os.fork()\\n    if pid == 0:                                       # child\\n        os.close(r)\\n        _child(fn, task, kwargs, w, cpu_limit, mode)\\n    os.close(w)\\n    buf, killed, too_big = bytearray(), False, False\\n    end = t0 + wall_limit\\n    while True:\\n        left = end - time.time()\\n        if left <= 0:\\n            os.kill(pid, signal.SIGKILL)\\n            killed = True\\n            break\\n        ready, _, _ = select.select([r], [], [], min(left, 1.0))\\n        if not ready:\\n            continue\\n        chunk = os.read(r, 1 << 16)\\n        if not chunk:                                  # end of file: the child has finished writing\\n            break\\n        buf += chunk\\n        if len(buf) > MAX_PAYLOAD:\\n            os.kill(pid, signal.SIGKILL)\\n            killed = too_big = True\\n            break\\n    os.close(r)\\n    _, status, ru = os.wait4(pid, 0)                   # always reap\\n    rec = {\\"wall_s\\": round(time.time() - t0, 3), \\"cpu_s\\": round(ru.ru_utime + ru.ru_stime, 3),\\n           \\"exit\\": os.WEXITSTATUS(status) if os.WIFEXITED(status) else None,\\n           \\"signal\\": os.WTERMSIG(status) if os.WIFSIGNALED(status) else None,\\n           \\"killed_by_parent\\": killed, \\"too_big\\": too_big}\\n    reply = None\\n    if not killed and buf:\\n        try:\\n            reply = json.loads(bytes(buf))\\n        except ValueError:\\n            reply = None\\n    return reply, rec\\n\\n\\ndef interpret(reply, rec, n_test):\\n    \\"\\"\\"The reply as {grids, status, module, program, ...}; malformed parts become None.\\"\\"\\"\\n    out = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test, \\"malformed\\": 0}\\n    if rec[\\"killed_by_parent\\"]:\\n        out[\\"status\\"] = \\"KILLED_TOO_BIG\\" if rec[\\"too_big\\"] else \\"KILLED_WALL\\"\\n        return out\\n    if rec[\\"signal\\"] is not None:\\n        out[\\"status\\"] = \\"KILLED_CPU\\" if rec[\\"signal\\"] in (signal.SIGXCPU, signal.SIGKILL) else f\\"SIGNAL_{rec[\'signal\']}\\"\\n        return out\\n    if not isinstance(reply, dict) or not reply.get(\\"ok\\"):\\n        out[\\"status\\"] = \\"ERROR_CHILD\\" if isinstance(reply, dict) else \\"ERROR_NO_REPLY\\"\\n        out[\\"error\\"] = reply.get(\\"error\\") if isinstance(reply, dict) else None\\n        return out\\n    res = reply.get(\\"res\\")\\n    grids = res.get(\\"grids\\") if isinstance(res, dict) else res\\n    out[\\"status\\"] = str(res.get(\\"status\\", \\"OK\\"))[:60] if isinstance(res, dict) else \\"OK\\"\\n    if isinstance(res, dict):\\n        out[\\"version\\"] = str(res.get(\\"version\\"))[:60]\\n        out[\\"cast_cpu_s\\"] = res.get(\\"cpu_s\\") if isinstance(res.get(\\"cpu_s\\"), (int, float)) else None\\n    if not isinstance(grids, list) or len(grids) != n_test:\\n        out[\\"malformed\\"] = n_test\\n        out[\\"status\\"] = \\"MALFORMED_SHAPE\\"\\n        return out\\n    for j, g in enumerate(grids):\\n        if g is None:\\n            continue\\n        if valid_grid(g):\\n            out[\\"grids\\"][j] = g\\n            if isinstance(res, dict):\\n                for k in (\\"module\\", \\"program\\"):\\n                    v = res.get(k)\\n                    if isinstance(v, list) and j < len(v):\\n                        out[k][j] = str(v[j])[:300] if v[j] is not None else None\\n        else:\\n            out[\\"malformed\\"] += 1\\n    return out\\n\\n\\ndef interpret_evidence(reply, rec, n_test):\\n    \\"\\"\\"An evidence reply: kill and error handling as for grids; every grid in the report re-validated.\\"\\"\\"\\n    out = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test, \\"malformed\\": 0,\\n           \\"evidence\\": None}\\n    if rec[\\"killed_by_parent\\"]:\\n        out[\\"status\\"] = \\"KILLED_TOO_BIG\\" if rec[\\"too_big\\"] else \\"KILLED_WALL\\"\\n        return out\\n    if rec[\\"signal\\"] is not None:\\n        out[\\"status\\"] = \\"KILLED_CPU\\" if rec[\\"signal\\"] in (signal.SIGXCPU, signal.SIGKILL) else f\\"SIGNAL_{rec[\'signal\']}\\"\\n        return out\\n    if not isinstance(reply, dict) or not reply.get(\\"ok\\"):\\n        out[\\"status\\"] = \\"ERROR_CHILD\\" if isinstance(reply, dict) else \\"ERROR_NO_REPLY\\"\\n        out[\\"error\\"] = reply.get(\\"error\\") if isinstance(reply, dict) else None\\n        return out\\n    ev = reply.get(\\"res\\")\\n    if not isinstance(ev, dict) or not isinstance(ev.get(\\"queries\\"), list) or len(ev[\\"queries\\"]) != n_test:\\n        out.update(status=\\"MALFORMED_SHAPE\\", malformed=n_test)\\n        return out\\n    clean_q = []\\n    for j, q in enumerate(ev[\\"queries\\"]):\\n        if not isinstance(q, dict):\\n            clean_q.append({\\"candidate\\": None, \\"candidate_module\\": None, \\"accepted\\": {}, \\"accepted_module\\": {}})\\n            out[\\"malformed\\"] += 1\\n            continue\\n        c = q.get(\\"candidate\\")\\n        if c is not None and not valid_grid(c):\\n            out[\\"malformed\\"] += 1\\n            c = None\\n        acc = {}\\n        for k, g in (q.get(\\"accepted\\") or {}).items():\\n            if g is not None and not valid_grid(g):\\n                out[\\"malformed\\"] += 1\\n                g = None\\n            acc[k] = g\\n        clean_q.append({\\"candidate\\": c, \\"candidate_module\\": q.get(\\"candidate_module\\"), \\"accepted\\": acc,\\n                        \\"accepted_module\\": q.get(\\"accepted_module\\") or {}})\\n        out[\\"grids\\"][j] = c\\n        out[\\"module\\"][j] = q.get(\\"candidate_module\\") if c is not None else None\\n    out[\\"status\\"] = str(ev.get(\\"status\\"))[:60]\\n    out[\\"version\\"] = ev.get(\\"version\\")\\n    out[\\"cast_cpu_s\\"] = ev.get(\\"cpu_s\\")\\n    out[\\"evidence\\"] = dict(ev, queries=clean_q)\\n    return out\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--package\\", required=True)\\n    ap.add_argument(\\"--entry\\", required=True)\\n    ap.add_argument(\\"--deadline\\", type=float, required=True)\\n    ap.add_argument(\\"--task-ids\\", default=\\"\\")\\n    ap.add_argument(\\"--cpu-s\\", type=float, default=15.0)\\n    ap.add_argument(\\"--wall-s\\", type=float, default=30.0)\\n    ap.add_argument(\\"--grace-s\\", type=float, default=10.0)\\n    ap.add_argument(\\"--nice\\", type=int, default=10)\\n    ap.add_argument(\\"--details\\", type=int, default=1)\\n    ap.add_argument(\\"--mode\\", default=\\"grids\\", choices=[\\"grids\\", \\"evidence\\"])\\n    a = ap.parse_args(argv)\\n    os.makedirs(a.out, exist_ok=True)\\n    t_start = time.time()\\n    st = {\\"t_start\\": t_start, \\"deadline\\": a.deadline, \\"package\\": a.package, \\"entry\\": a.entry, \\"cpu_s\\": a.cpu_s,\\n          \\"wall_s\\": a.wall_s, \\"grace_s\\": a.grace_s, \\"done\\": 0, \\"answered_tasks\\": 0, \\"answered_outputs\\": 0,\\n          \\"malformed_outputs\\": 0, \\"statuses\\": {}, \\"child_cpu_s\\": 0.0, \\"stopped\\": None, \\"error\\": None,\\n          \\"python\\": sys.version.split()[0], \\"pid\\": os.getpid()}\\n    status_path = os.path.join(a.out, \\"status.json\\")\\n    try:\\n        try:\\n            os.nice(a.nice)\\n        except OSError:\\n            pass\\n        import importlib\\n        mod = importlib.import_module(a.package)\\n        fn = getattr(mod, a.entry)\\n        st[\\"version\\"] = str(getattr(mod, \\"VERSION\\", None))\\n        try:\\n            import numpy\\n            st[\\"numpy\\"] = numpy.__version__\\n        except Exception:                              # noqa: BLE001\\n            st[\\"numpy\\"] = None\\n        ch = json.load(open(a.challenges))\\n        ids = [x for x in a.task_ids.split(\\",\\") if x in ch] if a.task_ids else list(ch)\\n        st[\\"n_tasks\\"] = len(ids)\\n        atomic_json(status_path, st)\\n        kwargs = {\\"cpu_s\\": a.cpu_s, \\"wall_s\\": a.wall_s}\\n        if a.details and a.mode == \\"grids\\":\\n            kwargs[\\"details\\"] = True\\n        st[\\"mode\\"] = a.mode\\n        wall_limit = a.wall_s + a.grace_s\\n        cpu_limit = int(a.cpu_s + a.grace_s) + 1\\n        for tid in ids:\\n            if time.time() + wall_limit > a.deadline:\\n                st[\\"stopped\\"] = \\"budget\\"\\n                break\\n            t = clean_task(ch[tid])\\n            n_test = len(t[\\"test\\"])\\n            try:\\n                reply, rec = run_one(fn, t, kwargs, wall_limit, cpu_limit, a.mode)\\n                res = (interpret_evidence if a.mode == \\"evidence\\" else interpret)(reply, rec, n_test)\\n            except Exception as e:                     # noqa: BLE001  (e.g. fork failed): no answer for this task\\n                rec = {\\"error\\": f\\"{type(e).__name__}: {e}\\"[:200]}\\n                res = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test,\\n                       \\"malformed\\": 0, \\"status\\": \\"ERROR_PARENT\\"}\\n            res.update(task=tid, n_test=n_test, child=rec)\\n            atomic_json(os.path.join(a.out, f\\"{tid}.json\\"), res)\\n            st[\\"done\\"] += 1\\n            k = sum(g is not None for g in res[\\"grids\\"])\\n            st[\\"answered_outputs\\"] += k\\n            st[\\"answered_tasks\\"] += int(k > 0)\\n            st[\\"malformed_outputs\\"] += res.get(\\"malformed\\", 0)\\n            st[\\"statuses\\"][res[\\"status\\"]] = st[\\"statuses\\"].get(res[\\"status\\"], 0) + 1\\n            st[\\"child_cpu_s\\"] = round(st[\\"child_cpu_s\\"] + float(rec.get(\\"cpu_s\\", 0.0) or 0.0), 3)\\n            st[\\"t_now\\"] = time.time()\\n            atomic_json(status_path, st)\\n        else:\\n            st[\\"stopped\\"] = \\"finished\\"\\n    except BaseException as e:                         # noqa: BLE001  fail-safe by contract\\n        st[\\"error\\"] = f\\"{type(e).__name__}: {str(e)[:300]}\\"\\n        st[\\"stopped\\"] = st[\\"stopped\\"] or \\"error\\"\\n    st[\\"wall_s\\"] = round(time.time() - t_start, 2)\\n    try:\\n        atomic_json(status_path, st)\\n    except BaseException:                              # noqa: BLE001\\n        pass\\n    print(json.dumps({k: st.get(k) for k in (\\"done\\", \\"n_tasks\\", \\"answered_tasks\\", \\"answered_outputs\\", \\"stopped\\",\\n                                              \\"error\\", \\"wall_s\\")}), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sfb/merge.py": "\\"\\"\\"Configuration B: merge CAST answers into configuration A\'s submission, changing attempt 2 only.\\n\\n  python -m sfb.merge --sub-a A.json --challenges C --cast-dir DIR --tier 0 --out-b B.json [--report R.json]\\n\\nPolicy (frozen per build; the tier comes from the build, default 0):\\n- Attempt 1 is never changed. It is always configuration A\'s neural attempt.\\n- A CAST answer is used only if:\\n  - its result file has status \\"OK\\" (the package\'s success status);\\n  - it has one entry per test input;\\n  - the grid is valid (rectangular, 1..30 x 1..30, ints 0..9).\\n- Tier 0 (dominance-safe, the default). For a query whose two A attempts are identical, a CAST grid different from\\n  them replaces attempt 2. That query keeps its only distinct neural answer as attempt 1, so under the official metric\\n  (any of the two attempts matching scores the query) the query\'s score cannot fall. Other queries are untouched.\\n- Tier 1 (off unless the build enables it). For a query whose two A attempts differ, a CAST grid different from both\\n  replaces attempt 2. This can lose a correct neural attempt 2. It is the earlier `cast_then_runA` combination.\\n- Anything unexpected keeps A\'s entry: a missing or unreadable result file, a wrong count, a malformed grid, a missing\\n  or malformed A entry.\\n- Self-check before anything is written. B passes the schema check against the challenge file; B\'s attempt 1 equals\\n  A\'s everywhere; every changed query satisfies the tier rule. If any check fails, B is not written and the caller\\n  keeps A.\\n\\"\\"\\"\\nimport argparse\\nimport copy\\nimport json\\nimport os\\n\\nATTEMPTS = (\\"attempt_1\\", \\"attempt_2\\")\\n\\n\\ndef valid_grid(g):\\n    if not isinstance(g, list) or not 1 <= len(g) <= 30:\\n        return False\\n    w = None\\n    for row in g:\\n        if not isinstance(row, list) or not 1 <= len(row) <= 30:\\n            return False\\n        if w is None:\\n            w = len(row)\\n        elif len(row) != w:\\n            return False\\n        for v in row:\\n            if type(v) is not int or not 0 <= v <= 9:\\n                return False\\n    return True\\n\\n\\ndef schema_problems(sub, challenges):\\n    \\"\\"\\"The official format: every task id, one entry per test input in order, both attempts valid grids.\\"\\"\\"\\n    problems = []\\n    if not isinstance(sub, dict):\\n        return [\\"submission is not a dict\\"]\\n    for tid, t in challenges.items():\\n        e = sub.get(tid)\\n        if not isinstance(e, list) or len(e) != len(t[\\"test\\"]):\\n            problems.append(f\\"{tid}: wrong entry count\\")\\n            continue\\n        for j, a in enumerate(e):\\n            if not isinstance(a, dict) or set(a) != set(ATTEMPTS):\\n                problems.append(f\\"{tid}[{j}]: keys\\")\\n                continue\\n            for k in ATTEMPTS:\\n                if not valid_grid(a[k]):\\n                    problems.append(f\\"{tid}[{j}].{k}: invalid grid\\")\\n    extra = set(sub) - set(challenges)\\n    if extra:\\n        problems.append(f\\"{len(extra)} unknown task ids\\")\\n    return problems\\n\\n\\ndef load_cast(cast_dir, tid):\\n    p = os.path.join(cast_dir, f\\"{tid}.json\\")\\n    try:\\n        with open(p) as f:\\n            r = json.load(f)\\n    except (OSError, ValueError):\\n        return None\\n    return r if isinstance(r, dict) else None\\n\\n\\ndef merge(sub_a, challenges, cast_results, tier=0):\\n    \\"\\"\\"sub_a: A\'s submission; cast_results: {task_id: result dict from sfb.cast_side}. Returns (sub_b, report).\\"\\"\\"\\n    if tier not in (0, 1):\\n        raise ValueError(\\"tier must be 0 or 1\\")\\n    sub_b = copy.deepcopy(sub_a)\\n    rep = {\\"tier\\": tier, \\"outputs\\": 0, \\"identical_attempt_outputs\\": 0, \\"cast_tasks_ok\\": 0, \\"cast_answers\\": 0,\\n           \\"replaced_tier0\\": 0, \\"replaced_tier1\\": 0, \\"tier1_candidates\\": 0, \\"same_as_attempt_1\\": 0,\\n           \\"same_as_attempt_2\\": 0, \\"skipped_malformed\\": 0, \\"skipped_count\\": 0, \\"skipped_a_entry\\": 0,\\n           \\"cast_status\\": {}, \\"changes\\": []}\\n    for tid, t in challenges.items():\\n        n = len(t[\\"test\\"])\\n        rep[\\"outputs\\"] += n\\n        entry = sub_b.get(tid)\\n        a_ok = isinstance(entry, list) and len(entry) == n and all(\\n            isinstance(x, dict) and set(x) == set(ATTEMPTS) and valid_grid(x[\\"attempt_1\\"]) and\\n            valid_grid(x[\\"attempt_2\\"]) for x in entry)\\n        if a_ok:\\n            rep[\\"identical_attempt_outputs\\"] += sum(x[\\"attempt_1\\"] == x[\\"attempt_2\\"] for x in entry)\\n        r = cast_results.get(tid)\\n        if r is None:\\n            continue\\n        st = str(r.get(\\"status\\"))\\n        rep[\\"cast_status\\"][st] = rep[\\"cast_status\\"].get(st, 0) + 1\\n        if st != \\"OK\\":\\n            continue\\n        grids = r.get(\\"grids\\")\\n        if not isinstance(grids, list) or len(grids) != n:\\n            rep[\\"skipped_count\\"] += 1\\n            continue\\n        rep[\\"cast_tasks_ok\\"] += 1\\n        if not a_ok:\\n            rep[\\"skipped_a_entry\\"] += 1\\n            continue\\n        for j, g in enumerate(grids):\\n            if g is None:\\n                continue\\n            if not valid_grid(g):\\n                rep[\\"skipped_malformed\\"] += 1\\n                continue\\n            rep[\\"cast_answers\\"] += 1\\n            a1, a2 = entry[j][\\"attempt_1\\"], entry[j][\\"attempt_2\\"]\\n            if g == a1:\\n                rep[\\"same_as_attempt_1\\"] += 1\\n                continue\\n            if a1 == a2:                                    # Tier 0: the two neural attempts are one answer\\n                entry[j] = {\\"attempt_1\\": a1, \\"attempt_2\\": g}\\n                rep[\\"replaced_tier0\\"] += 1\\n                rep[\\"changes\\"].append({\\"task\\": tid, \\"query\\": j, \\"tier\\": 0,\\n                                       \\"module\\": (r.get(\\"module\\") or [None] * n)[j],\\n                                       \\"program\\": (r.get(\\"program\\") or [None] * n)[j]})\\n                continue\\n            if g == a2:\\n                rep[\\"same_as_attempt_2\\"] += 1\\n                continue\\n            rep[\\"tier1_candidates\\"] += 1                    # counted whether or not Tier 1 is enabled\\n            if tier >= 1:\\n                entry[j] = {\\"attempt_1\\": a1, \\"attempt_2\\": g}\\n                rep[\\"replaced_tier1\\"] += 1\\n                rep[\\"changes\\"].append({\\"task\\": tid, \\"query\\": j, \\"tier\\": 1,\\n                                       \\"module\\": (r.get(\\"module\\") or [None] * n)[j],\\n                                       \\"program\\": (r.get(\\"program\\") or [None] * n)[j]})\\n    return sub_b, rep\\n\\n\\ndef self_check(sub_a, sub_b, challenges, tier):\\n    \\"\\"\\"Problems that must stop B from being written (an empty list means B may be written).\\"\\"\\"\\n    problems = schema_problems(sub_b, challenges)\\n    for tid, t in challenges.items():\\n        ea, eb = sub_a.get(tid), sub_b.get(tid)\\n        if not isinstance(ea, list) or not isinstance(eb, list) or len(ea) != len(eb):\\n            if ea != eb:\\n                problems.append(f\\"{tid}: entry changed in length or type\\")\\n            continue\\n        for j, (x, y) in enumerate(zip(ea, eb)):\\n            if not isinstance(x, dict) or not isinstance(y, dict):\\n                if x != y:\\n                    problems.append(f\\"{tid}[{j}]: entry changed\\")\\n                continue\\n            if x.get(\\"attempt_1\\") != y.get(\\"attempt_1\\"):\\n                problems.append(f\\"{tid}[{j}]: attempt_1 changed\\")\\n            if x.get(\\"attempt_2\\") != y.get(\\"attempt_2\\"):\\n                if tier == 0 and x.get(\\"attempt_1\\") != x.get(\\"attempt_2\\"):\\n                    problems.append(f\\"{tid}[{j}]: tier 0 changed a query with two different neural attempts\\")\\n                if y.get(\\"attempt_2\\") == y.get(\\"attempt_1\\"):\\n                    problems.append(f\\"{tid}[{j}]: replacement equals attempt_1\\")\\n    if set(sub_b) != set(sub_a):\\n        problems.append(\\"task id set changed\\")\\n    return problems\\n\\n\\ndef atomic_json(path, obj):\\n    tmp = f\\"{path}.tmp{os.getpid()}\\"\\n    with open(tmp, \\"w\\") as f:\\n        json.dump(obj, f)\\n        f.flush()\\n        os.fsync(f.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef merge_files(sub_a_path, challenges_path, cast_dir, tier, out_b, report_path=None):\\n    \\"\\"\\"Writes B to out_b only if every self-check passes. Returns the report (report[\\"ok\\"] says whether B was\\n    written).\\"\\"\\"\\n    with open(sub_a_path) as f:\\n        sub_a = json.load(f)\\n    with open(challenges_path) as f:\\n        ch = json.load(f)\\n    cast = {}\\n    for tid in ch:\\n        r = load_cast(cast_dir, tid)\\n        if r is not None:\\n            cast[tid] = r\\n    sub_b, rep = merge(sub_a, ch, cast, tier)\\n    problems = self_check(sub_a, sub_b, ch, tier)\\n    rep[\\"problems\\"] = problems[:20]\\n    rep[\\"ok\\"] = not problems\\n    try:\\n        with open(os.path.join(cast_dir, \\"status.json\\")) as f:\\n            rep[\\"cast_side_status\\"] = json.load(f)\\n    except (OSError, ValueError):\\n        rep[\\"cast_side_status\\"] = None\\n    if rep[\\"ok\\"]:\\n        atomic_json(out_b, sub_b)\\n    if report_path:\\n        atomic_json(report_path, rep)\\n    return rep\\n\\n\\ndef policy_tier(policy):\\n    \\"\\"\\"0 for \\"tier0\\", 1 for \\"tier1\\" and \\"tier1:<rule>\\".\\"\\"\\"\\n    if policy == \\"tier0\\":\\n        return 0\\n    if policy == \\"tier1\\" or str(policy).startswith(\\"tier1:\\"):\\n        return 1\\n    raise ValueError(f\\"unknown policy {policy!r}\\")\\n\\n\\ndef merge_packages(sub_a, challenges, packages, results, insert_fns=None):\\n    \\"\\"\\"The constructor hook. packages: ordered [{\\"name\\", \\"mode\\": \\"grids\\"|\\"evidence\\", \\"policy\\"}]; results:\\n    {name: {task_id: side-process result}}; insert_fns: {name: the package\'s insert_attempt2} for evidence packages.\\n    For every query, the first package (in order) that proposes a new attempt 2 under its own policy wins. Attempt 1\\n    is never changed. An evidence package\'s proposal comes from its own insert_attempt2(task, neural, policy,\\n    report=...), computed against the neural attempts. Any failure for a task keeps A\'s entry for that task.\\n    Returns (sub_b, report).\\"\\"\\"\\n    insert_fns = insert_fns or {}\\n    for pk in packages:\\n        policy_tier(pk[\\"policy\\"])\\n    sub_b = copy.deepcopy(sub_a)\\n    rep = {\\"packages\\": [dict(pk) for pk in packages], \\"outputs\\": 0, \\"identical_attempt_outputs\\": 0,\\n           \\"replaced_tier0\\": 0, \\"replaced_tier1\\": 0, \\"skipped_a_entry\\": 0, \\"per_package\\": {}, \\"changes\\": []}\\n    for pk in packages:\\n        rep[\\"per_package\\"][pk[\\"name\\"]] = {\\"status\\": {}, \\"tasks_ok\\": 0, \\"proposals\\": 0, \\"applied\\": 0,\\n                                          \\"same_as_attempt_1\\": 0, \\"rejected_output\\": 0, \\"errors\\": 0}\\n    for tid, t in challenges.items():\\n        n = len(t[\\"test\\"])\\n        rep[\\"outputs\\"] += n\\n        entry = sub_b.get(tid)\\n        a_ok = isinstance(entry, list) and len(entry) == n and all(\\n            isinstance(x, dict) and set(x) == set(ATTEMPTS) and valid_grid(x[\\"attempt_1\\"]) and\\n            valid_grid(x[\\"attempt_2\\"]) for x in entry)\\n        if not a_ok:\\n            rep[\\"skipped_a_entry\\"] += 1\\n            continue\\n        neural = [[x[\\"attempt_1\\"], x[\\"attempt_2\\"]] for x in entry]\\n        rep[\\"identical_attempt_outputs\\"] += sum(a1 == a2 for a1, a2 in neural)\\n        changed = [False] * n\\n        for pk in packages:\\n            pp = rep[\\"per_package\\"][pk[\\"name\\"]]\\n            r = (results.get(pk[\\"name\\"]) or {}).get(tid)\\n            if r is None:\\n                continue\\n            st = str(r.get(\\"status\\"))\\n            pp[\\"status\\"][st] = pp[\\"status\\"].get(st, 0) + 1\\n            if st != \\"OK\\":\\n                continue\\n            proposals = [None] * n\\n            try:\\n                if pk[\\"mode\\"] == \\"evidence\\":\\n                    ev = r.get(\\"evidence\\")\\n                    if not isinstance(ev, dict) or ev.get(\\"status\\") != \\"OK\\" or \\\\\\n                            not isinstance(ev.get(\\"queries\\"), list) or len(ev[\\"queries\\"]) != n:\\n                        continue\\n                    task = {\\"train\\": t[\\"train\\"], \\"test\\": [{\\"input\\": x[\\"input\\"]} for x in t[\\"test\\"]]}\\n                    out = insert_fns[pk[\\"name\\"]](task, copy.deepcopy(neural), policy=pk[\\"policy\\"], report=ev)\\n                    if not isinstance(out, list) or len(out) != n or any(\\n                            not isinstance(o, (list, tuple)) or len(o) != 2 or o[0] != neural[j][0]\\n                            for j, o in enumerate(out)):\\n                        pp[\\"rejected_output\\"] += 1\\n                        continue\\n                    for j, o in enumerate(out):\\n                        if o[1] is not None and o[1] != neural[j][1]:\\n                            proposals[j] = o[1]\\n                else:\\n                    grids = r.get(\\"grids\\")\\n                    if not isinstance(grids, list) or len(grids) != n:\\n                        continue\\n                    tier = policy_tier(pk[\\"policy\\"])\\n                    for j, g in enumerate(grids):\\n                        if g is None or g == neural[j][1]:\\n                            continue\\n                        if tier == 0 and neural[j][0] != neural[j][1]:\\n                            continue\\n                        proposals[j] = g\\n            except Exception:                              # noqa: BLE001  this package gives nothing for this task\\n                pp[\\"errors\\"] += 1\\n                continue\\n            pp[\\"tasks_ok\\"] += 1\\n            for j, g in enumerate(proposals):\\n                if g is None:\\n                    continue\\n                pp[\\"proposals\\"] += 1\\n                if not valid_grid(g):\\n                    pp[\\"rejected_output\\"] += 1\\n                    continue\\n                if g == neural[j][0]:\\n                    pp[\\"same_as_attempt_1\\"] += 1\\n                    continue\\n                if changed[j]:\\n                    continue\\n                tier_j = 0 if neural[j][0] == neural[j][1] else 1\\n                if tier_j > policy_tier(pk[\\"policy\\"]):\\n                    pp[\\"rejected_output\\"] += 1\\n                    continue\\n                entry[j] = {\\"attempt_1\\": neural[j][0], \\"attempt_2\\": g}\\n                changed[j] = True\\n                pp[\\"applied\\"] += 1\\n                rep[\\"replaced_tier0\\" if tier_j == 0 else \\"replaced_tier1\\"] += 1\\n                rep[\\"changes\\"].append({\\"task\\": tid, \\"query\\": j, \\"package\\": pk[\\"name\\"], \\"tier\\": tier_j})\\n    return sub_b, rep\\n\\n\\ndef merge_files_packages(sub_a_path, challenges_path, packages, out_b, report_path=None, code_dir=None):\\n    \\"\\"\\"packages: [{\\"name\\", \\"mode\\", \\"policy\\", \\"dir\\"}]. Imports each evidence package\'s insert_attempt2 by name.\\n    Writes B to out_b only if every self-check passes; report[\\"ok\\"] says whether it was written.\\"\\"\\"\\n    import importlib\\n    import sys\\n    if code_dir and code_dir not in sys.path:\\n        sys.path.insert(0, code_dir)\\n    with open(sub_a_path) as f:\\n        sub_a = json.load(f)\\n    with open(challenges_path) as f:\\n        ch = json.load(f)\\n    results, insert_fns, side = {}, {}, {}\\n    for pk in packages:\\n        res = {}\\n        for tid in ch:\\n            r = load_cast(pk[\\"dir\\"], tid)\\n            if r is not None:\\n                res[tid] = r\\n        results[pk[\\"name\\"]] = res\\n        if pk[\\"mode\\"] == \\"evidence\\":\\n            insert_fns[pk[\\"name\\"]] = importlib.import_module(pk[\\"name\\"]).insert_attempt2\\n        try:\\n            with open(os.path.join(pk[\\"dir\\"], \\"status.json\\")) as f:\\n                side[pk[\\"name\\"]] = json.load(f)\\n        except (OSError, ValueError):\\n            side[pk[\\"name\\"]] = None\\n    sub_b, rep = merge_packages(sub_a, ch, packages, results, insert_fns)\\n    tier = max(policy_tier(pk[\\"policy\\"]) for pk in packages) if packages else 0\\n    problems = self_check(sub_a, sub_b, ch, tier)\\n    rep.update(problems=problems[:20], ok=not problems, cast_side_status=side, tier=tier)\\n    if rep[\\"ok\\"]:\\n        atomic_json(out_b, sub_b)\\n    if report_path:\\n        atomic_json(report_path, rep)\\n    return rep\\n\\n\\ndef reconcile_guard_output(sub_path, challenges_path, kroot, code_dir):\\n    \\"\\"\\"Entry H only (C4\'s guard; a no-op without exactly one guarded_run_* folder): correct C4\'s child-path defect.\\n    The guard\'s final salvage reads the public file instead of the driver\'s private child_submission.json. Here C4\'s\\n    own, unchanged salvage() runs with the real child path: precedence completed pools > in-flight progress > child\\n    file > fallback, each query validated. The result replaces the public file only if it passes the full schema\\n    check. Otherwise the public file stays. Returns a report.\\"\\"\\"\\n    import glob\\n    import sys\\n    rep = {\\"applied\\": False, \\"reason\\": None, \\"changed_queries\\": 0}\\n    runs = sorted(glob.glob(os.path.join(kroot, \\"working\\", \\"guarded_run_*\\")))\\n    if len(runs) != 1 or not os.path.isdir(runs[0]):\\n        rep[\\"reason\\"] = f\\"{len(runs)} guarded_run folders\\"\\n        return rep\\n    if code_dir and code_dir not in sys.path:\\n        sys.path.insert(0, code_dir)\\n    from deployment_guard import salvage\\n    with open(challenges_path) as f:\\n        ch = json.load(f)\\n    with open(sub_path) as f:\\n        public = json.load(f)\\n    fixed, recovered = salvage(ch, os.path.join(runs[0], \\"child_submission.json\\"), runs[0])\\n    problems = schema_problems(fixed, ch)\\n    rep.update(run_dir=runs[0], recovered_queries=recovered, problems=problems[:10])\\n    if problems:\\n        rep[\\"reason\\"] = \\"salvaged file failed the schema check; public file kept\\"\\n        return rep\\n    rep[\\"changed_queries\\"] = sum(1 for tid, t in ch.items() for j in range(len(t[\\"test\\"]))\\n                                 if not (isinstance(public.get(tid), list) and j < len(public[tid]) and\\n                                         public[tid][j] == fixed[tid][j]))\\n    if rep[\\"changed_queries\\"]:\\n        atomic_json(sub_path, fixed)\\n    rep[\\"applied\\"] = True\\n    return rep\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--sub-a\\", required=True)\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--cast-dir\\", required=True)\\n    ap.add_argument(\\"--tier\\", type=int, default=0)\\n    ap.add_argument(\\"--out-b\\", required=True)\\n    ap.add_argument(\\"--report\\")\\n    a = ap.parse_args(argv)\\n    rep = merge_files(a.sub_a, a.challenges, a.cast_dir, a.tier, a.out_b, a.report)\\n    print(json.dumps({k: v for k, v in rep.items() if k not in (\\"changes\\", \\"cast_side_status\\")}))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
    _sha = {'cast_selective_v2/__init__.py': '9773129899fdb148b7ea3837228a123373e1f37cc58504b03787ccd230a3e28b', 'cast_selective_v2/_nd.py': '18e53fa3cbbc6ddae32211ef99d2bb5b1dffec10bc5e4e8dae1422cdf999826f', 'cast_selective_v2/absdom.py': '7a7664b78481a6c7301e329536c36ed109462157303c128207e0db97182394d6', 'cast_selective_v2/api.py': '2454441f4dca916bde3dc3f2f1bb4fe49a2853c5f5da2787406ab8fffa80f355', 'cast_selective_v2/candidates.py': '792825e4b6f067494ace0eceff766f9c78788e35a651d254ef22c1f925c200aa', 'cast_selective_v2/grids.py': '42c349f4c672bc506e3bfec30ff40c63fa86056c561a2b7ea33f32dec931a7f0', 'cast_selective_v2/ops.py': '380102ef6cac9196bfc8c8713326196533ed83ccb1d292df459799fc9e80a18e', 'cast_selective_v2/rewrite.py': '86381a6ae3f37215d4919621c3be1a3393c4943174f568b8e27159f1330eb88b', 'cast_selective_v2/search.py': 'a6b6261906ed1be03a823912546df486cd2ea9cacc3a4cca05f06ad0cc27e48d', 'cast_selective_v2/symmetry.py': '0040c861706b6ecf173bb2acb48c0d6aacad64c77fb8e73a90412ee28668b04c', 'sfb/__init__.py': '839f1efcafe73c661e0f7b726f169ace61e0c76c4fc1cc7623fd39533cf59829', 'sfb/cast_side.py': 'c072eccd3595bce1a3cd3fd028cc2f378b6fcf6ada838b4e7ea772dd22494853', 'sfb/merge.py': 'cd67102c6533fef06834d7194c48dc4613fdb0cdf8bbdc79fa2e484e81d279ac'}
    for _name, _text in _files.items():
        if hashlib.sha256(_text.encode("utf-8")).hexdigest() != _sha[_name]:
            raise RuntimeError("embedded file hash mismatch: " + _name)
        _p = os.path.join(CODE_DIR, _name)
        os.makedirs(os.path.dirname(_p), exist_ok=True)
        with open(_p, "w", encoding="utf-8") as _f:
            _f.write(_text)
    _kroot = os.environ.get("SF_KAGGLE_ROOT", "/kaggle")
    _comp = None
    for _d in (_kroot + "/input/competitions/arc-prize-2026-arc-agi-2", _kroot + "/input/arc-prize-2026-arc-agi-2"):
        if os.path.isdir(_d):
            _comp = _d
            break
    if _comp is None:
        import glob
        _comp = os.path.dirname(glob.glob(_kroot + "/input/**/arc-agi_test_challenges.json", recursive=True)[0])
    _ids = ""
    if os.environ.get("SF_REHEARSE_ON") == "evaluation":
        _ch_path = os.path.join(_comp, "arc-agi_evaluation_challenges.json")
    else:
        _ch_path = os.path.join(_comp, "arc-agi_test_challenges.json")
        with open(_ch_path) as _f:
            _test = json.load(_f)
        try:
            with open(os.path.join(_comp, "arc-agi_training_challenges.json")) as _f:
                _train_ids = set(json.load(_f))
        except (OSError, ValueError):
            _train_ids = set()
        if _test and set(_test) <= _train_ids:      # the public placeholder: the neural driver's commit subset
            _ids = ",".join(sorted(_test)[:int(os.environ.get("SF_COMMIT_TASKS", "8"))])
    CAST_B["ch_path"] = _ch_path
    _budget = min(float(os.environ["SF_CAST_TOTAL_S"]),
                  float(os.environ.get("SF_DEADLINE_S", str(11.5 * 3600))) - float(os.environ["SF_CAST_END_RESERVE_S"]))
    CAST_B["deadline"] = time.time() + max(0.0, _budget)
    _env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
                MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1", PYTHONHASHSEED="0", PYTHONUNBUFFERED="1")

    def _cast_watchdog(p, end):
        while p.poll() is None and time.time() < end:
            time.sleep(5)
        if p.poll() is None:
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except OSError:
                pass
    for _i, _pk in enumerate(B_PACKAGES):
        _out = _kroot + "/working/cora_side/%d_%s" % (_i, _pk["name"])
        os.makedirs(_out, exist_ok=True)
        _cmd = [sys.executable, "-m", "sfb.cast_side", "--challenges", _ch_path, "--out", _out,
                "--package", _pk["name"], "--entry", _pk["entry"], "--mode", _pk["mode"],
                "--deadline", str(CAST_B["deadline"]), "--cpu-s", _pk["cpu_s"], "--wall-s", _pk["wall_s"],
                "--grace-s", os.environ["SF_CAST_GRACE_S"], "--nice", os.environ["SF_CAST_NICE"]]
        if _ids:
            _cmd += ["--task-ids", _ids]
        _log = open(_out + ".log", "w")
        _proc = subprocess.Popen(_cmd, cwd=CODE_DIR, env=_env, stdout=_log, stderr=_log, start_new_session=True)
        CAST_B["procs"].append(_proc)
        CAST_B["packages"].append({"name": _pk["name"], "mode": _pk["mode"], "policy": _pk["policy"], "dir": _out})
        threading.Thread(target=_cast_watchdog, args=(_proc, CAST_B["deadline"] + 60), daemon=True).start()
        print("CORA side process started:", _pk["name"], _pk["version"], _pk["mode"], _pk["policy"],
              "budget_s", round(_budget), "tasks", _ids or "all", flush=True)
except BaseException as _e:
    CAST_B["error"] = repr(_e)
    print("CORA not started; the neural output will be submitted:", repr(_e), flush=True)


In [ ]:
import json, os, sys, time, uuid
from pathlib import Path
sys.path.insert(0, CODE_DIR)
from deployment_guard import Budget, find_challenges, supervise
_root = Path(os.environ.get("SF_KAGGLE_ROOT", "/kaggle"))
_challenge = find_challenges(_root, os.environ.get("SF_REHEARSE_ON") == "evaluation")
_ch = json.loads(_challenge.read_text())
_token = uuid.uuid4().hex
_driver = _root / "working" / ("guarded_driver_" + _token + ".py")
_driver.parent.mkdir(parents=True, exist_ok=True)
_driver.write_text("CODE_DIR=" + repr(CODE_DIR) + "\nCODE_SHA=" + repr(CODE_SHA) + "\n" + '\nimport glob, json, os, subprocess, sys, time\nKROOT = os.environ.get("SF_KAGGLE_ROOT", "/kaggle")   # a local dress rehearsal sets another root\nT0 = time.time()\nMODE = os.environ.get("SF_MODE", "submit")\ndef find_model():\n    for cfg in sorted(glob.glob(KROOT + "/input/**/config.json", recursive=True)):\n        try:\n            c = json.load(open(cfg))\n        except Exception:\n            continue\n        if c.get("model_type") == "qwen3" and c.get("vocab_size") == 16:\n            return os.path.dirname(cfg)\n    raise SystemExit("checkpoint not found under /kaggle/input")\ndef find_comp():\n    for d in (KROOT + "/input/competitions/arc-prize-2026-arc-agi-2", KROOT + "/input/arc-prize-2026-arc-agi-2"):\n        if os.path.isdir(d):\n            return d\n    hits = glob.glob(KROOT + "/input/**/arc-agi_test_challenges.json", recursive=True)\n    return os.path.dirname(hits[0])\nMODEL_DIR, COMP_DIR = find_model(), find_comp()\nOUT = os.environ.get("SF_RUN_DIR", KROOT + "/working/run")\nos.makedirs(OUT, exist_ok=True)\nSUB = os.environ.get("SF_CHILD_SUBMISSION", KROOT + "/working/submission.json")\nargs = ["--model", MODEL_DIR, "--out", OUT, "--submission", SUB, "--gpus", os.environ.get("SF_GPUS", "4"),\n        "--deadline-s", os.environ.get("SF_DEADLINE_S", str(11.5 * 3600)),\n        "--n-gen", os.environ.get("SF_N_GEN", "8"), "--n-score", os.environ.get("SF_N_SCORE", "16"),\n        "--task-cap-s", os.environ.get("SF_TASK_CAP_S", "900"), "--code-dir", CODE_DIR,\n        "--ttt-steps", os.environ.get("SF_TTT_STEPS", "0"), "--ttt-rank", os.environ.get("SF_TTT_RANK", "64"),\n        "--ttt-alpha", os.environ.get("SF_TTT_ALPHA", "64"), "--ttt-lr", os.environ.get("SF_TTT_LR", "1e-4"),\n        "--ttt-max-tokens", os.environ.get("SF_TTT_MAX_TOKENS", "8192"),\n        "--ttt-frac", os.environ.get("SF_TTT_FRAC", "0.7"), "--method", os.environ.get("SF_METHOD", "vote_score")]\nif int(os.environ.get("SF_TTT_STEPS", "0")) > 0:\n    whl = [p for p in glob.glob(KROOT + "/input/**/peft-0.21.2-py3-none-any.whl", recursive=True) if os.path.isfile(p)]\n    if whl:\n        os.environ["PYTHONPATH"] = whl[0] + os.pathsep + os.environ.get("PYTHONPATH", "")\ninfo = {"mode": MODE, "model_dir": MODEL_DIR, "comp_dir": COMP_DIR, "code_sha256": CODE_SHA,\n        "rerun_env": os.environ.get("KAGGLE_IS_COMPETITION_RERUN"), "python": sys.version.split()[0]}\nfor mod in ("torch", "transformers", "accelerate", "peft", "numpy", "scipy"):\n    try:\n        info[mod] = __import__(mod).__version__\n    except Exception as e:\n        info[mod] = None\ninfo["model_files"] = {f: os.path.getsize(os.path.join(MODEL_DIR, f)) for f in sorted(os.listdir(MODEL_DIR))}\nif MODE == "dev":\n    ch_path = os.path.join(COMP_DIR, "arc-agi_evaluation_challenges.json")\n    if os.environ.get("SF_TASK_IDS"):\n        args += ["--task-ids", os.environ["SF_TASK_IDS"]]\n    elif os.environ.get("SF_LIMIT"):\n        args += ["--limit", os.environ["SF_LIMIT"]]\nelif os.environ.get("SF_REHEARSE_ON") == "evaluation":\n    # timed rehearsal: the submit path end to end, on the public evaluation challenges in place of the test file\n    ch_path = os.path.join(COMP_DIR, "arc-agi_evaluation_challenges.json")\n    info["placeholder_test_file"] = False\n    info["rehearsal"] = "public evaluation challenges (CONTAMINATED for this checkpoint)"\nelse:\n    ch_path = os.path.join(COMP_DIR, "arc-agi_test_challenges.json")\n    with open(ch_path) as f:\n        test_ids = set(json.load(f))\n    try:                                    # a missing or unreadable training file means "not the placeholder"\n        with open(os.path.join(COMP_DIR, "arc-agi_training_challenges.json")) as f:\n            train_ids = set(json.load(f))\n    except (OSError, ValueError):\n        train_ids = set()\n    placeholder = bool(test_ids) and test_ids <= train_ids\n    info["placeholder_test_file"] = placeholder\n    if placeholder:\n        n = os.environ.get("SF_COMMIT_TASKS", "8")\n        ch_full = json.load(open(ch_path))\n        keep = sorted(ch_full)[:int(n)]\n        args += ["--task-ids", ",".join(keep)]\ninfo["challenges"] = ch_path\nwith open(KROOT + "/working/run_info.json", "w") as f:\n    json.dump(info, f, indent=1)\nprint(json.dumps(info), flush=True)\ntry:\n    subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])\nexcept OSError:\n    pass\nr = subprocess.run([sys.executable, "-m", "sf.runner", "orchestrate", "--challenges", ch_path] + args, cwd=CODE_DIR)\nif MODE != "dev":\n    # the runner wrote a file for the tasks it ran; complete it for every task of the official file\n    sys.path.insert(0, CODE_DIR)\n    from sf import select as S\n    ch_all = json.load(open(ch_path))\n    try:\n        sub = json.load(open(SUB))\n    except Exception:\n        sub = {}\n    for tid, t in ch_all.items():\n        if tid not in sub or len(sub[tid]) != len(t["test"]):\n            sub[tid] = [S.two_attempts([], x["input"], []) for x in t["test"]]\n    tmp = SUB + ".tmp"\n    with open(tmp, "w") as f:\n        json.dump(sub, f)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, SUB)\n    problems = S.check_submission(sub, ch_all)\n    print("schema problems:", len(problems), problems[:5])\nprint("returncode", r.returncode, "wall_s", round(time.time() - T0, 1))\n')
_run = _root / "working" / ("guarded_run_" + _token)
_budget = Budget(_GUARD_STARTED, float(os.environ.get("SF_GUARD_TOTAL_S", "42600")),
                 float(os.environ.get("SF_GUARD_SERIALIZE_S", "300")),
                 float(os.environ.get("SF_GUARD_SHUTDOWN_S", "30")))
_report = supervise(_ch, _root / "working/submission.json", _run,
                    [sys.executable, str(_driver)], budget=_budget, cwd=CODE_DIR)
print(json.dumps(_report), flush=True)


In [ ]:

# CORA side processes, part 2 of 2: CORA may change attempt 2 under each package's frozen policy (sfb/merge.py).
# Fail-safe: if anything in this cell fails, submission.json stays the neural file.
import json, os, shutil, signal, sys, time
try:
    _kroot = os.environ.get("SF_KAGGLE_ROOT", "/kaggle")
    _sub = _kroot + "/working/submission.json"
    _end = min((CAST_B["deadline"] or time.time()) + 30,
               CAST_B["t0"] + float(os.environ.get("SF_DEADLINE_S", str(11.5 * 3600))) + 300)
    for _p in CAST_B.get("procs") or []:              # still running: wait, but never past its own deadline
        while _p.poll() is None and time.time() < _end:
            time.sleep(2)
        if _p.poll() is None:
            try:
                os.killpg(_p.pid, signal.SIGKILL)
            except OSError:
                pass
            _p.wait(timeout=30)
    if CODE_DIR not in sys.path:
        sys.path.insert(0, CODE_DIR)
    from sfb import merge as _BM
    _ch_path = CAST_B.get("ch_path")
    if not _ch_path:
        with open(_kroot + "/working/run_info.json") as _f:
            _ch_path = json.load(_f)["challenges"]
    try:                                               # entry H: C4's own salvage with the real child path
        _recon = _BM.reconcile_guard_output(_sub, _ch_path, _kroot, CODE_DIR)
    except BaseException as _e:
        _recon = {"applied": False, "reason": repr(_e)}
    print("neural file reconciliation:", json.dumps(_recon)[:500], flush=True)
    shutil.copyfile(_sub, _kroot + "/working/submission_A.json")      # the neural file, frozen before CORA
    _rep = _BM.merge_files_packages(_sub, _ch_path, CAST_B["packages"], _kroot + "/working/submission_B.json",
                                    _kroot + "/working/cast_merge_report.json", CODE_DIR)
    if _rep["ok"]:
        with open(_kroot + "/working/submission_B.json") as _f:
            _BM.atomic_json(_sub, json.load(_f))
    print("CORA merge:", json.dumps({k: _rep.get(k) for k in (
        "ok", "tier", "identical_attempt_outputs", "replaced_tier0", "replaced_tier1", "per_package",
        "problems")}), flush=True)
except BaseException as _e:
    print("CORA merge skipped; submission.json is the neural file:", repr(_e), flush=True)
